## Loading Modules

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
import h5py
from scipy.stats import norm

import torch
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from torch import nn, optim
import scipy.io as sio
# import pandas as pd
import datetime
import os
# import readligo as rl
# from gwpy.timeseries import TimeSeries
import math
import random

import copy

import torch.nn.functional as F
import pickle
import itertools
import re

import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import LineChart, Reference

In [2]:
os.chdir('D:\OneDrive - HKUST Connect\Research\GWNMMAD\GWAD\Codes')

In [3]:
epochs = 60
rTrain = 0.8;
rTest = 0.1;
# input_vector_length = 100
batch_size = 32
num_bins = 40
coef_delta = 0

device = 'cuda:0'

In [85]:
np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz').keys()

In [86]:
np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['data'].shape

In [87]:
np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['data'].shape[-2:]

In [88]:
np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 1)

In [156]:
np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 1).shape

In [157]:
np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 0).shape

In [152]:
print(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'])

In [99]:
_, ax = plt.subplots(25, 2, figsize = (6.4*2,4.8*25))

for i in range(50):

# idx = np.random.choice(np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 1).flatten(),1).flatten()[0]
    idx = np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 1).flatten()[i]
    ax[i//2, i%2].plot(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['data'][idx,0])
    ax[i//2, i%2].plot(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['data'][idx,1])

In [ ]:
bbh_set = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/bbh_for_challenge.npy')
bbh_set = bbh_set / np.linalg.norm(bbh_set, axis = -1).reshape(-1,2,1)
bbh_set.shape

In [101]:
_, ax = plt.subplots(25, 2, figsize = (6.4*2,4.8*25))

for i in range(50):

# idx = np.random.choice(np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 1).flatten(),1).flatten()[0]
    # idx = np.argwhere(np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')['ids'] == 1).flatten()[i]
    ax[i//2, i%2].plot(np.load('E://GWNMMAD_data/Tw_dataset/Datasets/bbh_for_challenge.npy')[i,0])
    ax[i//2, i%2].plot(np.load('E://GWNMMAD_data/Tw_dataset/Datasets/bbh_for_challenge.npy')[i,1])

## Define the structure

In [5]:
ModelDir = './Sida_temp/for_K8S_training/Output'

In [6]:
class AE_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(AE_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.encoder_layers = nn.ModuleList()
        self.norm_encoder_layers = nn.ModuleList()
        self.decoder_layers = nn.ModuleList()
        self.norm_decoder_layers = nn.ModuleList()

        # print(self.encoder_struct.devide)
        

        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.encoder_layers.append(layer)
            self.norm_encoder_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

            layer = nn.Linear(self.encoder_struct[self.dep-1-i], self.encoder_struct[self.dep-2-i])
            nn.init.kaiming_normal_(layer.weight)
            self.decoder_layers.append(layer)
            self.norm_decoder_layers.append(nn.BatchNorm1d(self.encoder_struct[self.dep-2-i]))

    def forward(self, x):
        for i, layer in enumerate(self.encoder_layers):
            x = layer(x)
            x = self.relu(x)
            # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
            x = self.norm_encoder_layers[i](x)
            
        encoded = x;

        for i, layer in enumerate(self.decoder_layers):
            x = layer(x)
            if(i < self.dep-2):
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[self.dep-2-i])(x)
                x = self.norm_decoder_layers[i](x)

        decoded = nn.Sigmoid()(x)

        return encoded, decoded

In [7]:
class WSC_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()


        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        return x

In [8]:
class WSC_1det_struct_upd_2(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct_upd_2, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.sm = nn.Softmax(dim=1)
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()
        self.last_linear = nn.Linear(self.encoder_struct[-1], 1)
        # self.sig = nn.Sigmoid()

        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        # print(self.sm(x))
        x = self.last_linear(self.sm(x))

        return x;

## Load the data

In [9]:
os.getcwd()

In [50]:
np.load("../../../../Data_cached/Noise_processing/BBH_injection/Output/BBH_events_BBH_GW190403_051519.npz").keys()

In [44]:
test

In [53]:
test = np.load("../../../../Data_cached/Noise_processing/SGHF_injection/Output/SGHF_events_BBH_GW190403_051519.npz")['events']

In [55]:
test.shape

In [57]:
plt.plot(test[0,1])

In [7]:
dataset_wsl_fft_all = np.load("./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM.npy")

In [4]:
dataset_wsl_fft_all = np.load("./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_100k_newprocess.npy")

In [7]:
dataset_wsl_fft_all = np.load("./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_100k_GW190403_051519.npy")

In [63]:
GW_event = np.load('../Data_cached/Noise_processing/Events/BBH_events_BBH_GW190403_051519_2.npy')

In [13]:
dataset_wsl_fft_all = np.load("./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_100k_GW190408_181802.npy")

In [14]:
GW_event = np.load('../Data_cached/Noise_processing/Events/BBH_events_BBH_GW190408_181802_2.npy')

In [15]:
GW_event.shape

In [16]:
GW_event = GW_event / np.linalg.norm(GW_event, axis = -1).reshape(-1,2,1)

In [17]:
GW_event = np.abs(np.fft.rfft(GW_event, axis = -1))


In [18]:
GW_event = GW_event / np.linalg.norm(GW_event, axis = -1).reshape(-1,2,1)

In [19]:
GW_event = GW_event.reshape(-1,202)

In [20]:
GW_event.shape

In [70]:
np.linalg.norm(GW_event, axis = -1)

In [9]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sghf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sglf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [9]:
glitch_H_in_segment = np.load("../Data_cached/real_glitches_H_snrlt5_58803_6424_4000Hz_50ms.npz")['strain_time_data']
glitch_L_in_segment = np.load("../Data_cached/real_glitches_L_snrlt5_63917_7878_4000Hz_50ms.npz")['strain_time_data']

glitch_H_in_segment = glitch_H_in_segment / np.linalg.norm(glitch_H_in_segment, axis = 1).reshape(-1,1)
glitch_L_in_segment = glitch_L_in_segment / np.linalg.norm(glitch_L_in_segment, axis = 1).reshape(-1,1)

In [10]:
glitch_H_in_segment_fft = np.abs(np.fft.rfft(glitch_H_in_segment))
glitch_L_in_segment_fft = np.abs(np.fft.rfft(glitch_L_in_segment))

glitch_H_in_segment_fft = glitch_H_in_segment_fft / np.linalg.norm(glitch_H_in_segment_fft, axis = 1).reshape(-1,1)
glitch_L_in_segment_fft = glitch_L_in_segment_fft / np.linalg.norm(glitch_L_in_segment_fft, axis = 1).reshape(-1,1)

In [11]:
dataset_wsl_separated[0][:2500][:,:101].shape

In [12]:
noise_for_glitch_H = dataset_wsl_separated[0][:2500][:,:101].copy()
noise_for_glitch_L = dataset_wsl_separated[0][2500:][:,101:].copy()

In [13]:
dataset_wsl_separated_newglitch = {}

dataset_wsl_separated_newglitch[0] = np.vstack((np.hstack((noise_for_glitch_H,glitch_H_in_segment_fft[:2500])), np.hstack((glitch_L_in_segment_fft[:2500], noise_for_glitch_L))))        # glitch
dataset_wsl_separated_newglitch[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated_newglitch[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated_newglitch[3] = dataset_wsl_fft_all[80000:90000].copy()  # sghf
dataset_wsl_separated_newglitch[4] = dataset_wsl_fft_all[90000:].copy()       # sglf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [14]:
dataset_wsl_separated_newglitch[0].shape

In [15]:
dataset_wsl_fft_all_newglitch = np.empty((0,202))

for key in dataset_wsl_separated_newglitch.keys():
    dataset_wsl_fft_all_newglitch = np.append(dataset_wsl_fft_all_newglitch, dataset_wsl_separated_newglitch[key], axis = 0)

In [16]:
dataset_wsl_fft_all_newglitch.shape

In [14]:
os.getcwd()

In [201]:
dataset_wsl_fft_all_newglitch = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_enlarged_2.npy')

In [202]:
dataset_wsl_fft_all_newglitch.shape

In [73]:
dataset_wsl_fft_all_enlarged = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_enlarged_4_1.npy')

In [10]:
dataset_wsl_fft_all_enlarged.shape

In [7]:
dataset_wsl_fft_withccsn = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_withCCSN.npy')

In [62]:
dataset_wsl_fft_withccsn_enlarged = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_withCCSN_1.npy')

In [9]:
dataset_GWAK = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_GWAK_LM.npy')

In [10]:
dataset_GWAK.shape

In [92]:
dataset_wsl_fft_all.shape

In [93]:
dataset_withoutsghf = np.concatenate((dataset_wsl_fft_all[:80000],dataset_wsl_fft_all[90000:]), axis = 0)

In [94]:
dataset_withoutsghf.shape

## Analysis Procedure

### Analyzing functions

In [11]:
def return_model_with_least_valloss(model_list):
    # valloss_list = np.empty((0))
    # epoch_list = np.empty((0))
    # extracting the epochs the model is using
    # print(model_list.keys())
    epoch_list = [int(re.search(r'(\d+)valloss', item).group(1)) for item in model_list.keys() if isinstance(item, str) and 'valloss' in item]
    valloss_list = [model_list[key] for key in [item for item in model_list.keys() if isinstance(item, str) and 'valloss' in item]]
    # print(epoch_list[valloss_list.index(min(valloss_list))])
    return(model_list[epoch_list[valloss_list.index(min(valloss_list))]])

### First, let's make sure that all the model is well trained (i.e. they are in their place to do further analysis)

In [18]:
ModelDir = './Sida_temp/for_K8S_training/Output'

In [14]:
os.chdir('D:\OneDrive - HKUST Connect\Research\GWNMMAD\GWAD\Codes')

In [12]:
for i in np.arange(0, 1323, dtype=int):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv2.json doesn't exist!".format(i))

In [86]:
for i in np.arange(0, 1323, dtype=int):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv3.json doesn't exist!".format(i))

In [103]:
for i in np.arange(0, 1323, dtype=int):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTFTv5.json doesn't exist!".format(i))

In [319]:
for i in np.arange(0, 1323, dtype=int):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv9.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTFTv9.json doesn't exist!".format(i))

In [365]:
for i in np.arange(0, 1323, dtype=int):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv8.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTFTv8.json doesn't exist!".format(i))

In [324]:
for i in np.arange(0, 32, dtype=int):
    for j in range(3):
        if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(i,j)):
            print("Model SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json doesn't exist!".format(i,j))

In [ ]:
for i in np.arange(0, 378, dtype=int):
    for j in range(5):
        if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(i,j)):
            print("Model SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json doesn't exist!".format(i,j))

In [23]:
for i in np.arange(0, 2205, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv13.json doesn't exist!".format(i))

In [25]:
for i in np.arange(0, 245, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv14.json doesn't exist!".format(i))

In [10]:
for i in np.arange(0, 245, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv15.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv15.json doesn't exist!".format(i))

In [14]:
for i in np.arange(0, 245, dtype=int):
    for j in range(3):
        if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv15_new_{}.json'.format(i,j)):
            print("Model SeriesWSC_{}_ratio_scan_FFTTTv15_{}.json doesn't exist!".format(i,j))

In [11]:
for i in np.arange(0, 245, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv16.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTFTv16.json doesn't exist!".format(i))

In [42]:
for i in np.arange(0, 2205, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv17.json doesn't exist!".format(i))

In [13]:
for i in np.arange(0, 2205, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv22.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv21.json doesn't exist!".format(i))

In [200]:
for i in np.arange(0, 2205, dtype=int):
    # for j in range(5):
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv24.json'.format(i)):
        print("Model SeriesWSC_{}_ratio_scan_FFTTTv24.json doesn't exist!".format(i))

### Analyze the FPR performance

#### For AE+WSC, FFTTT case

In [46]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list[i] = FPR
    

In [36]:
np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

In [34]:
nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy()

In [40]:
SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

In [41]:
SIG_class_score

In [42]:
SIG_class_score.sort()

In [43]:
SIG_class_score

In [48]:
FPR_list.min()

In [53]:
np.argmin(FPR_list)

In [52]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

In [54]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(np.argmin(FPR_list)),map_location=torch.device('cpu'))

In [55]:
models.keys()

In [56]:
models['cut_vals']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [57]:
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(640),map_location=torch.device('cpu'))

In [58]:
models_ae.keys()

In [77]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [85]:


model_number = 640

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                for dt in range(5):
                    passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
                    
        

In [78]:
dt = 1
iStep = 1

dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)

In [80]:
err_score.sort()

In [81]:
err_score

In [82]:
np.sum(err_score > 0.00203559)

In [83]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*3*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(3), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1

In [84]:
cut_combination[640]

#### For AE, FFTTT case

In [87]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list[i] = FPR
    

In [ ]:
np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

In [ ]:
nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy()

In [ ]:
SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

In [ ]:
SIG_class_score

In [ ]:
SIG_class_score.sort()

In [ ]:
SIG_class_score

In [88]:
FPR_list.min()

In [89]:
np.argmin(FPR_list)

In [90]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

plt.title('FPR distribution for retrained AE FFTTT, LM WSC test data')

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

In [93]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(np.argmin(FPR_list)),map_location=torch.device('cpu'))

In [94]:
models.keys()

In [ ]:
models['cut_vals']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [91]:
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(520),map_location=torch.device('cpu'))

In [92]:
models_ae.keys()

In [98]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [99]:


model_number = 520

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

for dt in range(5):
    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
    passidx = err_score > cutAE[iStep+1]
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
dt = 1
iStep = 1

dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)

In [ ]:
err_score.sort()

In [ ]:
err_score

In [ ]:
np.sum(err_score > 0.00203559)

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*3*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(3), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1

In [ ]:
cut_combination[640]

#### For AE+WSC, FFTFT case

In [104]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list[i] = FPR
    

In [105]:
FPR_list.min()

In [106]:
np.argmin(FPR_list)

In [107]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

plt.title('FPR distribution for retrained AE+WSC AAWAW FFTFT, LM WSC test data')

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

plt.title('FPR distribution for retrained AE FFTTT, LM WSC test data')

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(np.argmin(FPR_list)),map_location=torch.device('cpu'))

In [ ]:
models.keys()

In [ ]:
models['cut_vals']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [ ]:
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(520),map_location=torch.device('cpu'))

In [ ]:
models_ae.keys()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [ ]:


model_number = 520

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

for dt in range(5):
    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
    passidx = err_score > cutAE[iStep+1]
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
dt = 1
iStep = 1

dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)

In [ ]:
err_score.sort()

In [ ]:
err_score

In [ ]:
np.sum(err_score > 0.00203559)

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*3*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(3), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1

In [ ]:
cut_combination[640]

#### Combined study

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAWAW_FFTFT = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAWAW_FFTFT[i] = FPR
    

In [109]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT[i] = FPR
    

In [235]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAWWW_FFTTT = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAWWW_FFTTT[i] = FPR
    

In [115]:
plt.hist(FPR_list_AAWAW_FFTFT, bins = 50, range = (0.375, 0.625), histtype = 'step', label = 'AAWAW_FFTFT (best FPR = {:.5f})'.format(FPR_list_AAWAW_FFTFT.min()))
plt.hist(FPR_list_AAAAA_FFTTT, bins = 50, range = (0.375, 0.625), histtype = 'step', label = 'AAAAA_FFTTT (best FPR = {:.5f})'.format(FPR_list_AAAAA_FFTTT.min()))
plt.hist(FPR_list_AAWWW_FFTTT, bins = 50, range = (0.375, 0.625), histtype = 'step', label = 'AAWWW_FFTTT (best FPR = {:.5f})'.format(FPR_list_AAWWW_FFTTT.min()))

plt.title('FPR distribution for models using ratio cut, LM WSC test data')
plt.legend()

In [237]:
plt.figure(figsize=(10,8))

plt.hist(FPR_list_AAWAW_FFTFT, bins = 50, range = (0.375, 0.625), histtype = 'step', label = 'AAWAW_FFTFT (best FPR = {:.5f})'.format(FPR_list_AAWAW_FFTFT.min()))
plt.hist(FPR_list_AAAAA_FFTTT, bins = 50, range = (0.375, 0.625), histtype = 'step', label = 'AAAAA_FFTTT (best FPR = {:.5f})'.format(FPR_list_AAAAA_FFTTT.min()))
plt.hist(FPR_list_AAWWW_FFTTT, bins = 50, range = (0.375, 0.625), histtype = 'step', label = 'AAWWW_FFTTT (best FPR = {:.5f})'.format(FPR_list_AAWWW_FFTTT.min()))

plt.title('FPR distribution for models using ratio cut, LM WSC test data')
plt.legend()

In [228]:
np.around(cut_combination[1],5)

In [234]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("A+W FPR \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{:.5f} \t {:.5f} \t {} \t {}'.format(FPR_list_AAWAW_FFTFT[sorted_indices[i]],FPR_list_AAAAA_FFTTT[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

In [244]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("AAAAA \t \t AAWAW \t \t AAWWW \t \t \t \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{:.5f} \t {:.5f} \t {:.5f} \t {} \t {} \t {}'.format(FPR_list_AAAAA_FFTTT[sorted_indices[i]], FPR_list_AAWAW_FFTFT[sorted_indices[i]], FPR_list_AAWWW_FFTTT[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      'better' if FPR_list_AAWWW_FFTTT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

In [263]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("ind \t AAAAA \t \t AAWAW \t \t AAWWW \t \t \t \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{:<4d} \t {:.5f} \t {:.5f} \t {:.5f} \t {} \t {} \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT[sorted_indices[i]], FPR_list_AAWAW_FFTFT[sorted_indices[i]], FPR_list_AAWWW_FFTTT[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      'better' if FPR_list_AAWWW_FFTTT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

##### For model numbered 520 (Best one for AAAAA FFTTT), study its flow

In [267]:
model_number = 520

In [ ]:
cut_combination

In [218]:
cut_combination[520]

In [256]:
print(FPR_list_AAAAA_FFTTT[model_number])
print(FPR_list_AAWAW_FFTFT[model_number])
print(FPR_list_AAWWW_FFTTT[model_number])

In [169]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [174]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                if iStep == 2:
                    # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

In [268]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAWWW FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAWWW FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAWWW FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                # if iStep == 2:
                #     # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                #     continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAWWW FFTFT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

##### For model numbered 539 (Best one for AAWAW FFTFT), study its flow

In [257]:
model_number = 539

In [282]:
cut_combination[539]

In [258]:
print(FPR_list_AAAAA_FFTTT[model_number])
print(FPR_list_AAWAW_FFTFT[model_number])
print(FPR_list_AAWWW_FFTTT[model_number])

In [177]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [178]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                if iStep == 2:
                    # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

In [265]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAWWW FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAWWW FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAWWW FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                # if iStep == 2:
                #     # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                #     continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAWWW FFTFT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

##### For model numbered 640 (Best one for AAWWW FFTTT), study its flow

In [260]:
model_number = 640

In [261]:
cut_combination[model_number]

In [246]:
print(FPR_list_AAAAA_FFTTT[model_number])
print(FPR_list_AAWAW_FFTFT[model_number])
print(FPR_list_AAWWW_FFTTT[model_number])

In [247]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [249]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAWAW FFTFT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAWAW FFTFT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAWAW FFTFT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                if iStep == 2:
                    # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAWAW FFTFT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

In [251]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv2.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAWWW FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAWWW FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAWWW FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                # if iStep == 2:
                #     # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                #     continue
                
                # fig = plt.figure()
                # ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAWWW FFTFT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

##### Casual look: How about a controversial cut for Noise and Aggresive cut for Signal?

In [182]:
np.argwhere(np.all(cut_combination == np.array([0.0023    , 0.001525  , 0.05      , 0.1, 0.1]), axis = 1))

In [209]:
(cut_combination[441] == np.array([0.0023    , 0.0015249999999999999  , 0.05      , 0.1, 0.1]))

In [186]:
np.all(cut_combination == np.array([0.0023    , 0.001525  , 0.05      , 0.1, 0.1]), axis = 1).shape

In [200]:
print(cut_combination[489-48])

In [207]:
cut_combination[441][1]

In [210]:
np.argwhere(np.all(cut_combination == np.array([0.0023    , 0.0015249999999999999  , 0.02      , 0.1, 0.1]), axis = 1))

In [202]:
model_number = 441

In [203]:
print(FPR_list_AAAAA_FFTTT[model_number])
print(FPR_list_AAWAW_FFTFT[model_number])

In [204]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [205]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                if iStep == 2:
                    # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

In [211]:
model_number = 490

In [212]:
print(FPR_list_AAAAA_FFTTT[model_number])
print(FPR_list_AAWAW_FFTFT[model_number])

In [213]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [214]:
step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv5.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                    
                    if iStep == 2:
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            else:
                
                if iStep == 2:
                    # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    continue
                
                fig = plt.figure()
                ax = fig.add_subplot(111)
                
                for dt in range(5):
                    score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                    passidx = (score>=0.5)
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(0.5)
                
                ax.set_title('Error distribution for {} WSC, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
plt.show()                
        

##### Classifier fluctuation look

In [281]:
model_number = 539

step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

for suf in range(12):
    print(suf)

    # print('Full analysis for the {}-th model.'.format(model_number))
    # print('Using Liyang set ratio')

    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv7_{}.json'.format(model_number, suf),map_location=torch.device('cpu'))
    models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv7_{}.json'.format(model_number, suf),map_location=torch.device('cpu'))

    foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
    models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
    models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

    models_ae['BBH'] = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")["mixed_202-20-20"].cpu().eval()

    cutAE = models['cut_vals']

    dataset_wsl_separated = {}

    dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
    dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
    dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
    dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
    dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

    num = {}

    num[0] = 5000
    num[1] = 65000
    num[2] = 10000
    num[3] = 10000
    num[4] = 10000

    # print('The cut scheme for this model is {}'.format(models['cut_vals']))

    # print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

    # print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

    for iStep in range(4):
        
        if iStep == 0:
            
            # figH = plt.figure()
            # axH = figH.add_subplot(111)
            # figL = plt.figure()
            # axL = figL.add_subplot(111)
            
            for dt in range(5):
                
                dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
                err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
                passH = err_score_H > cutAE[0]

                dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
                err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
                passL = err_score_L > cutAE[1]

                passidx = np.logical_and(passH, passL)
                dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                
                num[dt] = len(dataset_wsl_separated[dt])
                
            #     axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            #     axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
                
            #     axH.axvline(cutAE[0])
            #     axL.axvline(cutAE[1])
            
            # axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
            # axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
            
            # axH.legend()
            # axL.legend()
            
            print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
            
        else:
        
            # fig = plt.figure()
            # ax = fig.add_subplot(111)
        
            for minor_step in range(2):
                
                if minor_step == 0:
                    
                    for dt in range(5):
                        dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                        err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                        passidx = err_score > cutAE[iStep+1]
                        # print(cutAE[iStep-1])
                        # training_set_ae[iStep] = training_set_ae[iStep][passidx]
                        
                        if iStep == 2:
                            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        
                        num[dt] = np.sum(passidx)
                        
                    #     ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    #     ax.axvline(cutAE[iStep+1])
                    
                    # ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
            
                    # ax.legend()
                    if iStep == 1:   
                        print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                        iStep
                    
                else:
                    
                    if iStep == 2:
                        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        continue
                    
                    # fig = plt.figure()
                    # ax = fig.add_subplot(111)
                    
                    for dt in range(5):
                        score = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()
                        passidx = (score>=0.5)
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        
                        num[dt] = len(dataset_wsl_separated[dt])
                    #     ax.hist(score, bins = 50, range = (0,1), histtype = 'step', density = True, label = str(step2dt[dt]))
                    #     ax.axvline(0.5)
                    
                    # ax.set_title('Error distribution for {} WSC, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
            
                    # ax.legend()
                    if iStep == 1:   
                        print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                        print(np.around(FPR_list[suf],3))
                        
    # plt.show()                
        

In [275]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list = np.empty(12)

for i in np.arange(0, 12, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_539_ratio_scan_FFTFTv7_{}.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list[i] = FPR
    

In [276]:
FPR_list

#### After the shuffling procedure, the AAAAA FFTTT case result 

In [11]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_shuffled = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv9.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_shuffled[i] = FPR
    

In [12]:
FPR_list_AAAAA_FFTTT_shuffled.min()


In [310]:
FPR_list_AAAAA_FFTTT_shuffled.min()

In [311]:
np.argmin(FPR_list_AAAAA_FFTTT_shuffled)

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(np.argmin(FPR_list)),map_location=torch.device('cpu'))

In [ ]:
models.keys()

In [ ]:
models['cut_vals']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [ ]:
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(520),map_location=torch.device('cpu'))

In [ ]:
models_ae.keys()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [313]:


model_number = 628

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv9.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv9.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
dt = 1
iStep = 1

dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)

In [ ]:
err_score.sort()

In [ ]:
err_score

In [ ]:
np.sum(err_score > 0.00203559)

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*3*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(3), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1

In [314]:
cut_combination[628]

In [322]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_shuffled)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 1315):
    print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_shuffled[sorted_indices[i]],
                                      np.around(cut_combination[sorted_indices[i]],4)))

#### After the shuffling procedure, the AAAAA FFFFF case result 

In [341]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFFFF_shuffled = np.empty((32,3))

for i in np.arange(0, 32, dtype=int):
    for j in range(3):
        models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models['last_wsc']
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_AAAAA_FFFFF_shuffled[i][j] = FPR
        
    print(FPR_list_AAAAA_FFFFF_shuffled[i])
    

In [327]:
FPR_list_AAAAA_FFFFF_shuffled.min()


In [ ]:
FPR_list_AAAAA_FFTTT_shuffled.min()

In [330]:
np.argmin(FPR_list_AAAAA_FFTTT_shuffled)

In [328]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

In [329]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(np.argmin(FPR_list)),map_location=torch.device('cpu'))

In [ ]:
models.keys()

In [ ]:
models['cut_vals']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [ ]:
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(520),map_location=torch.device('cpu'))

In [ ]:
models_ae.keys()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [356]:


model_number = 25

retraining_number = 1

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

for idx in range(5):
    print('Full analysis for the {}-th model for the {}-th time.'.format(sorted_indices[idx], retraining_number))
    print('Using Liyang set ratio')

    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(sorted_indices[idx], retraining_number),map_location=torch.device('cpu'))
    models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(sorted_indices[idx], retraining_number),map_location=torch.device('cpu'))

    foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
    models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
    models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()
    foo = torch.load("./Sida_temp/for_K8S_training/Model/noise_AE_freq_new.json")
    models_ae['noise'] = foo["2det_202-40-20"].cpu().eval()
    foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
    models_ae['BBH'] = foo["mixed_202-20-20"].cpu().eval()
    foo = torch.load("./Sida_temp/for_K8S_training/Model/SG_AE_freq_new.json")
    models_ae['SGHF'] = foo["SGHF_2det_48-96_202-40"].cpu().eval()


    cutAE = models['cut_vals']

    dataset_wsl_separated = {}

    dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
    dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
    dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
    dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
    dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

    num = {}

    num[0] = 5000
    num[1] = 65000
    num[2] = 10000
    num[3] = 10000
    num[4] = 10000

    print('The cut scheme for this model is {}'.format(models['cut_vals']))

    print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

    print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

    for iStep in range(4):
        
        if iStep == 0:
            
            for dt in range(5):
                
                dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
                err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
                passH = err_score_H > cutAE[0]

                dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
                err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
                passL = err_score_L > cutAE[1]

                passidx = np.logical_and(passH, passL)
                dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                
                num[dt] = len(dataset_wsl_separated[dt])
            
            print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
            
        else:
        
            for minor_step in range(2):
                
                if minor_step == 0:
                    
                    for dt in range(5):
                        dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                        err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                        passidx = err_score > cutAE[iStep+1]
                        # print(cutAE[iStep-1])
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        
                        num[dt] = np.sum(passidx)
                        
                    print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
                # else:
                    
                #     for dt in range(5):
                #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
                #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        
                #         num[dt] = len(dataset_wsl_separated[dt])
                        
                #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
        
    # dataset_wsl_separated = {}

    # dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
    # dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
    # dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
    # dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
    # dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

    # num = {}

    # num[0] = 5000
    # num[1] = 65000
    # num[2] = 10000
    # num[3] = 10000
    # num[4] = 10000

    # for dt in range(5):
    #     dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        
    #     # err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
    #     passidx = dcd > cutAE[iStep+1]
    #     # print(cutAE[iStep-1])
    #     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
    #     num[dt] = len(dataset_wsl_separated[dt])
        
    # print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                        
        

In [449]:


model_number = 25

retraining_number = 1

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

for idx in range(5):
    print('Full analysis for the {}-th model for the {}-th time.'.format(sorted_indices[idx], retraining_number))
    print('Using Liyang set ratio')

    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(sorted_indices[idx], retraining_number),map_location=torch.device('cpu'))
    models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFFFFv10_{}.json'.format(sorted_indices[idx], retraining_number),map_location=torch.device('cpu'))

    foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
    models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
    models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()
    foo = torch.load("./Sida_temp/for_K8S_training/Model/noise_AE_freq_new.json")
    models_ae['noise'] = foo["2det_202-40-20"].cpu().eval()
    foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
    models_ae['BBH'] = foo["mixed_202-20-20"].cpu().eval()
    foo = torch.load("./Sida_temp/for_K8S_training/Model/SG_AE_freq_new.json")
    models_ae['SGHF'] = foo["SGHF_2det_48-96_202-40"].cpu().eval()


    cutAE = models['cut_vals']

    dataset_wsl_separated = {}

    dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
    dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
    dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
    dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
    dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

    num = {}

    num[0] = 5000
    num[1] = 65000
    num[2] = 10000
    num[3] = 10000
    num[4] = 10000

    print('The cut scheme for this model is {}'.format(models['cut_vals']))

    print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

    print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

    for iStep in range(4):
        
        if iStep == 0:
            
            for dt in range(5):
                
                dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
                err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
                passH = err_score_H > cutAE[0]

                dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
                err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
                passL = err_score_L > cutAE[1]

                passidx = np.logical_and(passH, passL)
                dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                
                num[dt] = len(dataset_wsl_separated[dt])
            
            print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
            
        else:
        
            for minor_step in range(2):
                
                if minor_step == 0:
                    
                    for dt in range(5):
                        dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                        err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                        passidx = err_score > cutAE[iStep+1]
                        # print(cutAE[iStep-1])
                        dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        
                        num[dt] = np.sum(passidx)
                        
                    print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                    
                # else:
                    
                #     for dt in range(5):
                #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
                #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                        
                #         num[dt] = len(dataset_wsl_separated[dt])
                        
                #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
        
    # dataset_wsl_separated = {}

    # dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
    # dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
    # dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
    # dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
    # dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

    # num = {}

    # num[0] = 5000
    # num[1] = 65000
    # num[2] = 10000
    # num[3] = 10000
    # num[4] = 10000

    # for dt in range(5):
    #     dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        
    #     # err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
    #     passidx = dcd > cutAE[iStep+1]
    #     # print(cutAE[iStep-1])
    #     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
    #     num[dt] = len(dataset_wsl_separated[dt])
        
    # print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                        
        

In [363]:
model_number = 31

step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_0.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFFFFv10_0.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/noise_AE_freq_new.json")
models_ae['noise'] = foo["2det_202-40-20"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
models_ae['BBH'] = foo["mixed_202-20-20"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/SG_AE_freq_new.json")
models_ae['SGHF'] = foo["SGHF_2det_48-96_202-40"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [364]:
model_number = 8

step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_0.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFFFFv10_0.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/noise_AE_freq_new.json")
models_ae['noise'] = foo["2det_202-40-20"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
models_ae['BBH'] = foo["mixed_202-20-20"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/SG_AE_freq_new.json")
models_ae['SGHF'] = foo["SGHF_2det_48-96_202-40"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [462]:
model_number = 12

step2dt = {0:'glitch', 1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFFFFv10_0.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFFFFv10_0.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/noise_AE_freq_new.json")
models_ae['noise'] = foo["2det_202-40-20"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
models_ae['BBH'] = foo["mixed_202-20-20"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/SG_AE_freq_new.json")
models_ae['SGHF'] = foo["SGHF_2det_48-96_202-40"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        figH = plt.figure()
        axH = figH.add_subplot(111)
        figL = plt.figure()
        axL = figL.add_subplot(111)
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
            
            axH.hist(err_score_H, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            axL.hist(err_score_L, bins = 50, range = (0,0.0125), histtype = 'step', density = True, label = str(step2dt[dt]))
            
            axH.axvline(cutAE[0])
            axL.axvline(cutAE[1])
        
        axH.set_title('Error distribution for glitch H AE, for {}-th AAAAA FFTTT model'.format(model_number))
        axL.set_title('Error distribution for glitch L AE, for {}-th AAAAA FFTTT model'.format(model_number))
        
        axH.legend()
        axL.legend()
        
        print('After the Glitch AE \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        fig = plt.figure()
        ax = fig.add_subplot(111)
        
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                    ax.hist(err_score, bins = 50, range = (0,0.01), histtype = 'step', density = True, label = str(step2dt[dt]))
                    ax.axvline(cutAE[iStep+1])
                
                ax.set_title('Error distribution for {} AE, for {}-th AAAAA FFTTT model'.format(step2dt[iStep],model_number))
        
                ax.legend()
                    
                print('After the {} AE \t  \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
plt.show()
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [339]:
models.keys()

In [ ]:
dt = 1
iStep = 1

dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)

In [ ]:
err_score.sort()

In [ ]:
err_score

In [ ]:
np.sum(err_score > 0.00203559)

In [463]:
Ncut = 3;
cut_glitchH = np.array([0.0023])
cut_glitchL = np.array([0.0015])
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.5, 0.3, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.array([0.75, 0.25])
cut_SGHF = np.array([0.75, 0.25])

cut_combination = np.empty((1 * 8 * 4, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(1), np.arange(1), np.arange(8), np.arange(2), np.arange(2)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1

In [347]:
cut_combination[31]

In [464]:
cut_combination[12]

In [460]:
sorted_indices = np.argsort(np.mean(FPR_list_AAAAA_FFFFF_shuffled, axis = 1))
# sorted_indices = np.argsort(np.var(FPR_list_AAAAA_FFFFF_shuffled, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank \t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t cut scheme")

for i in range(0, 32):
    print('{} \t {:<4d} \t {} \t \t {:.4f} \t {:.4f} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_AAAAA_FFFFF_shuffled[sorted_indices[i]],4),np.mean(FPR_list_AAAAA_FFFFF_shuffled[sorted_indices[i]]),
                                                                np.sqrt(np.var(FPR_list_AAAAA_FFFFF_shuffled[sorted_indices[i]])),np.around(cut_combination[sorted_indices[i]][2:],4)))

#### After the shuffling process, the AAWAW FFTFT case

In [13]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAWAW_FFTFT = np.empty(1323)

for i in np.arange(0, 1323, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv8.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAWAW_FFTFT[i] = FPR
    

In [14]:
FPR_list_AAWAW_FFTFT.min()

In [470]:
FPR_list_AAWAW_FFTFT.min()

In [368]:
np.argmin(FPR_list_AAWAW_FFTFT)

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

plt.title('FPR distribution for retrained AE+WSC AAWAW FFTFT, LM WSC test data')

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

plt.title('FPR distribution for retrained AE FFTTT, LM WSC test data')

In [ ]:
plt.hist(FPR_list, bins = 50, range = (0.375, 0.625), histtype = 'step')

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(np.argmin(FPR_list)),map_location=torch.device('cpu'))

In [ ]:
models.keys()

In [ ]:
models['cut_vals']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [ ]:
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv3.json'.format(520),map_location=torch.device('cpu'))

In [ ]:
models_ae.keys()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

In [ ]:


model_number = 970

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv3.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
dt = 1
iStep = 1

dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)

In [ ]:
err_score.sort()

In [ ]:
err_score

In [ ]:
np.sum(err_score > 0.00203559)

In [16]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*3*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(3), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1

In [17]:
cut_combination[640]

In [18]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_shuffled)
sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("A+W FPR \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{:.5f} \t {:.5f} \t {} \t {}'.format(FPR_list_AAWAW_FFTFT[sorted_indices[i]],FPR_list_AAAAA_FFTTT_shuffled[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT_shuffled[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

In [471]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_shuffled)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("A+W FPR \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{:.5f} \t {:.5f} \t {} \t {}'.format(FPR_list_AAWAW_FFTFT[sorted_indices[i]],FPR_list_AAAAA_FFTTT_shuffled[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT_shuffled[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

In [473]:
sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT)
sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("A+W FPR \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{} \t {:.5f} \t {:.5f} \t {} \t {}'.format(sorted_indices[i], FPR_list_AAWAW_FFTFT[sorted_indices[i]],FPR_list_AAAAA_FFTTT[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

In [371]:
# sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT)
sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("A+W FPR \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 1323):
    print('{:.5f} \t {:.5f} \t {} \t {}'.format(FPR_list_AAWAW_FFTFT[sorted_indices[i]],FPR_list_AAAAA_FFTTT[sorted_indices[i]],
                                      'better' if FPR_list_AAWAW_FFTFT[sorted_indices[i]]<FPR_list_AAAAA_FFTTT[sorted_indices[i]] else 'worse ',
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

#### WSC performance study, not all model is trained

In [407]:
wsc_training_set_1 = torch.load("../K8S_training/LastWSCtest/Data/wsc_training_set_new_shuffled.json")

with open("../K8S_training/LastWSCtest/Data/filtered_unknown_set_1_shuffled.pickle", 'rb') as handle:
    wsc_training_set_1[4] = pickle.load(handle)

In [409]:
for key in wsc_training_set_1.keys():
    print(wsc_training_set_1[key].shape)

In [408]:
wsc_training_set_1[4].shape

In [376]:
# for i in np.arange(0, 1323, dtype=int):

skipped_indices = [10,11,23,46,47,69,70,71,130,131,196,197,256,257,202,203,214,215,238,239,261,262,263,275,299,322,323]

FPR_list_WSC = np.empty((378,5))

for i in np.arange(0, 378, dtype=int):
    for j in range(5):
        if i in skipped_indices:
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR
    

In [377]:
for i in np.arange(0, 378, dtype=int):
    for j in range(5):
        if not os.path.exists('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}.pt'.format(i,j)):
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR

In [379]:
print(FPR_list_WSC)

In [380]:
hyperparam = {}

hyperparam['struct'] = []
hyperparam['batchsize'] = []
hyperparam['epochs'] = []
hyperparam['learning_rate'] = []

cnt = -1
    
for struct in [[202,64,16,8,5],[202,32,16,8,5],[202,64,16,5],[202,64,8,5],[202,32,16,5],[202,32,8,5]]:
    for (batchsize, epochs) in [(384,400),(384,300),(384,200),(256,300),(256,200),(256,100),(128,300),(128,200),(128,100)]:
        for learning_rate in [1e-4, 5e-5, 2e-5, 1e-5, 5e-6, 2e-6, 1e-6]:
            cnt += 1
            hyperparam['struct'].append(struct)
            hyperparam['batchsize'].append(batchsize)
            hyperparam['epochs'].append(epochs)
            hyperparam['learning_rate'].append(learning_rate)

In [381]:
hyperparam['epochs']

In [422]:
str(hyperparam['struct'][sorted_indices[i]])

In [454]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank\t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")

for i in range(0, 378):
    print('{} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {:<20} \t {} \t {} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      str(hyperparam['struct'][sorted_indices[i]]), hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [412]:
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t \t \t scheme")

for i in range(0, 378):
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [455]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank \t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")
j = 0
for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    j+=1
    print('{:<3d} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(j,sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [405]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [406]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,32,16,8,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [461]:
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]

print(cut_glitchH)
print(cut_glitchL)

del cut_glitchL, cut_glitchH

#### WSC overtrain study, not all model is trained

In [10]:
cnt = 0
    
for struct in [[202,64,16,5]]:
    for (batchsize, epochs) in [(384,300)]:
        for learning_rate in [5e-5]:
            for training_name in ['/wsc_training_set_new_20k_shuffled.json', '/wsc_training_set_new_40k_shuffled.json', '/wsc_training_set_new_enlarged_4_shuffled.json']:
                for testing_name in ['/wsc_test_set_new_LM_same_ratio_10k_shuffled.npy', '/wsc_test_set_new_LM_same_ratio_20k_shuffled.npy', '/wsc_test_set_new_LM_same_ratio_30k_shuffled.npy', '/wsc_test_set_new_LM_increased_noise_20k_shuffled.npy', '/wsc_test_set_new_LM_decreased_noise_20k_shuffled.npy']:

                    # wsc_training_set = torch.load(dataDir+training_name)

                    # wsc_training_set[4] = np.load(dataDir+testing_name)
                    
                    print(cnt, training_name, testing_name)

                    cnt += 1

In [ ]:
wsc_training_set_1 = torch.load("../K8S_training/LastWSCtest/Data/wsc_training_set_new_shuffled.json")

with open("../K8S_training/LastWSCtest/Data/filtered_unknown_set_1_shuffled.pickle", 'rb') as handle:
    wsc_training_set_1[4] = pickle.load(handle)

In [ ]:
for key in wsc_training_set_1.keys():
    print(wsc_training_set_1[key].shape)

In [ ]:
wsc_training_set_1[4].shape

In [13]:
os.getcwd()

In [16]:
dataset_wsl_fft_all_wsl = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_enlarged_4.npy')

In [17]:
dataset_wsl_fft_all_wsl.shape

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

skipped_indices = [10,11,23,46,47,69,70,71,130,131,196,197,256,257,202,203,214,215,238,239,261,262,263,275,299,322,323]

FPR_list_WSC = np.empty((378,5))

for i in np.arange(0, 378, dtype=int):
    for j in range(5):
        if i in skipped_indices:
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR
    

In [21]:
FPR_list_WSC = np.empty((15,5))

for i in np.arange(0, 15, dtype=int):
    for j in range(5):
        if not os.path.exists('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v5.pt'.format(i,j)):
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v5.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[:280000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[280000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR

In [22]:
print(FPR_list_WSC)

In [ ]:
hyperparam = {}

hyperparam['struct'] = []
hyperparam['batchsize'] = []
hyperparam['epochs'] = []
hyperparam['learning_rate'] = []

cnt = -1
    
for struct in [[202,64,16,8,5],[202,32,16,8,5],[202,64,16,5],[202,64,8,5],[202,32,16,5],[202,32,8,5]]:
    for (batchsize, epochs) in [(384,400),(384,300),(384,200),(256,300),(256,200),(256,100),(128,300),(128,200),(128,100)]:
        for learning_rate in [1e-4, 5e-5, 2e-5, 1e-5, 5e-6, 2e-6, 1e-6]:
            cnt += 1
            hyperparam['struct'].append(struct)
            hyperparam['batchsize'].append(batchsize)
            hyperparam['epochs'].append(epochs)
            hyperparam['learning_rate'].append(learning_rate)

In [ ]:
hyperparam['epochs']

In [ ]:
str(hyperparam['struct'][sorted_indices[i]])

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank\t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")

for i in range(0, 378):
    print('{} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {:<20} \t {} \t {} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      str(hyperparam['struct'][sorted_indices[i]]), hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t \t \t scheme")

for i in range(0, 378):
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank \t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")
j = 0
for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    j+=1
    print('{:<3d} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(j,sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,32,16,8,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]

print(cut_glitchH)
print(cut_glitchL)

del cut_glitchL, cut_glitchH

#### WSC overtrain study, not all model is trained (inner code v6)

In [7]:
cnt = 0
    
for struct in [[202,64,16,5]]:
    for (batchsize, epochs) in [(384,300)]:
        for learning_rate in [5e-5]:
            for training_name in ['/wsc_training_set_new_20k_shuffled.json', '/wsc_training_set_new_40k_shuffled.json', '/wsc_training_set_new_enlarged_4_shuffled.json']:
                for testing_name in ['/wsc_test_set_new_LM_same_ratio_10k_shuffled.npy', '/wsc_test_set_new_LM_same_ratio_20k_shuffled.npy', '/wsc_test_set_new_LM_same_ratio_30k_shuffled.npy', '/wsc_test_set_new_LM_increased_noise_20k_shuffled.npy', '/wsc_test_set_new_LM_decreased_noise_20k_shuffled.npy']:

                    # wsc_training_set = torch.load(dataDir+training_name)

                    # wsc_training_set[4] = np.load(dataDir+testing_name)
                    
                    print(cnt, training_name, testing_name)

                    cnt += 1

In [8]:
wsc_training_set_1 = torch.load("../K8S_training/LastWSCtest/Data/wsc_training_set_new_shuffled.json")

with open("../K8S_training/LastWSCtest/Data/filtered_unknown_set_1_shuffled.pickle", 'rb') as handle:
    wsc_training_set_1[4] = pickle.load(handle)

In [9]:
for key in wsc_training_set_1.keys():
    print(wsc_training_set_1[key].shape)

In [10]:
wsc_training_set_1[4].shape

In [ ]:
os.getcwd()

In [12]:
dataset_wsl_fft_all_wsl = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_enlarged_4.npy')

In [13]:
dataset_wsl_fft_all_wsl.shape

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

skipped_indices = [10,11,23,46,47,69,70,71,130,131,196,197,256,257,202,203,214,215,238,239,261,262,263,275,299,322,323]

FPR_list_WSC = np.empty((378,5))

for i in np.arange(0, 378, dtype=int):
    for j in range(5):
        if i in skipped_indices:
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR
    

In [14]:
FPR_list_WSC = np.empty((15,5))

for i in np.arange(0, 15, dtype=int):
    for j in range(5):
        if not os.path.exists('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v6.pt'.format(i,j)):
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v6.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[:280000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[280000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR

In [15]:
print(FPR_list_WSC)

In [ ]:
hyperparam = {}

hyperparam['struct'] = []
hyperparam['batchsize'] = []
hyperparam['epochs'] = []
hyperparam['learning_rate'] = []

cnt = -1
    
for struct in [[202,64,16,8,5],[202,32,16,8,5],[202,64,16,5],[202,64,8,5],[202,32,16,5],[202,32,8,5]]:
    for (batchsize, epochs) in [(384,400),(384,300),(384,200),(256,300),(256,200),(256,100),(128,300),(128,200),(128,100)]:
        for learning_rate in [1e-4, 5e-5, 2e-5, 1e-5, 5e-6, 2e-6, 1e-6]:
            cnt += 1
            hyperparam['struct'].append(struct)
            hyperparam['batchsize'].append(batchsize)
            hyperparam['epochs'].append(epochs)
            hyperparam['learning_rate'].append(learning_rate)

In [ ]:
hyperparam['epochs']

In [ ]:
str(hyperparam['struct'][sorted_indices[i]])

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank\t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")

for i in range(0, 378):
    print('{} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {:<20} \t {} \t {} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      str(hyperparam['struct'][sorted_indices[i]]), hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t \t \t scheme")

for i in range(0, 378):
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank \t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")
j = 0
for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    j+=1
    print('{:<3d} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(j,sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,32,16,8,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]

print(cut_glitchH)
print(cut_glitchL)

del cut_glitchL, cut_glitchH

#### WSC overtrain study, not all model is trained (inner code v7)

In [102]:
cnt = 0
    
for struct in [[202,64,16,5]]:
    for (batchsize, epochs) in [(384,300)]:
        for learning_rate in [5e-5]:
            for training_name in ['/wsc_training_set_new_20k_shuffled.json', '/wsc_training_set_new_40k_shuffled.json', '/wsc_training_set_new_enlarged_4_shuffled.json']:
                for testing_name in ['/wsc_test_set_new_LM_same_ratio_10k_shuffled.npy', '/wsc_test_set_new_LM_same_ratio_20k_shuffled.npy', '/wsc_test_set_new_LM_same_ratio_30k_shuffled.npy', '/wsc_test_set_new_LM_increased_noise_20k_shuffled.npy', '/wsc_test_set_new_LM_decreased_noise_20k_shuffled.npy']:

                    # wsc_training_set = torch.load(dataDir+training_name)

                    # wsc_training_set[4] = np.load(dataDir+testing_name)
                    
                    print(cnt, training_name, testing_name)

                    cnt += 1

In [ ]:
wsc_training_set_1 = torch.load("../K8S_training/LastWSCtest/Data/wsc_training_set_new_shuffled.json")

with open("../K8S_training/LastWSCtest/Data/filtered_unknown_set_1_shuffled.pickle", 'rb') as handle:
    wsc_training_set_1[4] = pickle.load(handle)

In [ ]:
for key in wsc_training_set_1.keys():
    print(wsc_training_set_1[key].shape)

In [ ]:
wsc_training_set_1[4].shape

In [104]:
os.getcwd()

In [105]:
dataset_wsl_fft_all_wsl = np.load('../K8S_training/LastWSCtest/Data/wsc_test_set_new_LM_enlarged_4_1.npy')

In [14]:
dataset_wsl_fft_all_wsl.shape

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

skipped_indices = [10,11,23,46,47,69,70,71,130,131,196,197,256,257,202,203,214,215,238,239,261,262,263,275,299,322,323]

FPR_list_WSC = np.empty((378,5))

for i in np.arange(0, 378, dtype=int):
    for j in range(5):
        if i in skipped_indices:
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR
    

In [113]:
FPR_list_WSC = np.empty((15,1))

for i in np.arange(0, 15, dtype=int):
    for j in range(1):
        if not os.path.exists('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v7.pt'.format(i,j)):
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v7.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[:280000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[280000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR

In [117]:
FPR_list_WSC = np.empty((15,1))

for i in np.arange(0, 15, dtype=int):
    for j in range(1):
        if not os.path.exists('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v7.pt'.format(i,j)):
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v7.pt'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[:280000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[280000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR

In [17]:
FPR_list_WSC = np.empty((5,1))

for i in np.arange(0, 5, dtype=int):
    for j in range(1):
        if not os.path.exists('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v8.pt'.format(i+10,j)):
            
            FPR_list_WSC[i][j] = np.nan
            continue
            
        models = torch.load('../K8S_training/LastWSCtest/Output/WSCmodel_{}_retraintime{}_v8.pt'.format(i+10,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[:280000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_wsl[280000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_WSC[i][j] = FPR

In [18]:
print(np.around(FPR_list_WSC,4))

In [ ]:
hyperparam = {}

hyperparam['struct'] = []
hyperparam['batchsize'] = []
hyperparam['epochs'] = []
hyperparam['learning_rate'] = []

cnt = -1
    
for struct in [[202,64,16,8,5],[202,32,16,8,5],[202,64,16,5],[202,64,8,5],[202,32,16,5],[202,32,8,5]]:
    for (batchsize, epochs) in [(384,400),(384,300),(384,200),(256,300),(256,200),(256,100),(128,300),(128,200),(128,100)]:
        for learning_rate in [1e-4, 5e-5, 2e-5, 1e-5, 5e-6, 2e-6, 1e-6]:
            cnt += 1
            hyperparam['struct'].append(struct)
            hyperparam['batchsize'].append(batchsize)
            hyperparam['epochs'].append(epochs)
            hyperparam['learning_rate'].append(learning_rate)

In [ ]:
hyperparam['epochs']

In [ ]:
str(hyperparam['struct'][sorted_indices[i]])

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank\t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")

for i in range(0, 378):
    print('{} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {:<20} \t {} \t {} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      str(hyperparam['struct'][sorted_indices[i]]), hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t \t \t scheme")

for i in range(0, 378):
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
sorted_indices = np.argsort(np.sqrt(np.nanvar(FPR_list_WSC, axis = 1)))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank \t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t \t \t scheme")
j = 0
for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    j+=1
    print('{:<3d} \t {:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(j,sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,64,16,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
sorted_indices = np.argsort(np.nanmean(FPR_list_WSC, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t  \t \t scheme")

for i in range(0, 378):
    if not (hyperparam['struct'][sorted_indices[i]] == [202,32,16,8,5]):
        continue
    print('{:<4d} \t {} \t {:.4f} \t {:.4f} \t {} \t {} \t {} \t {}'.format(sorted_indices[i], np.around(FPR_list_WSC[sorted_indices[i]],4),np.nanmean(FPR_list_WSC[sorted_indices[i]]), np.sqrt(np.nanvar(FPR_list_WSC[sorted_indices[i]])),
                                      hyperparam['struct'][sorted_indices[i]], hyperparam['batchsize'][sorted_indices[i]], hyperparam['epochs'][sorted_indices[i]], hyperparam['learning_rate'][sorted_indices[i]]))

In [ ]:
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]

print(cut_glitchH)
print(cut_glitchL)

del cut_glitchL, cut_glitchH

#### Enlarged WSC set study (inner code: v13)

In [73]:
dataset_wsl_fft_all.shape

In [74]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,5))

for i in np.arange(0, 2205, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [22]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [75]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [23]:
np.argmin(FPR_list_AAAAA_FFTTT_enlarged)

In [105]:
model_number = 1614

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [104]:
model

In [107]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [27]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

In [270]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

In [271]:
model = models['last_wsc']

In [95]:
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(0))

In [103]:
for i in ['bbh', 'lfsg', 'hfsg', 'ccsn']:
    for j in ['5-12', '12-24', '24-48', '48-96']:
        print(i,j)
        print(np.sum(np.logical_and(blind_test_set_ans[:,1] == i, blind_test_set_ans[:,2] == j)))

In [110]:
print(np.sum(np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,3] == '4')))

In [277]:
FPR_list = np.empty(100)

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [273]:
blind_test_set_ans

In [280]:
sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(18)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg', 'ccsn']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [159]:
blind_test_set.shape

In [160]:
Noise_score.shape

In [164]:
Noise_score

In [165]:
Signal_score

In [29]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [43]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [16]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [64]:
print(blind_test_set_ans)

##### print the score distribution of different types

In [126]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [39]:
len(blind_test_set)

In [41]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [38]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [25]:
Noise_score

In [26]:
Signal_score

In [21]:
np.sum(noise_idx)

In [22]:
np.sum(signal_idx)

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [58]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [18]:
dataset_wsl_fft_all.shape

In [19]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [64]:
datatype

In [20]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [21]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [117]:
dataset_wsl_separated.keys()

In [22]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [72]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [24]:
np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx)).shape

In [25]:
blind_test_set.shape

In [27]:
np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx)).shape

In [269]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [92]:
idx = 1000

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(0))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(0))

# blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(0))
# blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(0))
                
blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)


In [93]:
selected_idx = np.logical_and(blind_test_set_ans[:,1] == 'bbh', blind_test_set_ans[:,2] == '5-12')
np.sum(selected_idx)

In [94]:
blind_test_set_ans

In [95]:
blind_test_set_ans[selected_idx]

In [96]:
blind_test_set[selected_idx][idx].shape

In [97]:
idx = 600
plt.plot(blind_test_set[selected_idx][idx][0])


In [98]:
plt.plot(blind_test_set[selected_idx][idx][1])

In [68]:
idx = 400
plt.plot(blind_test_set[selected_idx][idx][0])
plt.plot(blind_test_set[selected_idx][idx][1])

In [57]:
plt.plot(blind_test_set[selected_idx][idx][0])

In [58]:
plt.plot(blind_test_set[selected_idx][idx][1])

In [47]:
Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

In [49]:
Score

In [48]:
plt.hist(Score)

In [268]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [267]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [90]:
os.getcwd()

In [104]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [105]:
dataset_cropped_all.shape

In [101]:
datatype_list

In [108]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [89]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [80]:
dataset_wsl_separated = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [83]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [94]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [182]:
# Set for training the model
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [87]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [13]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [16]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [78]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [19]:
np.random.choice(2,100)

In [82]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [24]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [69]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [70]:
test_sig

In [71]:
test_noise

In [72]:
print(detector_pick)

In [73]:
events_pick

In [74]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [62]:
test_sig[events_pick,detector_pick,:]

In [75]:
test_sig

In [54]:
test_noise

In [60]:
test_sig

In [84]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [88]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

#### Enlarged WSC set study, fixed error function (inner code: v13_new)

In [374]:
dataset_wsl_fft_all.shape

In [375]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,5))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 5)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [376]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [377]:
np.argmin(FPR_list_AAAAA_FFTTT_enlarged)

In [213]:
FPR_list_AAAAA_FFTTT_passing_number[1558]

In [206]:
model_number = 1558

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13_new.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv13_new.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [207]:
SIG_class_score.shape

In [208]:
np.sum(BKG_class_score > threshold) / 70000

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

###### With long dataset

In [321]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [322]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [281]:
data[58]

In [323]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [324]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [325]:
columns

In [326]:
df.columns = columns

In [327]:
df

In [328]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [329]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [330]:
df

In [331]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [293]:
df

In [332]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [333]:
df

In [334]:
df['total_numbers'] = total_numbers

In [335]:
df

In [336]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [337]:
excel_file = '../Pic_cached/output_l_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [300]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With long dataset, fix the events size

In [417]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [390]:
plt.hist(FPR_list, bins = 20)

In [391]:
np.mean(FPR_list)

In [392]:
np.var(FPR_list)

In [418]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [419]:
plt.hist(FPR_list, bins = 20)

In [339]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [437]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, fix the events size

In [443]:
noise_set = dataset_wsl_separated_leftover['bg']

In [448]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [447]:
plt.hist(FPR_list, bins = 20)

In [444]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [445]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [460]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [461]:
ccsn_rawset.shape

In [462]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [463]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [465]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [466]:
ccsn_rawset.shape

In [467]:
noise_set.shape

In [469]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset

In [301]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [302]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [303]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [304]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [305]:
columns

In [306]:
df.columns = columns

In [307]:
df

In [312]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [313]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [314]:
df

In [315]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [316]:
df

In [317]:
df['total_numbers'] = total_numbers

In [318]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [319]:
df

In [320]:
excel_file = '../Pic_cached/output_s_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With old blackbox set, fix the events size

In [450]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:100], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [470]:
np.load('../Data_cached/collected_cropped_sig_1_cut.npy').shape

In [478]:
noise_set = dataset_wsl_separated_leftover['bg']

In [479]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(5):
blind_test_set = np.load('../Data_cached/collected_cropped_sig_1_cut.npy')[:3]
# blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(3,4,1000,202)

noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))
for type in range(3):
    for snr in range(4):
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, blind_test_set_ffted[type, snr], axis = 0)
        
print(signal_set.shape)
print(noise_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [18]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [19]:
plt.hist(FPR_list, bins = 20)

In [20]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [21]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [22]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [23]:
columns

In [24]:
df.columns = columns

In [25]:
df

In [26]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [27]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [28]:
df

In [29]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [30]:
df

In [31]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [32]:
df

In [33]:
df['total_numbers'] = total_numbers

In [34]:
df

In [35]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [36]:
excel_file = '../Pic_cached/output_l_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [14]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset, all from 2nd set, timewindow corrected

In [37]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_{}_v3s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [38]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [39]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [40]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [41]:
df.columns = columns

In [42]:
df

In [43]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [44]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [45]:
df

In [46]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [47]:
df

In [48]:
df['total_numbers'] = total_numbers

In [49]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [50]:
df

In [51]:
excel_file = '../Pic_cached/output_s_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With remaining test set

In [368]:
dataset_wsl_separated_leftover.keys()

In [371]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

noise_set = dataset_wsl_separated_leftover['bg']

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [405]:
noise_set.shape

In [403]:
dataset_wsl_separated_leftover.keys()

In [404]:
noise_set = dataset_wsl_separated_leftover['bg']

In [420]:
perm = np.random.choice(1000000, 65000, replace=False)

noise_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)
noise_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)

noise_set = np.concatenate((noise_L, noise_H), axis = 1)

noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

In [421]:
noise_set_fft.shape

In [422]:
noise_set = noise_set_fft.reshape(-1,202)

In [473]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_1.npy')

In [474]:
noise_set.shape

In [475]:
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)

In [476]:
noise_set.shape

In [401]:
# Using noise in blackbox

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [431]:
# Using noise in blackbox, newly extracted set 1

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [436]:
# Using noise in blackbox, newly extracted set 2

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [407]:
# Using noise in reserved

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [411]:
# Using 10k noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [416]:
# Using noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [412]:
# Using 65k noise in test

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [423]:
# Using 65k noise just extracted

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [424]:
dataset_wsl_separated_leftover.keys()

In [425]:
# All reserved set, randomly extract 1000

FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # # noise_idx = (blind_test_set_ans[:,0] == '0')
    # noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['BBH', 'SGHF', 'SGLF']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = np.random.choice(10000,1000,replace=False)
            signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr][signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [426]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With initial test set

In [382]:
dataset_wsl_separated.keys()

In [387]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

# noise_set = dataset_wsl_fft_all[:70000]
noise_set = dataset_wsl_separated[1]

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [110]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [111]:
dataset_wsl_separated.keys()

In [112]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [348]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [364]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [365]:
noise_set.shape

In [366]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [352]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [367]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [357]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [113]:
# A not careful plot, using noise from v3l and signal from loud_v3l

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [103]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [59]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [11]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [12]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [361]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [362]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# Set for training the model
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

#### Enlarged WSC set study, with glitch specified to same time segment (inner code: v13)

In [65]:
dataset_wsl_fft_all.shape

In [71]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

for i in np.arange(0, 2205, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    

In [72]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [73]:
np.argmin(FPR_list_AAAAA_FFTTT_enlarged)

In [70]:
np.set_printoptions(precision=8)

model_number = 1614

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [69]:
np.set_printoptions(precision=8)

model_number = 1614

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000


model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

# FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

# FPR_list_AAAAA_FFTTT_enlarged[i] = FPR

for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))


              
        

In [25]:
Ncut = 3;
cut_glitchH = np.zeros(3)[1:-1]
cut_glitchL = np.zeros(3)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((1*1*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      np.around(cut_combination[sorted_indices[i]],4)))

#### Enlarged WSC set study, using loss function type2 (inner code: v17)

In [98]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i)):
        
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan 
        continue
    
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)
    
    SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    

In [31]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i)):
        
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan 
        continue
    
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:70000])).detach().numpy(), axis = 1)
    
    SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[70000:])).detach().numpy(), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    

In [99]:
models['last_wsc'](torch.FloatTensor(dataset_wsl_fft_all[:70000])).shape

In [100]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [101]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [33]:
print(np.nanmin(FPR_list_AAAAA_FFTTT_enlarged))
print(np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged))

In [43]:
os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(729))

In [95]:
model_number = 414

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000


model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy()), axis = 1)

SIG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy()), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

# FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

# FPR_list_AAAAA_FFTTT_enlarged[i] = FPR

for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy()), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [96]:
dataset_wsl_fft_all_newglitch.shape

In [97]:
dataset_wsl_separated_newglitch

In [98]:
key2label = ['glitch', 'noise', 'bbh', 'sghf', 'sglf']

In [99]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

for key in dataset_wsl_separated.keys():
    print(dataset_wsl_separated[key].shape)
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy(), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

##### We found that there's one more Sigmoid layer. Retrain it.  

In [43]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv18.json'.format(414),map_location=torch.device('cpu'))
    
# print(models.keys())

model = models['last_wsc']

BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)

SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)

In [44]:
for key in dataset_wsl_separated_newglitch.keys():
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated_newglitch[key]))).detach().numpy(), density = True, histtype='step', bins = 20, label = key)
plt.legend()

In [45]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(414),map_location=torch.device('cpu'))
    
# print(models.keys())

model = models['last_wsc']

BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)

SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)
for key in dataset_wsl_separated_newglitch.keys():
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated_newglitch[key]))).detach().numpy(), density = True, histtype='step', bins = 20, label = key)
plt.legend()

In [46]:
models

In [47]:
models_copytest = {}
models_copytest['test'] = copy.deepcopy(models)

In [48]:
models_copytest['test']

In [50]:
models_copytest['test'].keys()

In [51]:
200 % 20 == 0

#### Enlarged WSC set study, using loss function type2，40k wsc training, new glitch (inner code: v23)

In [28]:
with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv23.pickle'.format(1), 'rb') as handle:
    model = pickle.load(handle)

In [29]:
model.keys()

In [30]:
for epochs in range(120,500,20):
    print(model[str(epochs) + 'valloss'])

In [26]:
os.getcwd()

In [32]:
os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv23.json'.format(1))

In [48]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

epochs_list = range(120,501,20)

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv23.pickle'.format(i)):
        
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan 
        continue
    
    # models = torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv19.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    # model = models['last_wsc']
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv23.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    valloss = np.empty(0)
    
    for k in range(len(epochs_list)):
        valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
        
    model = model_list[epochs_list[np.argmin(valloss)]]
    
    
    BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:140000])).detach().numpy(), axis = 1)
    
    SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[140000:])).detach().numpy(), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    # print(FPR)
    # print(i)
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    # print(FPR_list_AAAAA_FFTTT_enlarged[i])
    

In [49]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [50]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [43]:
FPR_list_AAAAA_FFTTT_enlarged[1] = 0.1

In [44]:
FPR_list_AAAAA_FFTTT_enlarged

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i)):
        
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan 
        continue
    
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:70000])).detach().numpy(), axis = 1)
    
    SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[70000:])).detach().numpy(), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    

In [ ]:
models['last_wsc'](torch.FloatTensor(dataset_wsl_fft_all[:70000])).shape

In [ ]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
print(np.nanmin(FPR_list_AAAAA_FFTTT_enlarged))
print(np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged))

In [ ]:
os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(729))

In [54]:
model_number = 1805

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv23.pickle'.format(i), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
    
model = model_list[epochs_list[np.argmin(valloss)]]

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:10000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[10000:140000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[140000:160000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[160000:180000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[180000:].copy()       # sghf

num = {}

num[0] = 10000
num[1] = 130000
num[2] = 20000
num[3] = 20000
num[4] = 20000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:10000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[10000:140000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[140000:160000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[160000:180000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[180000:].copy()       # sghf

num = {}

num[0] = 10000
num[1] = 130000
num[2] = 20000
num[3] = 20000
num[4] = 20000

    
BKG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:140000]))).detach().numpy()), axis = 1)

SIG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[140000:]))).detach().numpy()), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

# FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

# FPR_list_AAAAA_FFTTT_enlarged[i] = FPR

for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy()), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
dataset_wsl_fft_all_newglitch.shape

In [ ]:
dataset_wsl_separated_newglitch

In [ ]:
key2label = ['glitch', 'noise', 'bbh', 'sghf', 'sglf']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

for key in dataset_wsl_separated.keys():
    print(dataset_wsl_separated[key].shape)
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy(), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

##### We found that there's one more Sigmoid layer. Retrain it.  

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv18.json'.format(414),map_location=torch.device('cpu'))
    
# print(models.keys())

model = models['last_wsc']

BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)

SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)

In [ ]:
for key in dataset_wsl_separated_newglitch.keys():
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated_newglitch[key]))).detach().numpy(), density = True, histtype='step', bins = 20, label = key)
plt.legend()

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(414),map_location=torch.device('cpu'))
    
# print(models.keys())

model = models['last_wsc']

BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)

SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)
for key in dataset_wsl_separated_newglitch.keys():
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated_newglitch[key]))).detach().numpy(), density = True, histtype='step', bins = 20, label = key)
plt.legend()

In [ ]:
models

In [ ]:
models_copytest = {}
models_copytest['test'] = copy.deepcopy(models)

In [ ]:
models_copytest['test']

In [ ]:
models_copytest['test'].keys()

In [ ]:
200 % 20 == 0

#### Enlarged WSC set study, using loss function type2，80k wsc training, new glitch (inner code: v24)

In [206]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

epochs_list = range(120,501,20)

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv24.pickle'.format(i)):
        
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan 
        continue
    
    # models = torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv19.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    # model = models['last_wsc']
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv24.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    valloss = np.empty(0)
    
    for k in range(len(epochs_list)):
        valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
        
    model = model_list[epochs_list[np.argmin(valloss)]]
    
    
    BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:140000])).detach().numpy(), axis = 1)
    
    SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[140000:])).detach().numpy(), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    # print(FPR)
    # print(i)
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    # print(FPR_list_AAAAA_FFTTT_enlarged[i])
    

In [207]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [208]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged[1] = 0.1

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i)):
        
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan 
        continue
    
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:70000])).detach().numpy(), axis = 1)
    
    SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[70000:])).detach().numpy(), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    

In [ ]:
models['last_wsc'](torch.FloatTensor(dataset_wsl_fft_all[:70000])).shape

In [ ]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
print(np.nanmin(FPR_list_AAAAA_FFTTT_enlarged))
print(np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged))

In [ ]:
os.path.exists(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(729))

In [ ]:
model_number = 1805

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv23.pickle'.format(i), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
    
model = model_list[epochs_list[np.argmin(valloss)]]

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:10000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[10000:140000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[140000:160000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[160000:180000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[180000:].copy()       # sghf

num = {}

num[0] = 10000
num[1] = 130000
num[2] = 20000
num[3] = 20000
num[4] = 20000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:10000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[10000:140000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[140000:160000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[160000:180000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[180000:].copy()       # sghf

num = {}

num[0] = 10000
num[1] = 130000
num[2] = 20000
num[3] = 20000
num[4] = 20000

    
BKG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:140000]))).detach().numpy()), axis = 1)

SIG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[140000:]))).detach().numpy()), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

# FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

# FPR_list_AAAAA_FFTTT_enlarged[i] = FPR

for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy()), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
dataset_wsl_fft_all_newglitch.shape

In [ ]:
dataset_wsl_separated_newglitch

In [ ]:
key2label = ['glitch', 'noise', 'bbh', 'sghf', 'sglf']

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

for key in dataset_wsl_separated.keys():
    print(dataset_wsl_separated[key].shape)
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy(), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

##### We found that there's one more Sigmoid layer. Retrain it.  

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv18.json'.format(414),map_location=torch.device('cpu'))
    
# print(models.keys())

model = models['last_wsc']

BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)

SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)

In [ ]:
for key in dataset_wsl_separated_newglitch.keys():
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated_newglitch[key]))).detach().numpy(), density = True, histtype='step', bins = 20, label = key)
plt.legend()

In [ ]:
models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv17.json'.format(414),map_location=torch.device('cpu'))
    
# print(models.keys())

model = models['last_wsc']

BKG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[:70000])).detach().numpy(), axis = 1)

SIG_class_score = np.sum(model(torch.FloatTensor(dataset_wsl_fft_all[70000:])).detach().numpy(), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)
for key in dataset_wsl_separated_newglitch.keys():
    plt.hist(nn.Sigmoid()(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated_newglitch[key]))).detach().numpy(), density = True, histtype='step', bins = 20, label = key)
plt.legend()

In [ ]:
models

In [ ]:
models_copytest = {}
models_copytest['test'] = copy.deepcopy(models)

In [ ]:
models_copytest['test']

In [ ]:
models_copytest['test'].keys()

In [ ]:
200 % 20 == 0

#### Supervised study of WSC with loss function type 2

In [13]:
os.getcwd()

In [12]:
ModelDir_LastWSC = "../K8S_training/LastWSCtest/Output"

In [80]:
FPR_list = np.empty((4,1))
epochs_list = range(120,501,20)


for i in range(4):
    for j in range(1):
        # ModelDir_LastWSC = "../K8S_training/LastWSCtest/Output"
        model_list = torch.load(ModelDir_LastWSC + "/WSCmodel_{}_retraintime{}_v11.pt".format(i,j), map_location='cpu')
        
        val_loss = np.empty(0)
        for k in range(len(range(120,501,20))):
            val_loss = np.append(val_loss, model_list[str(epochs_list[k])+'valloss'])
        model = model_list[epochs_list[np.argmin(val_loss)]]
        # model = model_list[500]
        # print(epochs_list[np.argmin(val_loss)])
        
        BKG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_enlarged[:280000]))).detach().numpy()), axis = 1)
    
        SIG_class_score = np.sum((nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_enlarged[280000:]))).detach().numpy()), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list[i][j] = FPR
        

In [75]:
BKG_class_score

In [76]:
SIG_class_score

In [77]:
threshold

In [78]:
plt.hist(BKG_class_score)
plt.hist(SIG_class_score)

In [81]:
FPR_list

In [51]:
model.keys()

In [52]:
model[120]

#### Without glitch study (inner code: v14)

In [64]:
# for i in np.arange(0, 1323, dtype=int):

total_model_numbers = 245

FPR_list_AAAAA_FFTTT_withoutglitch = np.empty(total_model_numbers)

for i in np.arange(0, total_model_numbers, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_withoutglitch[i] = FPR
    

In [88]:
# for i in np.arange(0, 1323, dtype=int):

total_model_numbers = 245

FPR_list_AAAAA_FFTTT_withoutglitch = np.empty((total_model_numbers))

for i in np.arange(0, total_model_numbers, dtype=int):
    # for j in range(3):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all_newglitch[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_withoutglitch[i] = FPR
    
    print(FPR_list_AAAAA_FFTTT_withoutglitch[i])
      

In [36]:
# for i in np.arange(0, 1323, dtype=int):

total_model_numbers = 245

FPR_list_AAAAA_FFTTT_withoutglitch = np.empty(245)

for i in np.arange(0, 245, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_withoutglitch[i] = FPR
    

In [37]:
FPR_list_AAAAA_FFTTT_withoutglitch.min()

In [89]:
FPR_list_AAAAA_FFTTT_withoutglitch.min()

In [90]:
np.argmin(FPR_list_AAAAA_FFTTT_withoutglitch)

In [82]:
model_number = 14

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        continue
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > 0

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > 0

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000


model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

# FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

# FPR_list_AAAAA_FFTTT_enlarged[i] = FPR

for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [91]:
model_number = 14

np.set_printoptions()

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_newglitch[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all_newglitch[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all_newglitch[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all_newglitch[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all_newglitch[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

np.set_printoptions(formatter={'float': '{: 0.2f}'.format})

for iStep in range(4):
    
    if iStep == 0:
        
        continue
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > 0

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > 0

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
model_number = 14

np.set_printoptions()

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv14.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))



for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > 0

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > 0

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [84]:
np.set_printoptions(formatter={'float': '{: 0.2f}'.format})

In [85]:
Ncut = 3;
cut_glitchH = np.zeros(3)[1:-1]
cut_glitchL = np.zeros(3)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((1*1*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(1), np.arange(1), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_withoutglitch)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 245):
    print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]],
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

#### Without glitch study, a more strict one (inner code: v15)

In [9]:
# for i in np.arange(0, 1323, dtype=int):

total_model_numbers = 245

FPR_list_AAAAA_FFTTT_withoutglitch = np.empty(total_model_numbers)

for i in np.arange(0, total_model_numbers, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv15.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_withoutglitch[i] = FPR
    

In [16]:
FPR_list_AAAAA_FFTTT_withoutglitch.min()

In [18]:
np.argmin(FPR_list_AAAAA_FFTTT_withoutglitch)

In [11]:
model_number = 34

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv15.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv15.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(1,5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > 0

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > 0

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(1,5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

                    
        

In [24]:
Ncut = 3;
cut_glitchH = np.zeros(3)[1:-1]
cut_glitchL = np.zeros(3)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((1*1*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(1), np.arange(1), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_withoutglitch)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 245):
    print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]],
                                      np.around(cut_combination[sorted_indices[i]],4)))

#### Without glitch study, a more strict one. The epoch is raised to 300 (inner code: v15_new)

In [56]:
# for i in np.arange(0, 1323, dtype=int):

total_model_numbers = 245

FPR_list_AAAAA_FFTTT_withoutglitch = np.empty((total_model_numbers,3))

for i in np.arange(0, total_model_numbers, dtype=int):
    for j in range(3):
        models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv15_new_{}.json'.format(i,j),map_location=torch.device('cpu'))
        
        # print(models.keys())
        
        model = models['last_wsc']
        
        BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
        
        SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
        
        SIG_class_score.sort()
        
        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
        
        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list_AAAAA_FFTTT_withoutglitch[i][j] = FPR
        
    print(FPR_list_AAAAA_FFTTT_withoutglitch[i])
      

In [57]:
np.nanmean(FPR_list_AAAAA_FFTTT_withoutglitch, axis = 1).min()

In [58]:
np.argmin(np.nanmean(FPR_list_AAAAA_FFTTT_withoutglitch, axis = 1))

In [78]:
model_number = 93

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv15_new_1.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv15_new_1.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(1,5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > 0

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > 0

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(1,5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
                    
        

In [45]:
Ncut = 3;
cut_glitchH = np.zeros(3)[1:-1]
cut_glitchL = np.zeros(3)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((1*1*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(1), np.arange(1), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


# sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_withoutglitch)
# # sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# # sorted_indices = np.arange(1323)

# # print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
# print("idx \t A FPR \t \t \t \t \t cut scheme")

# for i in range(0, 245):
#     print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]],
#                                     np.around(cut_combination[sorted_indices[i]],4)))
    
sorted_indices = np.argsort(np.mean(FPR_list_AAAAA_FFTTT_withoutglitch, axis = 1))
# sorted_indices = np.argsort(np.var(FPR_list_AAAAA_FFFFF_shuffled, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank \t idx \t A FPR \t \t \t \t \t \t \t \t \t \t \t \t \t cut scheme")

for i in range(0, 32):
    print('{} \t {:<4d} \t {} \t \t {:.4f} \t {:.4f} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]],4),np.mean(FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]]),
                                                                np.sqrt(np.var(FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]])),np.around(cut_combination[sorted_indices[i]][2:],4)))

In [50]:
Ncut = 3;
cut_glitchH = np.zeros(3)[1:-1]
cut_glitchL = np.zeros(3)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((1*1*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(1), np.arange(1), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


# sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_withoutglitch)
# # sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# # sorted_indices = np.arange(1323)

# # print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
# print("idx \t A FPR \t \t \t \t \t cut scheme")

# for i in range(0, 245):
#     print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]],
#                                     np.around(cut_combination[sorted_indices[i]],4)))
    
sorted_indices = np.argsort(np.mean(FPR_list_AAAAA_FFTTT_withoutglitch, axis = 1))
# sorted_indices = np.argsort(np.var(FPR_list_AAAAA_FFFFF_shuffled, axis = 1))
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("rank idx \t FPR \t \t \t \t \t \t \t mean \t \t err\t \t \t cut scheme")

for i in range(0, 32):
    print('{} \t {:<4d} \t {} \t \t {:.4f} \t {:.4f} \t {}'.format(i+1, sorted_indices[i], np.around(FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]],4),np.mean(FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]]),
                                                                np.sqrt(np.var(FPR_list_AAAAA_FFTTT_withoutglitch[sorted_indices[i]])),np.around(cut_combination[sorted_indices[i]][2:],2)))

#### Without glitch study, a more strict one, with WSC (inner code: v16)

In [59]:
# for i in np.arange(0, 1323, dtype=int):

total_model_numbers = 245

FPR_list_AAWAW_FFTFT_withoutglitch = np.empty(total_model_numbers)

for i in np.arange(0, total_model_numbers, dtype=int):
    models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv16.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    model = models['last_wsc']
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAWAW_FFTFT_withoutglitch[i] = FPR
    

In [60]:
FPR_list_AAWAW_FFTFT_withoutglitch.min()

In [61]:
np.argmin(FPR_list_AAWAW_FFTFT_withoutglitch)

In [80]:
model_number = 169

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTFTv16.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTFTv16.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()
foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
models_ae['BBH'] = foo["mixed_202-20-20"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

# num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {}'.format(num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        continue
        
        for dt in range(1,5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > 0

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > 0

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(1,5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {}'.format(step2dt[iStep], num[1], num[2], num[3], num[4]))
                
            else:
                
                if iStep == 2:
                    continue
                
                for dt in range(1,5):
                    passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = len(dataset_wsl_separated[dt])
                    
                print('After the {} WSC \t  \t  {} \t {} \t {} \t {}'.format(step2dt[iStep], num[1], num[2], num[3], num[4]))
    
# dataset_wsl_separated = {}

# dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
# dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
# dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
# dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# num = {}

# num[0] = 5000
# num[1] = 65000
# num[2] = 10000
# num[3] = 10000
# num[4] = 10000

# for dt in range(5):
#     dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
#     passidx = err_score > cutAE[iStep+1]
#     # print(cutAE[iStep-1])
#     dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     num[dt] = len(dataset_wsl_separated[dt])
    
# print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t {} \t {} \t {} \t {}'.format(num[1], num[2], num[3], num[4]))              
        

In [55]:
Ncut = 3;
cut_glitchH = np.zeros(3)[1:-1]
cut_glitchL = np.zeros(3)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((1*1*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(1), np.arange(1), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT_withoutglitch)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 245):
    print('{:<4d} \t {:.5f} \t \t {}'.format(sorted_indices[i], FPR_list_AAWAW_FFTFT_withoutglitch[sorted_indices[i]],
                                      np.around(cut_combination[sorted_indices[i]][2:],2)))

#### Enlarged WSC set study, include CCSN (inner code: v25)

In [129]:
dataset_wsl_fft_withccsn.shape

In [143]:
# for i in np.arange(0, 1323, dtype=int):

epochs_list = range(120,501,20)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:10000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[10000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,6))

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv25.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 6)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv25.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    valloss = np.empty(0)
    
    for k in range(len(epochs_list)):
        valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
        
    model = model_list[epochs_list[np.argmin(valloss)]]
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[70000:100000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(6):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [134]:
# for i in np.arange(0, 1323, dtype=int):

epochs_list = range(120,501,20)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:10000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[10000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,6))

for i in np.arange(0, 2205, dtype=int):
    
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv25.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 6)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv25.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    valloss = np.empty(0)
    
    for k in range(len(epochs_list)):
        valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
        
    model = model_list[epochs_list[np.argmin(valloss)]]
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(6):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [144]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [145]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [140]:
# Best model for dataset with ccsn, last one
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [138]:
# Best model for dataset with ccsn, least valloss
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [137]:
# Best model for dataset with ccsn
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [152]:
model_number = 826

step2dt = {1:'noise', 2:'BBH', 3:'SGHF', }

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF \t CCSN')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(6):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(6):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4], num[5]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv25.pickle'.format(i), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
    
model = model_list[epochs_list[np.argmin(valloss)]]
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[70000:100000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(6):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))

                    
        

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

In [ ]:
model = models['last_wsc']

In [ ]:
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(0))

In [ ]:
for i in ['bbh', 'lfsg', 'hfsg', 'ccsn']:
    for j in ['5-12', '12-24', '24-48', '48-96']:
        print(i,j)
        print(np.sum(np.logical_and(blind_test_set_ans[:,1] == i, blind_test_set_ans[:,2] == j)))

In [ ]:
print(np.sum(np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,3] == '4')))

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
            j += 1
    i += 1
fig.show()

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
model_number = 1321

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(i), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

#### Enlarged WSC set study, with BBH as anomaly (inner code: v26)

In [153]:
dataset_wsl_fft_all.shape

In [158]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,5))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 5)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    valloss = np.empty(0)
    
    for k in range(len(epochs_list)):
        valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
    
    model = model_list[epochs_list[np.argmin(valloss)]]
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [159]:
FPR_list_AAAAA_FFTTT_enlarged

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [160]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [161]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [168]:
FPR_list_AAAAA_FFTTT_passing_number[1321]

In [170]:
model_number = 1321

step2dt = {1:'noise', 2:'SGHF', 3:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv26.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv26.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(i), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

In [ ]:
model = models['last_wsc']

In [ ]:
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(0))

In [ ]:
for i in ['bbh', 'lfsg', 'hfsg', 'ccsn']:
    for j in ['5-12', '12-24', '24-48', '48-96']:
        print(i,j)
        print(np.sum(np.logical_and(blind_test_set_ans[:,1] == i, blind_test_set_ans[:,2] == j)))

In [ ]:
print(np.sum(np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,3] == '4')))

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
            j += 1
    i += 1
fig.show()

##### Using not the black box set but our leftover testing set for study of distribution

In [171]:
dataset_wsl_separated.keys()

In [196]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

dataset_wsl_separated['bg'] = dataset_wsl_separated[1].copy()
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        print()
        j+=1
    i+=1

In [197]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [195]:
dataset_wsl_separated = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [163]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [189]:
model_number = 1321

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
# with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
#     model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [184]:
dataset_wsl_separated.keys()

In [187]:
# This is the result for the dataset training the model

model_number = 1321

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)

# with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
#     model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [191]:
dataset_wsl_separated.keys()

In [192]:
model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [199]:
dataset_wsl_separated.keys()

In [198]:
model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

In [167]:
FPR_list

#### Enlarged and Extended WSC set study, include CCSN (inner code: v27)

In [618]:
dataset_wsl_fft_withccsn.shape

In [619]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,6))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv27.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 6)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv27.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    valloss = np.empty(0)
    
    for k in range(len(epochs_list)):
        valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])
    
    model = model_list[epochs_list[np.argmin(valloss)]]
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[:70000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[70000:100000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(6):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [620]:
FPR_list_AAAAA_FFTTT_enlarged

In [558]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [621]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [622]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [617]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [615]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [616]:
FPR_list_AAAAA_FFTTT_passing_number[1370]

In [ ]:
FPR_list_AAAAA_FFTTT_passing_number[1321]

In [623]:
model_number = 1360

step2dt = {1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTTv27.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTTv27.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))

for iStep in range(5):
    
    if iStep == 0:
        
        for dt in range(6):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(6):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4], num[5]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv27.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[:70000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[70000:100000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(6):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])

print('')
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))

                    
        

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf


# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

In [ ]:
model = models['last_wsc']

In [ ]:
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(0))

In [ ]:
for i in ['bbh', 'lfsg', 'hfsg', 'ccsn']:
    for j in ['5-12', '12-24', '24-48', '48-96']:
        print(i,j)
        print(np.sum(np.logical_and(blind_test_set_ans[:,1] == i, blind_test_set_ans[:,2] == j)))

In [ ]:
print(np.sum(np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,3] == '4')))

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
            j += 1
    i += 1
fig.show()

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

dataset_wsl_separated['bg'] = dataset_wsl_separated[1].copy()
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        print()
        j+=1
    i+=1

In [742]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

dataset_wsl_separated['bg'] = dataset_wsl_separated[1].copy()
i = 0
for type in ['BBH', 'SGHF', 'SGLF', 'CCSN']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * len(dataset_wsl_separated[i+2])//4:j * len(dataset_wsl_separated[i+2])//4 + len(dataset_wsl_separated[i+2])//4].copy()
        print()
        j+=1
    i+=1

In [743]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [571]:
datatype_list

In [ ]:
dataset_wsl_separated = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [740]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg',
              'CCSN':'ccsn'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF',
              'ccsn':'CCSN'}

In [741]:
dataset_wsl_separated[0].shape

In [745]:
# model_number = 1370

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv27.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']    

# with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
#     model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 4, figsize = (6.4 * 4, 4.8 * 4))

row = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
# This is the result for the dataset training the model

model_number = 1321

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)

# with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
#     model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

In [ ]:
FPR_list

#### Enlarged and Extended WSC set study, include enlarged CCSN (inner code: v29)

In [63]:
dataset_wsl_fft_withccsn_enlarged.shape

In [64]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn_enlarged[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn_enlarged[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn_enlarged[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn_enlarged[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn_enlarged[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn_enlarged[100000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,6))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv29.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 6)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv29.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn_enlarged[:70000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn_enlarged[70000:]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(6):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [65]:
FPR_list_AAAAA_FFTTT_enlarged

In [331]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [66]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [332]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [67]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [334]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [335]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [336]:
FPR_list_AAAAA_FFTTT_passing_number[1024]

In [ ]:
FPR_list_AAAAA_FFTTT_passing_number[1321]

In [ ]:
model_number = 1360

step2dt = {1:'noise', 2:'BBH', 3:'SGHF', 4:'SGLF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTTv27.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTTv27.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))

for iStep in range(5):
    
    if iStep == 0:
        
        for dt in range(6):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.var(dataset_wsl_separated[dt][:, 101:]-dcd, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.var(dataset_wsl_separated[dt][:, :101]-dcd, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(6):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.var(dataset_wsl_separated[dt] - dcd, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4], num[5]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000
num[5] = 4000

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv27.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[:70000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_withccsn[70000:100000]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(6):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])

print('')
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4], num[5]))

                    
        

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf


# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))

In [ ]:
model = models['last_wsc']

In [ ]:
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(0))

In [ ]:
for i in ['bbh', 'lfsg', 'hfsg', 'ccsn']:
    for j in ['5-12', '12-24', '24-48', '48-96']:
        print(i,j)
        print(np.sum(np.logical_and(blind_test_set_ans[:,1] == i, blind_test_set_ans[:,2] == j)))

In [ ]:
print(np.sum(np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,3] == '4')))

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With long dataset, all from 2nd set, timewindow corrected, fixed ratio

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [71]:
FPR_list = np.empty(100)

model_number = 1024

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv29.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg','ccsn']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [70]:
FPR_list = np.empty(100)

model_number = 1024

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv29.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg','ccsn']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
            j += 1
    i += 1
fig.show()

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

dataset_wsl_separated['bg'] = dataset_wsl_separated[1].copy()
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_withccsn[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_withccsn[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_withccsn[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_withccsn[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_withccsn[90000:100000].copy()       # sghf
dataset_wsl_separated[5] = dataset_wsl_fft_withccsn[100000:].copy()

dataset_wsl_separated['bg'] = dataset_wsl_separated[1].copy()
i = 0
for type in ['BBH', 'SGHF', 'SGLF', 'CCSN']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * len(dataset_wsl_separated[i+2])//4:j * len(dataset_wsl_separated[i+2])//4 + len(dataset_wsl_separated[i+2])//4].copy()
        print()
        j+=1
    i+=1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
datatype_list

In [ ]:
dataset_wsl_separated = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg',
              'CCSN':'ccsn'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF',
              'ccsn':'CCSN'}

In [ ]:
dataset_wsl_separated[0].shape

In [ ]:
# model_number = 1370

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTTv27.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']    

# with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
#     model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 4, figsize = (6.4 * 4, 4.8 * 4))

row = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
# This is the result for the dataset training the model

model_number = 1321

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)

# with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
#     model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
    model_list = pickle.load(handle)

valloss = np.empty(0)

for k in range(len(epochs_list)):
    valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

model = model_list[epochs_list[np.argmin(valloss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

In [ ]:
FPR_list

#### Enlarged WSC set study, fixed error function, GWAK set (inner code: v30)

In [76]:
dataset_GWAK.shape

In [85]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(315)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((315,4))

for i in np.arange(0, 315, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv30.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 4)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv30.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(4):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [86]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [87]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [88]:
FPR_list_AAAAA_FFTTT_passing_number[314]

In [75]:
with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv30.pickle'.format(314), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

torch.save(model.state_dict(), '../Upload_competition/model.pt')

In [58]:
model

In [91]:
model_number = 314

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTv30.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTv30.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv30.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

for iStep in range(3):
    
    if iStep == 0:
        
        for dt in range(4):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(4):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(4):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

                    
        

In [ ]:
SIG_class_score.shape

In [ ]:
np.sum(BKG_class_score > threshold) / 70000

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [ ]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
noise_set.shape

In [ ]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# Set for training the model
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

##### See the distribution for the competition dataset

In [76]:
Competiton_dataset = np.load('E://GWNMMAD_data/Tw_dataset/ligo_bb_50.npz')

In [77]:
Competiton_dataset.keys()

In [78]:
Competiton_dataset_idx = Competiton_dataset['ids']
Competiton_dataset = Competiton_dataset['data']

In [79]:
Competiton_dataset_idx.shape

In [80]:
Competiton_dataset.shape

In [83]:
Competiton_dataset = Competiton_dataset / np.linalg.norm(Competiton_dataset, axis = -1).reshape(-1,2,1)
Competiton_dataset = np.abs(np.fft.rfft(Competiton_dataset, axis = -1))

Competiton_dataset = (Competiton_dataset / np.linalg.norm(Competiton_dataset, axis = -1).reshape(-1,2,1)).reshape(-1,202)

In [81]:
with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv30.pickle'.format(314), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# torch.save(model.state_dict(), '../Upload_competition/model.pt')

In [84]:
Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(Competiton_dataset[Competiton_dataset_idx == 1]))).detach().numpy() * np.array([0,0,1,1]), axis = -1)
plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True)
Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(Competiton_dataset[Competiton_dataset_idx == 0]))).detach().numpy() * np.array([0,0,1,1]), axis = -1)
plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True)

#### Enlarged WSC set study, fixed error function, shortened GWAK set (inner code: v32)

In [11]:
dataset_GWAK.shape

In [12]:
dataset_wsl_fft_all.shape

In [13]:
dataset_combined_extra_SGHF = np.concatenate((dataset_GWAK, dataset_wsl_fft_all[-5000:]),axis = 0)

In [26]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(315)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((315,4))

for i in np.arange(0, 315, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 4)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(4):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [39]:
dataset_wsl_fft_all.shape

In [40]:
dataset_wsl_fft_all = ((dataset_wsl_fft_all.reshape(-1,2,101))[:,[1,0],:]).reshape(-1,202)

In [59]:
with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(74), 'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)

torch.save(model, '../Model_cached/model.pt')

In [41]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500 * 2
num[1] = 32500 * 2
num[2] = 5000 * 2
num[3] = 5000 * 2
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(315)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((315,5))

for i in np.arange(0, 315, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 4)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [46]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_combined_extra_SGHF[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_combined_extra_SGHF[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_combined_extra_SGHF[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_combined_extra_SGHF[40000:45000].copy()  # sglf
dataset_wsl_separated[4] = dataset_combined_extra_SGHF[45000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
num[4] = 5000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(315)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((315,5))

for i in np.arange(0, 315, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 4)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [21]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [43]:
# for our new model

np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [30]:
# GWAK set + sghf

np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [22]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [22]:
# for our new model

np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [31]:
# GWAK set + sghf

np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [23]:
FPR_list_AAAAA_FFTTT_passing_number[146]

In [23]:
FPR_list_AAAAA_FFTTT_passing_number[283]

In [32]:
FPR_list_AAAAA_FFTTT_passing_number[74]

In [24]:
model_number = 74

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTv32.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTv32.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

for iStep in range(3):
    
    if iStep == 0:
        
        for dt in range(4):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(4):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(4):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

                    
        

In [103]:
model_number = 74

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTv32.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTv32.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_combined_extra_SGHF[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_combined_extra_SGHF[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_combined_extra_SGHF[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_combined_extra_SGHF[40000:45000].copy()  # sglf
dataset_wsl_separated[4] = dataset_combined_extra_SGHF[45000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
num[4] = 5000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGLF \t  SGHF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(3):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_combined_extra_SGHF[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_combined_extra_SGHF[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_combined_extra_SGHF[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_combined_extra_SGHF[40000:45000].copy()  # sglf
dataset_wsl_separated[4] = dataset_combined_extra_SGHF[45000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
num[4] = 5000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [102]:
model_number = 74

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTv32.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTv32.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv32.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

for iStep in range(3):
    
    if iStep == 0:
        
        for dt in range(4):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(4):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_GWAK[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_GWAK[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_GWAK[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_GWAK[40000:45000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 32500
num[2] = 5000
num[3] = 5000
# num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_GWAK[35000:45000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(4):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

                    
        

In [ ]:
SIG_class_score.shape

In [ ]:
np.sum(BKG_class_score > threshold) / 70000

In [ ]:
model

In [106]:
key2label = ['glitch', 'noise', 'bbh', 'sghf', 'sglf']


dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_combined_extra_SGHF[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_combined_extra_SGHF[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_combined_extra_SGHF[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_combined_extra_SGHF[40000:45000].copy()  # sglf
dataset_wsl_separated[4] = dataset_combined_extra_SGHF[45000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [56]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7, 4))
cnt = 0
ic = np.zeros(4, dtype=int)
for ic[0], ic[1], ic[2], ic[3] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205 // 7):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      list(map(int,FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]])), np.around(cut_combination[sorted_indices[i]],4)))

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [ ]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
noise_set.shape

In [ ]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# Set for training the model
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

#### Enlarged WSC set study, fixed error function, Chia-Jui set (inner code: v31)

In [95]:
dataset_withoutsghf.shape

In [96]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_withoutsghf[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_withoutsghf[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_withoutsghf[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_withoutsghf[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
# num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(315)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((315,4))

for i in np.arange(0, 315, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv31.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 4)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv31.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_withoutsghf[:70000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_withoutsghf[70000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(4):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [97]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [98]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [100]:
FPR_list_AAAAA_FFTTT_passing_number[258]

In [102]:
model_number = 258

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTv31.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTv31.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTv31.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_withoutsghf[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_withoutsghf[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_withoutsghf[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_withoutsghf[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
# num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

for iStep in range(3):
    
    if iStep == 0:
        
        for dt in range(4):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(4):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_withoutsghf[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_withoutsghf[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_withoutsghf[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_withoutsghf[80000:90000].copy()  # sglf
# dataset_wsl_separated[4] = dataset_GWAK[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
# num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_withoutsghf[:70000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_withoutsghf[70000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(4):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3]))

                    
        

In [ ]:
SIG_class_score.shape

In [ ]:
np.sum(BKG_class_score > threshold) / 70000

In [ ]:
model

In [ ]:
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [ ]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
noise_set.shape

In [ ]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# Set for training the model
model_number = 1957

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

#### WSC set study, GWAK-like pipeline (inner code: v33)

In [10]:
dataset_wsl_fft_all.shape

In [55]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sghf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sglf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,5))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 5)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [57]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [58]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_passing_number[1558]

In [59]:
model_number = 1957

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv33.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv33.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [12]:
model_number = 1957

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv33_mini.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv33_mini.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33_mini.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [60]:
SIG_class_score.shape

In [61]:
np.sum(BKG_class_score > threshold) / 70000

In [62]:
model

In [64]:
key2label = {
    0:'glitch',
    1:'noise',
    2:'BBH', 
    3:'SGHF', 
    4:'SGLF'
}

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

In [65]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

###### With long dataset

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With long dataset, fix the events size

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
df

In [ ]:
excel_file = '../Pic_cached/output_s_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With old blackbox set, fix the events size

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:100], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
np.load('../Data_cached/collected_cropped_sig_1_cut.npy').shape

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(5):
blind_test_set = np.load('../Data_cached/collected_cropped_sig_1_cut.npy')[:3]
# blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(3,4,1000,202)

noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))
for type in range(3):
    for snr in range(4):
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, blind_test_set_ffted[type, snr], axis = 0)
        
print(signal_set.shape)
print(noise_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset, all from 2nd set, timewindow corrected

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_{}_v3s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
df

In [ ]:
excel_file = '../Pic_cached/output_s_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With remaining test set

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

noise_set = dataset_wsl_separated_leftover['bg']

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
noise_set.shape

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
perm = np.random.choice(1000000, 65000, replace=False)

noise_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)
noise_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)

noise_set = np.concatenate((noise_L, noise_H), axis = 1)

noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

In [ ]:
noise_set_fft.shape

In [ ]:
noise_set = noise_set_fft.reshape(-1,202)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_1.npy')

In [ ]:
noise_set.shape

In [ ]:
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)

In [ ]:
noise_set.shape

In [ ]:
# Using noise in blackbox

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in blackbox, newly extracted set 1

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in blackbox, newly extracted set 2

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in reserved

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 10k noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 65k noise in test

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 65k noise just extracted

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
# All reserved set, randomly extract 1000

FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # # noise_idx = (blind_test_set_ans[:,0] == '0')
    # noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['BBH', 'SGHF', 'SGLF']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = np.random.choice(10000,1000,replace=False)
            signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr][signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With initial test set

In [66]:
dataset_wsl_separated.keys()

In [ ]:
FPR_list = np.empty(100)

model_number = 1957

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

# noise_set = dataset_wsl_fft_all[:70000]
noise_set = dataset_wsl_separated[1]

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [ ]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
noise_set.shape

In [ ]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# A not careful plot, using noise from v3l and signal from loud_v3l

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [67]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [68]:
dataset_wsl_fft_all.shape

In [69]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [70]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [71]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [79]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [77]:
torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu')).keys()

In [81]:
trans_dict_1

In [82]:
dataset_wsl_separated.keys()

In [83]:
datatype_list

In [84]:
# Set for training the model
model_number = 1957

# torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu'))

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [85]:
FPR_list

In [89]:
# Set for training the model
model_number = 1957

# torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu'))

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
TPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            Score_bg.sort()
            threshold = Score_bg[-1]
            TPR = np.sum(Score > threshold) / len(Score)
            
            threshold_list[datatype] = threshold
            TPR_list[datatype] = TPR
            
            print("For datatype {}, the threshold is {} and the TPR is {}.".format(datatype, threshold, TPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, TPR = {}'.format(datatype, np.around(threshold,3), np.around(TPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

#### WSC set study, GWAK-like pipeline, rediscovery (inner code: v34)

In [16]:
dataset_wsl_fft_all.shape

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sghf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sglf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,5))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 5)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [ ]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [ ]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_passing_number[1558]

In [31]:
model_number = 1957

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv34_mini.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv34_mini.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv34_mini.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [35]:
model_number = 1957

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv35_mini.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv35_mini.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv35_mini.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [ ]:
SIG_class_score.shape

In [73]:
42/67500

In [32]:
np.sum(BKG_class_score > threshold) / 70000

In [ ]:
model

In [71]:
key2label = {
    0:'glitch',
    1:'noise',
    2:'BBH', 
    3:'SGHF', 
    4:'SGLF'
}

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])

plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(GW_event))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1).max(), density = True, histtype='step', bins = 50, label = 'GW190403_051519')

plt.legend()



In [72]:
GW_event.shape

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

###### With long dataset

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With long dataset, fix the events size

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
df

In [ ]:
excel_file = '../Pic_cached/output_s_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With old blackbox set, fix the events size

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:100], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
np.load('../Data_cached/collected_cropped_sig_1_cut.npy').shape

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(5):
blind_test_set = np.load('../Data_cached/collected_cropped_sig_1_cut.npy')[:3]
# blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(3,4,1000,202)

noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))
for type in range(3):
    for snr in range(4):
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, blind_test_set_ffted[type, snr], axis = 0)
        
print(signal_set.shape)
print(noise_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset, all from 2nd set, timewindow corrected

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_{}_v3s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
df

In [ ]:
excel_file = '../Pic_cached/output_s_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With remaining test set

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

noise_set = dataset_wsl_separated_leftover['bg']

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
noise_set.shape

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
perm = np.random.choice(1000000, 65000, replace=False)

noise_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)
noise_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)

noise_set = np.concatenate((noise_L, noise_H), axis = 1)

noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

In [ ]:
noise_set_fft.shape

In [ ]:
noise_set = noise_set_fft.reshape(-1,202)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_1.npy')

In [ ]:
noise_set.shape

In [ ]:
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)

In [ ]:
noise_set.shape

In [ ]:
# Using noise in blackbox

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in blackbox, newly extracted set 1

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in blackbox, newly extracted set 2

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in reserved

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 10k noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 65k noise in test

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 65k noise just extracted

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
# All reserved set, randomly extract 1000

FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # # noise_idx = (blind_test_set_ans[:,0] == '0')
    # noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['BBH', 'SGHF', 'SGLF']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = np.random.choice(10000,1000,replace=False)
            signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr][signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With initial test set

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
FPR_list = np.empty(100)

model_number = 1957

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

# noise_set = dataset_wsl_fft_all[:70000]
noise_set = dataset_wsl_separated[1]

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [ ]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
noise_set.shape

In [ ]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# A not careful plot, using noise from v3l and signal from loud_v3l

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu')).keys()

In [ ]:
trans_dict_1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
datatype_list

In [ ]:
# Set for training the model
model_number = 1957

# torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu'))

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

In [ ]:
# Set for training the model
model_number = 1957

# torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu'))

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
TPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            Score_bg.sort()
            threshold = Score_bg[-1]
            TPR = np.sum(Score > threshold) / len(Score)
            
            threshold_list[datatype] = threshold
            TPR_list[datatype] = TPR
            
            print("For datatype {}, the threshold is {} and the TPR is {}.".format(datatype, threshold, TPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, TPR = {}'.format(datatype, np.around(threshold,3), np.around(TPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

#### WSC set study, GWAK-like pipeline, rediscovery (inner code: v35)

In [21]:
dataset_wsl_fft_all.shape

In [ ]:
# for i in np.arange(0, 1323, dtype=int):

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:5000].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sghf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sglf

num = {}

num[0] = 5000
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

FPR_list_AAAAA_FFTTT_enlarged = np.empty(2205)
FPR_list_AAAAA_FFTTT_passing_number = np.empty((2205,5))

for i in np.arange(0, 2205, dtype=int):
    if not os.path.exists(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(i)):
        FPR_list_AAAAA_FFTTT_enlarged[i] = np.nan
        FPR_list_AAAAA_FFTTT_passing_number[i] = np.array([np.nan] * 5)
        continue
    
    # models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv25.json'.format(i),map_location=torch.device('cpu'))
    
    # print(models.keys())
    
    with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(i), 'rb') as handle:
        model_list = pickle.load(handle)
    
    model = return_model_with_least_valloss(model_list)
    
    BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    
    SIG_class_score.sort()
    
    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]
    
    FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
    
    FPR_list_AAAAA_FFTTT_enlarged[i] = FPR
    
    for dt in range(5):
        # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
        err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
        passidx = err_score > threshold
        # print(cutAE[iStep-1])
        # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
        
        FPR_list_AAAAA_FFTTT_passing_number[i][dt] = np.sum(passidx)
    

In [ ]:
np.nanmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_enlarged.min()

In [ ]:
np.nanargmin(FPR_list_AAAAA_FFTTT_enlarged)

In [ ]:
FPR_list_AAAAA_FFTTT_passing_number[1558]

In [22]:
model_number = 1957

step2dt = {1:'noise', 2:'BBH', 3:'SGHF'}

print('Full analysis for the {}-th model.'.format(model_number))
print('Using Liyang set ratio')

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv35_mini.json'.format(model_number),map_location=torch.device('cpu'))
models_ae = torch.load(ModelDir + '/AE_for_SeriesWSC_{}_ratio_scan_FFTTTv35_mini.json'.format(model_number),map_location=torch.device('cpu'))

foo = torch.load("./Sida_temp/for_K8S_training/Model/glitch_AE_freq_new.json")
models_ae['glitch_H'] = foo["H_101-10-20-10"].cpu().eval()
models_ae['glitch_L'] = foo["L_101-10-10"].cpu().eval()

cutAE = models['cut_vals']

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv34_mini.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

print('The cut scheme for this model is {}'.format(models['cut_vals']))

print('Passing state of the dataset \t GLT \t Noise \t BBH \t SGHF \t SGLF')

print('Initial state \t  \t  \t {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

for iStep in range(4):
    
    if iStep == 0:
        
        for dt in range(5):
            
            dcd = models_ae['glitch_H'](torch.FloatTensor(dataset_wsl_separated[dt][:, 101:]))[1].detach().numpy()
            err_score_H = np.mean((dataset_wsl_separated[dt][:, 101:]-dcd)**2, axis=1)
            passH = err_score_H > cutAE[0]

            dcd = models_ae['glitch_L'](torch.FloatTensor(dataset_wsl_separated[dt][:, :101]))[1].detach().numpy()
            err_score_L = np.mean((dataset_wsl_separated[dt][:, :101]-dcd)**2, axis=1)
            passL = err_score_L > cutAE[1]

            passidx = np.logical_and(passH, passL)
            dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
            
            num[dt] = len(dataset_wsl_separated[dt])
        
        print('After the Glitch AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))
        
    else:
    
        for minor_step in range(2):
            
            if minor_step == 0:
                
                for dt in range(5):
                    dcd = models_ae[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
                    err_score = np.mean((dataset_wsl_separated[dt] - dcd)**2, axis=1)
                    passidx = err_score > cutAE[iStep+1]
                    # print(cutAE[iStep-1])
                    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
                    num[dt] = np.sum(passidx)
                    
                print('After the {} AE \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
                
            # else:
                
            #     for dt in range(5):
            #         passidx = nn.Sigmoid()(models[step2dt[iStep]](torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy().flatten()>=0.5
            #         dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
                    
            #         num[dt] = len(dataset_wsl_separated[dt])
                    
            #     print('After the {} WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(step2dt[iStep], num[0], num[1], num[2], num[3], num[4]))
    
dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

num = {}

num[0] = 2500
num[1] = 65000
num[2] = 10000
num[3] = 10000
num[4] = 10000

# model = models['last_wsc']
    
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[:70000]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_fft_all[70000:]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]


for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    num[dt] = len(dataset_wsl_separated[dt])
    
print('After the Final WSC \t  \t  {} \t {} \t {} \t {} \t {}'.format(num[0], num[1], num[2], num[3], num[4]))

                    
        

In [24]:
49/6750

In [25]:
42/6750

In [ ]:
SIG_class_score.shape

In [ ]:
42/67500

In [ ]:
np.sum(BKG_class_score > threshold) / 70000

In [ ]:
model

In [26]:
key2label = {
    0:'glitch',
    1:'noise',
    2:'BBH', 
    3:'SGHF', 
    4:'SGLF'
}

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])

# plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(GW_event))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1).max(), density = True, histtype='step', bins = 50, label = 'GW190403_051519')

plt.legend()



In [29]:
key2label = {
    0:'glitch',
    1:'noise',
    2:'BBH', 
    3:'SGHF', 
    4:'SGLF'
}

dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_wsl_fft_all[5000:70000].copy()   # noise
dataset_wsl_separated[2] = dataset_wsl_fft_all[70000:80000].copy()  # bbh
dataset_wsl_separated[3] = dataset_wsl_fft_all[80000:90000].copy()  # sglf
dataset_wsl_separated[4] = dataset_wsl_fft_all[90000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])

plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(GW_event))).detach().numpy() * np.array([0,0,1,1,1])), axis = 1).max() + 0.2, density = True, histtype='step', bins = 50, label = 'GW190403_051519')

plt.legend()



In [ ]:
GW_event.shape

In [ ]:
Ncut = 3;
cut_glitchH = np.linspace(0.0016, 0.003, 5)[1:-1]
cut_glitchL = np.linspace(0.0012, 0.0025, 5)[1:-1]
# cut_noise = np.linspace(0.0006, 0.0018, Ncut+2)[1:-1]
cut_noise = np.array([0.1, 0.08, 0.05, 0.02, 0.01])
cut_BBH = np.linspace(0.1, 0.9, 7)
cut_SGHF = np.linspace(0.1, 0.9, 7)

cut_combination = np.empty((3*3*5*7*7, 5))
cnt = 0
ic = np.zeros(5, dtype=int)
for ic[0], ic[1], ic[2], ic[3], ic[4] in itertools.product(np.arange(3), np.arange(3), np.arange(5), np.arange(7), np.arange(7)):
    cut_combination[cnt] = [cut_glitchH[ic[0]], cut_glitchL[ic[1]], cut_noise[ic[2]], cut_BBH[ic[3]], cut_SGHF[ic[4]]]
    cnt += 1


sorted_indices = np.argsort(FPR_list_AAAAA_FFTTT_enlarged)
# sorted_indices = np.argsort(FPR_list_AAWAW_FFTFT)
# sorted_indices = np.arange(1323)

# print("index \t A+W FPR \t A FPR \t \t \t \t \t cut scheme")
print("idx \t A FPR \t \t \t \t \t cut scheme")

for i in range(0, 2205):
    print('{:<4d} \t {:.5f} \t {} \t \t {}'.format(sorted_indices[i], FPR_list_AAAAA_FFTTT_enlarged[sorted_indices[i]],
                                      FPR_list_AAAAA_FFTTT_passing_number[sorted_indices[i]], np.around(cut_combination[sorted_indices[i]],4)))

##### We use the v13 best model to do the blind test

###### With long dataset

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With long dataset, fix the events size

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
df

In [ ]:
excel_file = '../Pic_cached/output_s_sglf.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With old blackbox set, fix the events size

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:100], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
np.load('../Data_cached/collected_cropped_sig_1_cut.npy').shape

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(5):
blind_test_set = np.load('../Data_cached/collected_cropped_sig_1_cut.npy')[:3]
# blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(3,4,1000,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(3,4,1000,202)

noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))
for type in range(3):
    for snr in range(4):
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, blind_test_set_ffted[type, snr], axis = 0)
        
print(signal_set.shape)
print(noise_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers

In [ ]:
df

In [ ]:
for snr in ['5-12', '12-24', '24-48','48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
df

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
excel_file = '../Pic_cached/output_l_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With long dataset, all from 2nd set, timewindow corrected, fix the events size

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list)

In [ ]:
np.mean(FPR_list)

In [ ]:
np.var(FPR_list)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

noise_set = dataset_wsl_separated[1]


for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_2.npy')
noise_set.shape
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)
noise_set.shape

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:1000], axis = 0)
            
    print(signal_set.shape)
    
    perm = np.random.choice(60000,10000,replace = False)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set[perm]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
ccsn_rawset = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:3,:,:,:,6,:]

In [ ]:
ccsn_rawset.shape

In [ ]:
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = np.abs(np.fft.rfft(ccsn_rawset, axis = -1))
ccsn_rawset = ccsn_rawset / np.linalg.norm(ccsn_rawset, axis = -1).reshape(3,4,3000,2,1)

In [ ]:
ccsn_rawset = ccsn_rawset.reshape(3,4,3000,202)

In [ ]:
ccsn_rawset.shape

In [ ]:
noise_set.shape

In [ ]:
FPR_list = np.empty(10)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(10):
    # blind_test_set = np.load('../Data_cached/blind_dataset_loud_{}_v3l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blind_dataset_ans_loud_{}_v3l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # noise_idx = (blind_test_set_ans[:,0] == '0')
    noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in range(3):
        for snr in range(4):
            perm = np.random.choice(3000,1000,replace=False)
            # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, ccsn_rawset[type, snr][perm], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With short dataset, all from 2nd set, timewindow corrected

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_{}_v3s.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (np.logical_and(blind_test_set_ans[:,0] == '1', blind_test_set_ans[:,1] != 'ccsn'))
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
data = {}

sorted_indices = np.argsort(FPR_list)

for i in range(100):
    num = np.empty(14)
    j = 0
    idx = sorted_indices[i]
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3s/blind_dataset_ans_{}_v3s.npy'.format(idx))
    for type in ['glitch', 'bg']:
        num[j] = np.sum(blind_test_set_ans[:,1] == type)
        j+=1
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12', '12-24', '24-48', '48-96']:
            num[j] = np.sum(np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            j+=1
        
    data[idx] = np.append(num, FPR_list[idx])
            
    print('{}, {}, {}'.format(idx, num, FPR_list[idx]))

In [ ]:
data[58]

In [ ]:
df = pd.DataFrame(data).T

# 保存为 Excel 文件


In [ ]:
columns = []
columns.append('glitch')
columns.append('bg')

for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        columns.append(type + snr)

columns.append('FPR')

In [ ]:
columns

In [ ]:
df.columns = columns

In [ ]:
df

In [ ]:
for snr in ['5-12','12-24','24-48','48-96']:
    df[snr] = df[[type + snr for type in ['bbh', 'hfsg', 'lfsg']]].sum(axis=1)

In [ ]:
for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[[type + snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

In [ ]:
df

In [ ]:
total_numbers = df[[snr for snr in ['5-12','12-24','24-48','48-96']]].sum(axis=1)

for type in ['bbh', 'hfsg', 'lfsg']:
    df[type] = df[type] / total_numbers
for snr in ['5-12', '12-24', '24-48', '48-96']:
    df[snr] = df[snr] / total_numbers

In [ ]:
df

In [ ]:
df['total_numbers'] = total_numbers

In [ ]:
b_column = df.pop('FPR')
df['FPR'] = b_column

In [ ]:
df

In [ ]:
excel_file = '../Pic_cached/output_s_sglf_window_corrected.xlsx'
df.to_excel(excel_file, index=True)

In [ ]:
Noise_score.shape

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)

In [ ]:
print(blind_test_set_ans)

###### With remaining test set

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

noise_set = dataset_wsl_separated_leftover['bg']

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
noise_set.shape

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
noise_set = dataset_wsl_separated_leftover['bg']

In [ ]:
perm = np.random.choice(1000000, 65000, replace=False)

noise_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)
noise_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:1000000][perm].reshape(-1,1,200)

noise_set = np.concatenate((noise_L, noise_H), axis = 1)

noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

In [ ]:
noise_set_fft.shape

In [ ]:
noise_set = noise_set_fft.reshape(-1,202)

In [ ]:
noise_set = np.load('../Data_cached/collected_cropped_bg_larger_1.npy')

In [ ]:
noise_set.shape

In [ ]:
noise_set = noise_set / np.linalg.norm(noise_set, axis = -1).reshape(-1,2,1)

noise_set_fft = np.abs(np.fft.rfft(noise_set, axis = -1))
noise_set_fft = noise_set_fft / np.linalg.norm(noise_set_fft, axis = -1).reshape(-1,2,1)

noise_set = noise_set_fft.reshape(-1,202)

In [ ]:
noise_set.shape

In [ ]:
# Using noise in blackbox

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in blackbox, newly extracted set 1

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in blackbox, newly extracted set 2

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, ))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise in reserved

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 10k noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using noise just extracted

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 65k noise in test

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
# Using 65k noise just extracted

noise_set = dataset_wsl_separated[1]
print(noise_set.shape)

event_numbers = [1000,3000,5000]

FPR_list = np.empty((3,3))


model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

i = 0
for event_number in event_numbers:

    for times in range(3):
        
        signal_set = np.empty((0,202))
        
        for type in ['BBH', 'SGHF', 'SGLF']:
            for snr in ['5-12','12-24','24-48', '48-96']:
                
                perm = np.random.choice(54999, 54999, replace = False)
                data_cached = np.load('../Data_cached/injected_{}_55k_snr{}_0th_events_before_merger_time_windowlength_200.npz'.format(type, snr))['strain'][perm][:event_number].reshape(-1,2,200)
                
                data_cached = data_cached / np.linalg.norm(data_cached, axis = -1).reshape(-1,2,1)
                
                data_cached_fft =np.abs(np.fft.rfft(data_cached, axis = -1))
                data_cached_fft = data_cached_fft / np.linalg.norm(data_cached_fft, axis = -1).reshape(-1,2,1)
                
                signal_set = np.append(signal_set, data_cached_fft.reshape(-1,202), axis = 0)
        
        print(signal_set.shape)
        
        Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

        Signal_score.sort()

        threshold = Signal_score[int(0.1 * len(Signal_score))]

        FPR = np.sum(Noise_score > threshold) / len(Noise_score)

        FPR_list[i,times] = FPR
        
        print(FPR)
    
    i+=1
        

In [ ]:
dataset_wsl_separated_leftover.keys()

In [ ]:
# All reserved set, randomly extract 1000

FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    # # noise_idx = (blind_test_set_ans[:,0] == '0')
    # noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
    signal_set = np.empty((0,202))
    for type in ['BBH', 'SGHF', 'SGLF']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = np.random.choice(10000,1000,replace=False)
            signal_set = np.append(signal_set, dataset_wsl_separated_leftover[type + '_' + snr][signal_idx], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    # noise_set = blind_test_set_ffted[noise_idx]
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

In [ ]:
plt.hist(FPR_list, bins = 20)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

###### With initial test set

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
FPR_list = np.empty(100)

model_number = 1957

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

# for idx in range(100):
    # blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    # blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
# noise_idx = (blind_test_set_ans[:,0] == '0')
signal_set = np.empty((0,202))

# noise_set = dataset_wsl_fft_all[:70000]
noise_set = dataset_wsl_separated[1]

for type in ['BBH', 'SGHF', 'SGLF']:
    for snr in ['5-12','12-24','24-48','48-96']:
        # signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set = np.append(signal_set, dataset_wsl_separated[type + '_' + snr], axis = 0)
        
print(signal_set.shape)

Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)

Signal_score.sort()

threshold = Signal_score[int(0.1 * len(Signal_score))]

FPR = np.sum(Noise_score > threshold) / len(Noise_score)

FPR_list[idx] = FPR

print(FPR)

In [ ]:
FPR_list = np.empty(100)

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_set = np.empty((0,202))
    for type in ['bbh', 'hfsg', 'lfsg']:
        for snr in ['5-12','12-24','24-48','48-96']:
            signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
            signal_set = np.append(signal_set, blind_test_set_ffted[signal_idx][:500], axis = 0)
            
    print(signal_set.shape)
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    FPR_list[idx] = FPR
    
    print(FPR)

##### print the score distribution of different types

In [ ]:
for idx in range(5):
    blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
    blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise({})'.format(len(Score)))

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal({})'.format(len(Score)))
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
len(blind_test_set)

In [ ]:

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
plt.legend()

In [ ]:
for type in ['glitch', 'bg', 'bbh', 'hfsg', 'lfsg','ccsn']:
    idx = (blind_test_set_ans[:,1] == type)
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format(type, np.sum(idx)))
    
    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type)
    
plt.legend()

In [ ]:
Noise_score

In [ ]:
Signal_score

In [ ]:
np.sum(noise_idx)

In [ ]:
np.sum(signal_idx)

In [ ]:
i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        # dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx][:1000]

In [ ]:
noise_set.shape

In [ ]:
for key in signal_set.keys():
    print(key)
    print(signal_set[key].shape)

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
# former black box test set

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
# A not careful plot, using noise from v3l and signal from loud_v3l

# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
idx = 0

signal_set = {}

blind_test_set = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_{}_v3l.npy'.format(idx))
blind_test_set_ans = np.load('../Data_cached/blackboxtest/v3l/blind_dataset_ans_{}_v3l.npy'.format(idx))

blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)

blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)

noise_idx = (np.logical_and(blind_test_set_ans[:,0] == '0', blind_test_set_ans[:,1] == 'bg'))
noise_set = blind_test_set_ffted[noise_idx]
# signal_set = np.empty((0,202))
for type in ['bbh', 'hfsg', 'lfsg']:
    for snr in ['5-12','12-24','24-48','48-96']:
        signal_idx = (np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr))
        signal_set[type + '_' + snr] = blind_test_set_ffted[signal_idx]

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(noise_set))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(signal_set[datatype_1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

##### A more careful plot of the score distribution, to make sure that score for the same type and snr are alike

In [ ]:
snr_range = ['5-12','12-24','24-48','48-96']
datatype_list = ['glitch','bg','bbh','hfsg','lfsg','ccsn']
string2intdict = {'5-12':[5,12],
                  '12-24':[12,24],
                  '24-48':[24,48],
                  '48-96':[48,96]}

In [ ]:
dataset_wsl_fft_all.shape

In [ ]:
num = {}

num['glitch'] = 5000
num['bg'] = 65000
num['bbh'] = 10000
num['hfsg'] = 10000
num['lfsg'] = 10000

In [ ]:
datatype

In [ ]:
dataset_wsl_separated = {}
dataset_wsl_separated['glitch'] = dataset_wsl_fft_all[:5000]
dataset_wsl_separated['bg'] = dataset_wsl_fft_all[5000:70000]

i = 0
j = 0
for type in datatype_list[2:5]:
    j = 0
    for snr in snr_range:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_fft_all[70000 + i * 10000 + j * 2500: 70000 + i * 10000 + j * 2500 + 2500]
        print(70000 + i * 10000 + j * 2500)
        j += 1
    i += 1

In [ ]:
for key in dataset_wsl_separated.keys():
    print(key)
    print(dataset_wsl_separated[key].shape)

In [ ]:
for idx in range(100):
    blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
    blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
    
    blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
    blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
    
    blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
    
    noise_idx = (blind_test_set_ans[:,0] == '0')
    signal_idx = (blind_test_set_ans[:,0] == '1')
    
    Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    
    Signal_score.sort()
    
    threshold = Signal_score[int(0.1 * len(Signal_score))]
    
    FPR = np.sum(Noise_score > threshold) / len(Noise_score)
    
    print(FPR)
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

    plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Signal')
    
    plt.title('Score distribution for blind dataset {}.'.format(idx))
    plt.axvline(threshold)
        
    plt.legend()
    plt.show()

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_loud_{}_v2l.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2l/blind_dataset_ans_loud_{}_v2l.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
            blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(5):
                blind_test_set = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_loud_{}_v2s.npy'.format(idx))
                blind_test_set_ans = np.load('../Data_cached/blackboxtest/v2s/blind_dataset_ans_loud_{}_v2s.npy'.format(idx))
                
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                if np.sum(selected_idx) < 100:
                    print('Not enough events, skip plotting. ')
                    continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
os.getcwd()

In [ ]:
dataset_cropped_all = np.load('../Data_cached/blackboxtest/collected_cropped_sig_res.npy')[:,:,:,:,6]

In [ ]:
dataset_cropped_all.shape

In [ ]:
datatype_list

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig, axs = plt.subplots(4, 6, figsize = (6.4*6,4.8*4))

i = 0

for type in datatype_list:
    if type in ['glitch', 'bg']:
        continue
        j = 0
        datatype = type
        axs[j,i].set_title('{}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(5):
            blind_test_set = dataset_cropped_all[i-2,]
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = blind_test_set_ans[:,1] == type
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}({})'.format(idx, len(Score)), color = colors[idx+1])
            axs[j,i].set_ylim(0,3.3)
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        # plt.legend()
        # plt.show()
        
    else:
        j = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            axs[j,i].set_title('{}'.format(datatype))
            
            if type != 'ccsn':
                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
                print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            for idx in range(1):
                # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
                # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
                blind_test_set = dataset_cropped_all[i,j]
                blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
                blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
                
                blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
                
                # noise_idx = (blind_test_set_ans[:,0] == '0')
                # signal_idx = (blind_test_set_ans[:,0] == '1')
                
                # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                
                # Signal_score.sort()
                
                # threshold = Signal_score[int(0.1 * len(Signal_score))]
                
                # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
                
                # print(FPR)
                
                # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
                # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
                # if np.sum(selected_idx) < 100:
                #     print('Not enough events, skip plotting. ')
                #     continue
                
                # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

                # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

                Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
                # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

                axs[j,i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_{}({})'.format(idx,len(Score)), color = colors[idx+1])
                
                # plt.title('Score distribution for blind dataset {}.'.format(idx))
                # plt.axvline(threshold)
                axs[j,i].legend()
                axs[j,i].set_ylim(0,3.3)
            j += 1
    i += 1
fig.show()

In [ ]:
i

##### Using not the black box set but our leftover testing set for study of distribution

In [ ]:
dataset_wsl_separated_leftover = torch.load('../Data_cached/10k_dataset_for_analysis.json')

In [ ]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [ ]:
for key in dataset_wsl_separated_leftover.keys():
    print(key)
    print(dataset_wsl_separated_leftover[key].shape)

In [ ]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

model_number = 1558

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv13_new.pickle'.format(model_number), 'rb') as handle:
    model_list = pickle.load(handle)
    
model = return_model_with_least_valloss(model_list)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            datatype_1 = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[1]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[1])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_benchmark', color = colors[1])
            
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_blackbox', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(noise_set)))
            
            
            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated_leftover[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'signal_blackbox', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu')).keys()

In [ ]:
trans_dict_1

In [ ]:
dataset_wsl_separated.keys()

In [ ]:
datatype_list

In [ ]:
# Set for training the model
model_number = 1957

# torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu'))

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold = Score[int(0.1 * len(Score))]
            FPR = np.sum(Score_bg > threshold) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, FPR = {}'.format(datatype, np.around(threshold,3), np.around(FPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

In [ ]:
# Set for training the model
model_number = 1957

# torch.load(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),map_location=torch.device('cpu'))

with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv33.pickle'.format(model_number),'rb') as handle:
    model_list = pickle.load(handle)

model = return_model_with_least_valloss(model_list)
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
TPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19,20))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = type + '_' + snr
            
            Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            Score_bg.sort()
            threshold = Score_bg[-1]
            TPR = np.sum(Score > threshold) / len(Score)
            
            threshold_list[datatype] = threshold
            TPR_list[datatype] = TPR
            
            print("For datatype {}, the threshold is {} and the TPR is {}.".format(datatype, threshold, TPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype, color = colors[1])
            axs[col, row].axvline(threshold)
            # axs[col, row].set_ylim(0,3.3)
            
            axs[col, row].set_title('{}, threshold = {}, TPR = {}'.format(datatype, np.around(threshold,3), np.around(TPR,3)))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [ ]:
FPR_list

##### Study whether the detection comes from Correlation or the waveform itself

In [ ]:
model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [ ]:
data_high_snr = {}

data_high_snr['BBH'] = dataset_wsl_fft_all[77500:80000]
data_high_snr['SGHF'] = dataset_wsl_fft_all[87500:90000]
data_high_snr['SGLF'] = dataset_wsl_fft_all[97500:]
# data_high_snr['CCSN']

data_noise_standby = dataset_wsl_fft_all[:7500]

data_noise_leftover = dataset_wsl_fft_all[7500:65000]

In [ ]:
data_high_snr_withoutcorr = {}

for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    # print(cache[0])
    np.random.shuffle(cache[:,0,:])
    # print(cache[0])
    data_high_snr_withoutcorr[key] = cache.copy().reshape(-1,202)

In [ ]:
np.random.choice(2,100)

In [ ]:
data_high_snr_halfsig = {}

i = 0
for key in data_high_snr.keys():
    cache = data_high_snr[key].copy().reshape(-1,2,101)
    events_pick = np.random.choice(2500,2500, replace = False)
    detector_pick = np.random.choice(2,2500)
    cache[events_pick,detector_pick,:] = data_noise_standby[i * 2500: i * 2500+2500].reshape(-1,2,101)[range(2500),detector_pick,:]
    data_high_snr_halfsig[key] = cache.copy().reshape(-1,202)
    i+=1

In [ ]:
events_pick = np.random.choice(3,3, replace = False)
detector_pick = np.random.choice(2,3)

print(events_pick)
print(detector_pick)

In [ ]:
test_sig = (np.ones((3,6)) * np.array([[1],[2],[3]])).reshape((3,2,3))
test_noise = (np.ones((3,6)) * np.array([[4],[5],[6]])).reshape((3,2,3))

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
print(detector_pick)

In [ ]:
events_pick

In [ ]:
test_sig[events_pick,detector_pick,:] = test_noise[range(3),detector_pick,:]

In [ ]:
test_sig[events_pick,detector_pick,:]

In [ ]:
test_sig

In [ ]:
test_noise

In [ ]:
test_sig

In [ ]:
for key in data_high_snr.keys():
    print(data_high_snr[key].shape)
    print(np.linalg.norm(data_high_snr[key], axis = -1))
    print(data_high_snr_withoutcorr[key].shape)
    print(np.linalg.norm(data_high_snr_withoutcorr[key], axis = -1))
    print(data_high_snr_halfsig[key].shape)
    print(np.linalg.norm(data_high_snr_halfsig[key], axis = -1))

In [ ]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

model_number = 1614

models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in data_high_snr.keys():
    Score_bg = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_noise_leftover))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'High snr', color = colors[1])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_withoutcorr[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Without corr', color = colors[2])
    
    Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(data_high_snr_halfsig[key]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
    axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Half signal', color = colors[3])
    
    axes[i].set_title(key)
    
    i+=1
plt.legend()

### Analyze the CNN+LSTM performance

In [4]:
class WaveletCNNAE_xc(nn.Module):
    
    def __init__(
        self, 
        num_ifos: int,
        c_depth: int=8, 
        n_chann: int=64, 
        l1: int=1024
        # lx: int=200
    ):
        
        super(WaveletCNNAE_xc, self).__init__()
        
        self.c_depth = c_depth
        self.n_chann = n_chann
        
        self.cap_norm = nn.GroupNorm(num_ifos, num_ifos)
        
        self.Conv_In_encode = nn.Conv1d(
                in_channels=num_ifos, 
                out_channels=self.n_chann, 
                kernel_size=1
            )
        
        self.Conv_Out_encode = nn.Conv1d(
                in_channels=self.n_chann, 
                out_channels=1, 
                kernel_size=1
            )
        
        self.Conv_In_decode = nn.Conv1d(
                in_channels=1, 
                out_channels=self.n_chann, 
                kernel_size=1
            )
        
        self.Conv_Out_decode = nn.Conv1d(
                in_channels=self.n_chann, 
                out_channels=num_ifos, 
                kernel_size=1
            )
        
        self.body_norm_encode = nn.GroupNorm(4 ,n_chann)
        self.body_norm_decode = nn.GroupNorm(4 ,n_chann)
        self.end_norm_encode = nn.BatchNorm1d(1)
        self.end_norm_decode = nn.BatchNorm1d(1)
        
        self.WaveNet_layers_encode = nn.ModuleList()
        self.WaveNet_layers_decode = nn.ModuleList()
        self.WaveNet_layers_dp = nn.ModuleList()
        
        
        for i in range(self.c_depth):

            conv_layer = nn.Conv1d(
                in_channels=self.n_chann, 
                out_channels=self.n_chann,
                kernel_size=2,
                dilation=2**i
            )
            
            self.WaveNet_layers_encode.append(conv_layer)
            
        for i in range(self.c_depth-1, -1, -1):

            conv_layer = nn.Conv1d(
                in_channels=self.n_chann, 
                out_channels=self.n_chann,
                kernel_size=2,
                dilation=2**i
            )
            
            self.WaveNet_layers_decode.append(conv_layer)
            self.WaveNet_layers_dp.append(nn.ZeroPad1d(2**i))
        
        
#         self.Padding_layer = nn.ZeroPad1d(2**c_depth - 1)
                
        # self.L1 = nn.Linear(8192-2**c_depth, l1)
        
        # Consider replacing other batch normalizatoin layers with other nor method
        # Because batch norm are baised by the population of the CCSN rate in one batch 
        # This may produce overfitting model and will not be able to found at test phase
        # Question: Will we be able to figure out the side effect at infereceing phase?
                
#         self.conv_norm = nn.BatchNorm1d(200-2**c_depth + 1)
        self.L1 = nn.Linear(200-2**c_depth + 1, l1)
        self.L1_norm = nn.BatchNorm1d(l1)
        self.L2 = nn.Linear(l1, 200-2**c_depth + 1)
        self.L2_norm = nn.BatchNorm1d(200-2**c_depth + 1)

        nn.init.kaiming_normal_(self.Conv_In_encode.weight)
        nn.init.kaiming_normal_(self.Conv_Out_encode.weight)
        nn.init.constant_(self.Conv_In_encode.bias, 0.001)
        nn.init.constant_(self.Conv_Out_encode.bias, 0.001)
        
        nn.init.kaiming_normal_(self.Conv_In_decode.weight)
        nn.init.kaiming_normal_(self.Conv_Out_decode.weight)
        nn.init.constant_(self.Conv_In_decode.bias, 0.001)
        nn.init.constant_(self.Conv_Out_decode.bias, 0.001)

        # Initialize all the convolutional layer in between
        for conv_layer in self.WaveNet_layers_encode:
            nn.init.kaiming_normal_(conv_layer.weight)
            nn.init.constant_(conv_layer.bias, 0.001)
            
        for conv_layer in self.WaveNet_layers_decode:
            nn.init.kaiming_normal_(conv_layer.weight)
            nn.init.constant_(conv_layer.bias, 0.001)    

        nn.init.kaiming_uniform_(self.L1.weight)
        nn.init.kaiming_uniform_(self.L2.weight)
        nn.init.constant_(self.L1.bias, 0.001)
        nn.init.constant_(self.L2.bias, 0.001)
        
    def encode(self, x):
        
        x = self.cap_norm(x)
        x = self.Conv_In_encode(x)
        x = F.relu(x)
        
        # x = self.norm(x)
        
        for what_are_u_wavin_at in self.WaveNet_layers_encode:
            x = self.body_norm_encode(x)
            x = what_are_u_wavin_at(x)
            x = F.relu(x)
            
        x = self.Conv_Out_encode(x)
        x = F.relu(x)
        x = self.end_norm_encode(x)
        
        # print(x.shape)
        x = torch.flatten(x, 1)
        x = self.L1_norm(F.relu(self.L1(x)))
        
        # print('Encoder done')
        
        return x
    
    def decode(self, x):
        x = self.L2_norm(F.relu(self.L2(x)))
        
#         x = self.Padding_layer(x)
        
        x = torch.unsqueeze(x,1)

        # print(x.shape)
        
        # x = self.cap_norm(x)
        x = self.Conv_In_decode(x)
        x = F.relu(x)
        
        # x = self.norm(x)
        
        for (pad, dcd) in zip(self.WaveNet_layers_dp, self.WaveNet_layers_decode):
            # print(x.shape)
            x = self.body_norm_decode(x)
            x = pad(x)
            x = torch.flip(dcd(torch.flip(x, [-1])), [-1])
            x = F.relu(x)
        
        # print(x.shape)
        # print('CNN done')
        
        x = self.Conv_Out_decode(x)
        # print(x.shape)
        x = F.tanh(x)
        # print(x.shape)
        # x = self.end_norm_decode(x)
        
        # x = torch.flatten(x, 1)
        
        
        return x
    
    def forward(self,x):
        return self.decode(self.encode(x))
    


# A fast examine, not asking for the bunch normalization

# No need for the activition function here, since the LSTM layer already introduces non-linearity

class TimeDistributed(nn.Module):
    def __init__(self, layer):
        super(TimeDistributed, self).__init__()
        self.layer = layer

    def forward(self, x):
        batch_size, time_steps, features = x.size()
        x = x.reshape(-1, features)
        outputs = self.layer(x)
        outputs = outputs.view(batch_size, time_steps, -1)
        return outputs

class LSTMAutoencoder(nn.Module):
    def __init__(self, encoder_struct, decoder_struct):
        super(LSTMAutoencoder, self).__init__()
        
        self.encoder_dep = len(encoder_struct)
        self.encoder_struct = encoder_struct
        
        self.decoder_dep = len(decoder_struct)
        self.decoder_struct = decoder_struct
        
        self.encoder_layer = nn.ModuleList()
        
        # self.encoder_layer = nn.LSTM(2, , batch_first=True)  # 第一层编码器
        # self.encoder_layer2 = nn.LSTM(32, 8, batch_first=True)  # 第二层编码器
        
        for i in range(self.encoder_dep-1):
            layer = nn.LSTM(self.encoder_struct[i], self.encoder_struct[i+1])
            self.encoder_layer.append(layer)


        self.decoder_layer = nn.ModuleList()
        
        # self.decoder_layer = nn.LSTM(8, 32, batch_first=True)  # 第一层解码器
        
        layer = nn.LSTM(self.encoder_struct[-1], self.decoder_struct[0])
        self.decoder_layer.append(layer)
        
        for i in range(self.decoder_dep-1):
            layer = nn.LSTM(self.decoder_struct[i], self.decoder_struct[i+1])
            self.decoder_layer.append(layer)
        
        
        self.time_distributed = TimeDistributed(nn.Linear(self.decoder_struct[-1],self.encoder_struct[0]))
        
        self.act = nn.Tanh()


    def forward(self, x):
        # 编码器
        # encoded_seq, _ = self.encoder_layer1(x)  # 第一层编码器输出
        # encoded_seq, _ = self.encoder_layer2(encoded_seq)  # 第二层编码器输出
        
        # print(len(self.encoder_layer))
        
        x = x.permute(0,2,1)
        
        for layer in self.encoder_layer:
            # print(x.shape)
            # print(i)
            x, _ = layer(x)
            
        # print(x)
        if len(x.size()) == 3:

            x = x[:, -1, :].unsqueeze(1)
            x = x.repeat(1, 200, 1)

        elif len(x.size()) == 2:

            x = x[-1].unsqueeze(0)
            x = x.repeat(200, 1)
        
        # 解码器
        # decoded_seq, _ = self.decoder_layer1(encoded_seq)  # 第一层解码器输出
        
        for layer in self.decoder_layer:
            x, _ = layer(x)
        
        decoded_seq = self.time_distributed(x)

        decoded_seq = self.act(decoded_seq)
        
        return decoded_seq.permute(0,2,1)
    
    
class CNNplusLSTM(nn.Module):
    def __init__(self, 
        encoder_struct: list, 
        decoder_struct: list,
        num_ifos: int,
        c_depth: int=8, 
        n_chann: int=64, 
        l1: int=1024):
        super(CNNplusLSTM, self).__init__()
        self.CNN = WaveletCNNAE_xc(num_ifos, c_depth, n_chann, l1)
        self.LSTM = LSTMAutoencoder(encoder_struct, decoder_struct)
        
    def forward(self, x):
        unnoised = self.CNN(x)
        
        final = self.LSTM(unnoised)
        
        return unnoised, final

In [9]:
os.getcwd()

In [32]:
ModelDir_CNN = '../K8S_training/CNN_high_snr_train/Output'

cd = 3
nch = 4
nl1 = 10

version = 'v6'

model = torch.load(ModelDir_CNN + '/model_' + "dep_"+str(cd)+"_chnl_"+str(nch)+"_btn_"+str(nl1) + "_" + version + '.pt')

In [33]:
model

In [34]:
os.getcwd()

In [40]:
realbbh = np.load("../data_cached/injected_BBH_55k_snr48-96_0th_events_before_merger_time_windowlength_200.npz")['strain'].reshape(-1,200)

In [41]:
realbbh = realbbh / np.linalg.norm([realbbh], axis=2).T
realbbh = realbbh.reshape(-1,2,200);


perm = np.random.permutation(len(realbbh))
realbbh = realbbh[perm]

In [50]:
realbbh.shape
realbbh[idx].shape

In [56]:
realbbh[[idx]]

In [55]:
model(torch.FloatTensor(realbbh[[idx]])).detach().numpy()

In [46]:
idx = 0

plt.plot(model(realbbh[idx])[0])

In [60]:
ModelDir_CNN = '../K8S_training/CNN_high_snr_train/Output'
plot_idx = 0


cnt = -1
for cd in [3, 4, 5, 6]:
    for nch in [4, 8, 12]:
        for nl1 in [10, 20, 30]:
            cnt += 1
            fig, ax = plt.subplots(1, 2, figsize=(14, 5))
            model = torch.load(ModelDir_CNN + '/model_' + "dep_"+str(cd)+"_chnl_"+str(nch)+"_btn_"+str(nl1) + "_" + version + '.pt')
            
            ax[0].plot(model(torch.FloatTensor(realbbh[[idx]])).detach().numpy()[0,0])
            ax[1].plot(model(torch.FloatTensor(realbbh[[idx]])).detach().numpy()[0,1])
            
            fig.suptitle("dep_"+str(cd)+"_chnl_"+str(nch)+"_btn_"+str(nl1))

In [ ]:
modelName = "dep_"+str(cd)+"_chnl_"+str(nch)+"_btn_"+str(nl1)+"_encoder_"+str(LSTM_encoder)+"_decoder_"+str(LSTM_decoder)
                    
                    with open(ModelDir_CNN + 'model_' + modelName + '_v7.pickle','rb') as handle:
                        model_list = pickle.load(handle)

#### Analyze for CNN+LSTM (inner code: v7)

##### Benchmark: Best AE freq model for BBH

In [22]:
foo = torch.load("./Sida_temp/for_K8S_training/Model/BBH_AE_freq.json")
model_benchmark = foo["mixed_202-20-20"].cpu().eval()

In [23]:
model_benchmark

In [31]:
np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))[1].detach().numpy() - dataset_wsl_fft_all[5000:70000], axis = 1).shape

In [36]:
SIG_class_score = np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))[1].detach().numpy() - dataset_wsl_fft_all[5000:70000], axis = 1)
        
BKG_class_score = np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[70000:80000]))[1].detach().numpy() - dataset_wsl_fft_all[70000:80000], axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

In [37]:
FPR

In [143]:
SIG_class_score = np.var(model_benchmark(torch.FloatTensor(dataset_raw_fft['noise']))[1].detach().numpy() - dataset_raw_fft['noise'], axis = 1)
        
BKG_class_score = np.var(model_benchmark(torch.FloatTensor(dataset_raw_fft['BBH']))[1].detach().numpy() - dataset_raw_fft['BBH'], axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

FPR

In [129]:
dataset_raw_fft['noise'].shape

In [130]:
dataset_wsl_fft_all[5000:70000].shape

In [131]:
np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))[1].detach().numpy() - dataset_wsl_fft_all[5000:70000], axis = 1)

In [132]:
np.var(model_benchmark(torch.FloatTensor(dataset_raw_fft['noise']))[1].detach().numpy() - dataset_raw_fft['noise'], axis = 1)

In [47]:
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[5000:70000]))[1].detach().numpy() - dataset_wsl_fft_all[5000:70000], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'Noise')
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[70000:80000]))[1].detach().numpy() - dataset_wsl_fft_all[70000:80000], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'BBH')
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[80000:90000]))[1].detach().numpy() - dataset_wsl_fft_all[80000:90000], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'SGHF')
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[90000:]))[1].detach().numpy() - dataset_wsl_fft_all[90000:], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'SGLF')
plt.legend()

In [142]:
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_raw_fft['noise']))[1].detach().numpy() - dataset_raw_fft['noise'], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'Noise')
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[70000:80000]))[1].detach().numpy() - dataset_wsl_fft_all[70000:80000], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'BBH')
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[80000:90000]))[1].detach().numpy() - dataset_wsl_fft_all[80000:90000], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'SGHF')
plt.hist(np.var(model_benchmark(torch.FloatTensor(dataset_wsl_fft_all[90000:]))[1].detach().numpy() - dataset_wsl_fft_all[90000:], axis = 1), range = (0,0.008), bins = 50, histtype='step', density = True, label = 'SGLF')
plt.legend()

##### new model

In [88]:
os.getcwd()

In [85]:
np.load('./Sida_temp/for_K8S_training/Data/ind_not_in_wsc_set_glitchH.npy').shape

In [89]:
np.load('./Sida_temp/for_K8S_training/Data/ind_not_in_wsc_set_SGLF_5-12.npy')

In [90]:
np.load('./Sida_temp/for_K8S_training/Data/ind_not_in_wsc_set_BBH_5-12.npy')

In [91]:
np.load('./Sida_temp/for_K8S_training/Data/ind_not_in_wsc_set_SGHF_5-12.npy')

In [5]:
dataset_raw = {}

dataset_raw['noise'] = np.concatenate((np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:65000].reshape(-1,1,200), np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:65000].reshape(-1,1,200)), axis = 1)
dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)


for datatype in ['BBH', 'SGHF']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [:2500])
    
    dataset_raw[datatype] = cache.copy()    
    
    dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
        
for datatype in ['SGLF']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [:2500])
        
    dataset_raw[datatype] = cache.copy()    
    
    dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)

In [6]:
for key in dataset_raw:
    print(dataset_raw[key].shape)
    print(np.sum(np.linalg.norm(dataset_raw[key].reshape(-1,400), axis=-1)))

In [134]:
print(np.linalg.norm(dataset_raw['noise'][0], axis = -1))

In [135]:
print(np.linalg.norm(dataset_raw_fft['noise'][0], axis = -1))

In [140]:
dataset_raw_fft = {}

for key in dataset_raw.keys():
    cache = np.abs(np.fft.rfft(dataset_raw[key], axis=-1))
    cache = cache / np.linalg.norm(cache, axis = -1).reshape(-1,2,1)
    cache = cache.reshape(-1,202)
    dataset_raw_fft[key] = cache.copy()

In [141]:
for key in dataset_raw_fft:
    print(dataset_raw_fft[key].shape)
    print(np.sum(np.linalg.norm(dataset_raw_fft[key], axis=-1)))

In [33]:
ModelDir_CNN = '../K8S_training/CNN_high_snr_train/Output/'

FPR_list_le = np.empty(540)

cnt = -1
for cd in [3, 4, 5, 6]:
    for nch in [4, 8, 12]:
        for nl1 in [10, 20, 30]:
            for LSTM_encoder in [[2,4], [2,8], [2,32], [2,8,4], [2,32,8]]:
                for LSTM_decoder in [[4], [8], [8,8]]:
                    cnt += 1
                    modelName = "dep_"+str(cd)+"_chnl_"+str(nch)+"_btn_"+str(nl1)+"_encoder_"+str(LSTM_encoder)+"_decoder_"+str(LSTM_decoder)
                    if not os.path.exists(ModelDir_CNN + 'model_' + modelName + '_v7.pt'):
                        FPR_list_le[cnt] = np.nan
                        continue
                    model = torch.load(ModelDir_CNN + 'model_' + modelName + '_v7.pt')
                    
                    intermediate, output = model(torch.FloatTensor(dataset_raw['noise'][:10000]))
                    
                    intermediate = intermediate.detach().numpy()
                    
                    output = output.detach().numpy()
                    
                    SIG_class_score = np.sum(np.var(intermediate - output, axis = -1), axis = -1)
                    
                    intermediate, output = model(torch.FloatTensor(dataset_raw['BBH']))
                    
                    intermediate = intermediate.detach().numpy()
                    
                    output = output.detach().numpy()
        
                    BKG_class_score = np.sum(np.var(intermediate - output, axis = -1), axis = -1)

                    SIG_class_score.sort()

                    threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

                    FPR = np.sum(BKG_class_score >= threshold) / len(BKG_class_score)    
                    
                    print(FPR)
                    
                    FPR_list_le[cnt] = FPR              
                    

In [8]:
FPR

In [34]:
plt.hist(FPR_list_le, bins = 50, histtype='step')

In [31]:
plt.hist(FPR_list, bins = 50, histtype='step')

In [36]:
np.nanmin(FPR_list_le)

In [37]:
np.nanargmin(FPR_list_le)

In [38]:
cnt = -1
for cd in [3, 4, 5, 6]:
    for nch in [4, 8, 12]:
        for nl1 in [10, 20, 30]:
            for LSTM_encoder in [[2,4], [2,8], [2,32], [2,8,4], [2,32,8]]:
                for LSTM_decoder in [[4], [8], [8,8]]:
                    cnt += 1
                    modelName = "dep_"+str(cd)+"_chnl_"+str(nch)+"_btn_"+str(nl1)+"_encoder_"+str(LSTM_encoder)+"_decoder_"+str(LSTM_decoder)
                    if cnt != 425:
                        continue
                    print(modelName)
                    model = torch.load(ModelDir_CNN + 'model_' + modelName + '_v7.pt')

In [19]:
model

In [39]:
intermediate, output = model(torch.FloatTensor(dataset_raw['noise'][:10000]))

In [40]:
intermediate.shape

In [41]:
plt.plot(intermediate.detach().numpy()[0][0])
plt.plot(intermediate.detach().numpy()[0][1])

In [42]:
intermediate, output = model(torch.FloatTensor(dataset_raw['BBH'][:10000]))
intermediate.shape
plt.plot(intermediate.detach().numpy()[1][0], label = 'detector L')
plt.plot(intermediate.detach().numpy()[1][1], label = 'detector H')
plt.legend()

In [21]:
intermediate.detach().numpy()

In [20]:
intermediate

In [147]:
np.sum(np.var(np.random.rand(65000,2,200) - np.random.rand(65000,2,200), axis = -1), axis = -1).shape

In [49]:
output = model()

## The Noise WSC is problematic. Consider investigate its fluctuation for fixed AE

### First, Let's fix the AE to investigate the reason for WSC fluc, we will try 4 noise cuts, namely 0.01, 0.05, 0.1 and 0.3 and train 10 WSC for each of structure

#### Below is a trial for noise cut 0.01. If take too much time, shall consider using the cluster

In [ ]:
noise_cut = 0.01



In [ ]:
ae_for_noise = 

## We somehow believe that the Final WSC also requires careful investigation

### The code from Sida

In [291]:
#!/usr/bin/env python
# coding: utf-8

# # Load modules

import numpy as np
import matplotlib.pyplot as plt
import time
import h5py
from scipy.stats import norm

import torch
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from torch import nn, optim
import scipy.io as sio
# import pandas as pd
import datetime
import os
# import readligo as rl
# from gwpy.timeseries import TimeSeries
import math
import random

import copy

import torch.nn.functional as F

import pickle


epochs = 60
rTrain = 0.8;
rTest = 0.1;
# input_vector_length = 100
batch_size = 32
num_bins = 40
coef_delta = 0

class WSC_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()


        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            # self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                # x = self.norm_layers[i](x)

        return x;

def trainWSC_CEloss(wsc, datasets, 
                    outPathAndName,
                    epchs: int=60,
                    bs: int=32, # batch size of the training
                    lr: float=0.00005,
                    weight_class: bool=True,
                    wd: float=0.
                   ):
# datasets: multiple datasets, datasets[0, 1, ...] are for the 1st, 2nd, ... class
# datasets should have the keys to be the integres 0, 1, 2, ...
    
    Nclass = len(datasets)
    wclass = torch.FloatTensor([len(datasets[0])/len(datasets[i]) for i in range(Nclass)])
    nTotal = {}
    nTrain = {}
    nTest = {}
    nbkg_train = 0
    nsig_train = 0
    for i in np.arange(Nclass):
        nTotal[i] = datasets[i].shape[0]
        nTrain[i] = int(rTrain*nTotal[i])
        nTest[i] = int(rTest*nTotal[i])
        if i < 2:
            nbkg_train += nTrain[i]
        else:
            nsig_train += nTrain[i]

    X_train = np.concatenate([datasets[i][:nTrain[i]] for i in range(Nclass)])
    X_test = np.concatenate([datasets[i][-nTest[i]:] for i in range(Nclass)])
    X_validation = np.concatenate([datasets[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    Y_train = np.concatenate([i*np.ones(nTrain[i], dtype=int) for i in np.arange(Nclass)])
    Y_validation = np.concatenate([i*np.ones(nTotal[i]-nTrain[i]-nTest[i], dtype=int) for i in np.arange(Nclass)])

    train_dataset = TensorDataset(torch.FloatTensor(X_train).cuda(), torch.LongTensor(Y_train).cuda())
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).cuda(), torch.LongTensor(Y_validation).cuda())
    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=bs, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=bs, shuffle=True, drop_last=True)

    # associated with a direct sum of probability, see below
    optimizer = optim.Adam(wsc.parameters(), lr=lr, weight_decay=wd)
    loss_func = nn.CrossEntropyLoss(weight=wclass).cuda() if weight_class else nn.CrossEntropyLoss().cuda()
    loss_train = np.empty(epchs)
    loss_validation = np.empty(epchs)

    for epoch in range(epchs):
#         t0 = time.time()
        wsc.train()
        for batchidx, (x, y) in enumerate(trainDataLoader):
            yprime = wsc(x)
            loss = loss_func(yprime, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, (x, y) in enumerate(validationDataLoader):
                yprime = wsc(x)
                lossVal = loss_func(yprime, y)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)

        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
#         print(time.time() - t0)
        
    wsc.cpu().eval()
    
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    foo = ax[1].hist(nn.Softmax(dim=1)(wsc(torch.FloatTensor(X_train))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")
    foo = ax[1].hist(nn.Softmax(dim=1)(wsc(torch.FloatTensor(X_test))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")

    # plt.savefig(outPathAndName)
    plt.show()
    plt.close()
    
    return wsc.cpu().eval()

# dataDir = "../data/Datasets"
# dataDir2 = "../results/SeriesWSC"
# modelDir = "../results/TimeFreqCmp"
# outputDir = "../results/SeriesRAE"

wsc_training_set = torch.load("Sida_temp/for_K8S_training/Data/wsc_training_set_new.json")

with open("Sida_temp/for_K8S_training/filtered_unknown_set_1.pickle", 'rb') as handle:
    wsc_training_set[4] = pickle.load(handle)

#### Below, removing the batchnorm

In [292]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=0.,
                weight_class=True,
                outPathAndName=None
                )

In [293]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=1e-5,
                weight_class=True,
                outPathAndName=None
                )

In [294]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=1e-4,
                weight_class=True,
                outPathAndName=None
                )

In [295]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=1e-6,
                weight_class=True,
                outPathAndName=None
                )

In [296]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=1,
                weight_class=True,
                outPathAndName=None
                )

In [298]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=300,
                bs=256,
                lr=5e-5,
                wd=1e-2,
                weight_class=True,
                outPathAndName=None
                )

In [299]:
testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=300,
                bs=256,
                lr=5e-5,
                wd=1e-3,
                weight_class=True,
                outPathAndName=None
                )

In [301]:
for wdp in [1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1, 1]:
    print(wdp)
    testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=wdp,
                weight_class=True,
                outPathAndName=None
                )

#### Below, Bring back the batchnorm

In [302]:
class WSC_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()


        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        return x;

In [303]:
for wdp in [1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1, 1]:
    print(wdp)
    testwsc = trainWSC_CEloss(WSC_1det_struct([202, 32, 16, 5]).cuda(), wsc_training_set,
                epchs=100,
                bs=256,
                lr=5e-5,
                wd=wdp,
                weight_class=True,
                outPathAndName=None
                )

In [305]:
cnt = -1

for struct in [[202,64,16,8,5],[202,32,16,8,5],[202,64,16,5],[202,64,8,5],[202,32,16,5],[202,32,8,5]]:
    for (batchsize, epochs) in [(384,400),(384,300),(384,200),(256,300),(256,200),(256,100),(128,300),(128,200),(128,100)]:
        for learning_rate in [1e-4, 5e-5, 2e-5, 1e-5, 5e-6, 2e-6, 1e-6]:
            cnt += 1
            if (batchsize, epochs) == (128,900):
                print(cnt, struct, batchsize, epochs, learning_rate)    
            

# Further study of adding the Pearson correlation

## Adding the pearson correlation to our training

### Model definition and training function

In [203]:
class WSC_1det_struct_upd_2_withPcorr(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct_upd_2_withPcorr, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.sm = nn.Softmax(dim=1)
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()
        self.last_linear = nn.Linear(self.encoder_struct[-1] + 1, 1)
        # self.sig = nn.Sigmoid()

        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x, P_corr):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        assert x.shape[0] == P_corr.shape[0]
        
        # print(self.sm(x))
        x = self.last_linear(torch.cat((self.sm(x), P_corr), dim = 1))

        return x;

In [204]:
class WSC_1det_struct_upd_2_withoutPcorr(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct_upd_2_withoutPcorr, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.sm = nn.Softmax(dim=1)
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()
        self.last_linear = nn.Linear(self.encoder_struct[-1], 1)
        # self.sig = nn.Sigmoid()

        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x, P_corr):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        assert x.shape[0] == P_corr.shape[0]
        
        # print(self.sm(x))
        x = self.last_linear(self.sm(x))

        return x;

In [224]:
device = 'cuda:0'

def trainSeriesWSC_struct_upd_2w_withPcorr(datasets, P_corr, struct, save_path):
# the weighted softmax version

    # Detecting input here。 P_corr should have the same size as the datasets

    epochs_wsc = 500
    batch_size_wsc = 256
    lr_wsc = 2e-5
    
    least_epochs = 100
    epochs_interval = 20
    model_list = {}

    wsc = WSC_1det_struct_upd_2_withPcorr(struct).to(device)
    nparam = sum(p.numel() for p in wsc.parameters() if p.requires_grad)
    
    Nclass = len(datasets)
    wclass = torch.FloatTensor([1.]*(Nclass-1)+[len(datasets[0])/len(datasets[Nclass-1])]).to(device)
    nTotal = {}
    nTrain = {}
    nTest = {}
    nbkg_train = 0
    nsig_train = 0
    for i in np.arange(Nclass):
        nTotal[i] = datasets[i].shape[0]
        nTrain[i] = int(rTrain*nTotal[i])
        nTest[i] = int(rTest*nTotal[i])
        if i < 2:
            nbkg_train += nTrain[i]
        else:
            nsig_train += nTrain[i]

    X_train = np.concatenate([datasets[i][:nTrain[i]] for i in range(Nclass)])
    X_test = np.concatenate([datasets[i][-nTest[i]:] for i in range(Nclass)])
    X_validation = np.concatenate([datasets[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    Y_train = np.concatenate([i*np.ones(nTrain[i], dtype=int) for i in np.arange(Nclass)])
    Y_validation = np.concatenate([i*np.ones(nTotal[i]-nTrain[i]-nTest[i], dtype=int) for i in np.arange(Nclass)])

    Z_train = np.concatenate([(np.zeros(nTrain[i]) if i<2 else np.ones(nTrain[i])) for i in np.arange(Nclass)])
    Z_validation = np.concatenate([(np.zeros(nTotal[i]-nTrain[i]-nTest[i]) if i<2 else np.ones(nTotal[i]-nTrain[i]-nTest[i])) for i in np.arange(Nclass)])
    
    P_train = np.concatenate([P_corr[i][:nTrain[i]] for i in range(Nclass)])
    P_test = np.concatenate([P_corr[i][-nTest[i]:] for i in range(Nclass)])
    P_validation = np.concatenate([P_corr[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(Y_train).to(device), torch.FloatTensor(Z_train).to(device), torch.FloatTensor(P_train).to(device))
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).to(device), torch.LongTensor(Y_validation).to(device), torch.FloatTensor(Z_validation).to(device), torch.FloatTensor(P_validation).to(device))
    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)

    # associated with a direct sum of probability, see below
    optimizer = optim.Adam(wsc.parameters(), lr=lr_wsc)
    calcProbClass = nn.Sigmoid().to(device)
    # projection = torch.FloatTensor(np.concatenate(([0, 0], np.ones(Nclass-2)))).to(device)
    # loss_func = nn.BCEWithLogitsLoss(pos_weight=torch.FloatTensor([nbkg_train/nsig_train])).to(device)
    # associated with the linear combination picture, see below (MAY need to change the structure of the network?)
    # loss_func = 
    # optimizer = 
    # optimizer.param_groups.append({'params': extra_params })
    loss_train = np.empty(epochs_wsc)
    loss_validation = np.empty(epochs_wsc)

    for epoch in range(epochs_wsc):
        t0 = time.time()
        wsc.train()
        for batchidx, (x, y, z, p) in enumerate(trainDataLoader):
            # x = x.to(device) # already loaded before
            # y = y.to(device)
            # z = z.to(device)
            # yprime = calcProbClass(wsc(x))
            # this corresponds to a direct summation of the probability along all the signal directions
            # prob = torch.tensordot(yprime, projection, dims=[[1], [0]])#.to(device)
            
            prob = wsc(x,p).flatten()
            loss_func = nn.BCEWithLogitsLoss(weight=wclass[y], pos_weight=torch.FloatTensor([nbkg_train/nsig_train])).to(device)
            loss = loss_func(prob, z)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, (x, y, z, p) in enumerate(validationDataLoader):
                # x = x.to(device)
                # y = y.to(device)
                # z = z.to(device)
                # yprime = calcProbClass(wsc(x))
                # prob = torch.tensordot(yprime, projection, dims=[[1], [0]])#.to(device)
                
                prob = wsc(x, p).flatten()
                
                loss_func = nn.BCEWithLogitsLoss(weight=wclass[y], pos_weight=torch.FloatTensor([nbkg_train/nsig_train])).to(device)
                lossVal = loss_func(prob, z)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)

        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
        print(epoch)
        print(time.time() - t0)

        if ((epoch+1) > least_epochs) and ((epoch+1) % epochs_interval == 0):
            model_list[(epoch+1)] = copy.deepcopy(wsc.cpu().eval())
            model_list[str(epoch+1)+'valloss'] = val_loss
            wsc.to(device)
            
    # wsc.to(device).eval()
    
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    foo = ax[1].hist(calcProbClass(wsc(torch.FloatTensor(X_train).to(device), torch.FloatTensor(P_train).to(device))).cpu().detach().numpy().flatten(), range=(0, 1), bins=20, density=True, histtype="step")
    foo = ax[1].hist(calcProbClass(wsc(torch.FloatTensor(X_test ).to(device), torch.FloatTensor(P_test ).to(device))).cpu().detach().numpy().flatten(), range=(0, 1), bins=20, density=True, histtype="step")

    # plt.show()
    plt.savefig(save_path)
    plt.close()
    
    return model_list

In [225]:
device = 'cuda:0'

def trainSeriesWSC_struct_upd_2w_withoutPcorr(datasets, P_corr, struct, save_path):
# the weighted softmax version

    # Detecting input here。 P_corr should have the same size as the datasets

    epochs_wsc = 500
    batch_size_wsc = 256
    lr_wsc = 2e-5
    
    least_epochs = 100
    epochs_interval = 20
    model_list = {}

    wsc = WSC_1det_struct_upd_2_withoutPcorr(struct).to(device)
    nparam = sum(p.numel() for p in wsc.parameters() if p.requires_grad)
    
    Nclass = len(datasets)
    wclass = torch.FloatTensor([1.]*(Nclass-1)+[len(datasets[0])/len(datasets[Nclass-1])]).to(device)
    nTotal = {}
    nTrain = {}
    nTest = {}
    nbkg_train = 0
    nsig_train = 0
    for i in np.arange(Nclass):
        nTotal[i] = datasets[i].shape[0]
        nTrain[i] = int(rTrain*nTotal[i])
        nTest[i] = int(rTest*nTotal[i])
        if i < 2:
            nbkg_train += nTrain[i]
        else:
            nsig_train += nTrain[i]

    X_train = np.concatenate([datasets[i][:nTrain[i]] for i in range(Nclass)])
    X_test = np.concatenate([datasets[i][-nTest[i]:] for i in range(Nclass)])
    X_validation = np.concatenate([datasets[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    Y_train = np.concatenate([i*np.ones(nTrain[i], dtype=int) for i in np.arange(Nclass)])
    Y_validation = np.concatenate([i*np.ones(nTotal[i]-nTrain[i]-nTest[i], dtype=int) for i in np.arange(Nclass)])

    Z_train = np.concatenate([(np.zeros(nTrain[i]) if i<2 else np.ones(nTrain[i])) for i in np.arange(Nclass)])
    Z_validation = np.concatenate([(np.zeros(nTotal[i]-nTrain[i]-nTest[i]) if i<2 else np.ones(nTotal[i]-nTrain[i]-nTest[i])) for i in np.arange(Nclass)])
    
    P_train = np.concatenate([P_corr[i][:nTrain[i]] for i in range(Nclass)])
    P_test = np.concatenate([P_corr[i][-nTest[i]:] for i in range(Nclass)])
    P_validation = np.concatenate([P_corr[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(Y_train).to(device), torch.FloatTensor(Z_train).to(device), torch.FloatTensor(P_train).to(device))
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).to(device), torch.LongTensor(Y_validation).to(device), torch.FloatTensor(Z_validation).to(device), torch.FloatTensor(P_validation).to(device))
    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)

    # associated with a direct sum of probability, see below
    optimizer = optim.Adam(wsc.parameters(), lr=lr_wsc)
    calcProbClass = nn.Sigmoid().to(device)
    # projection = torch.FloatTensor(np.concatenate(([0, 0], np.ones(Nclass-2)))).to(device)
    # loss_func = nn.BCEWithLogitsLoss(pos_weight=torch.FloatTensor([nbkg_train/nsig_train])).to(device)
    # associated with the linear combination picture, see below (MAY need to change the structure of the network?)
    # loss_func = 
    # optimizer = 
    # optimizer.param_groups.append({'params': extra_params })
    loss_train = np.empty(epochs_wsc)
    loss_validation = np.empty(epochs_wsc)

    for epoch in range(epochs_wsc):
        t0 = time.time()
        wsc.train()
        for batchidx, (x, y, z, p) in enumerate(trainDataLoader):
            # x = x.to(device) # already loaded before
            # y = y.to(device)
            # z = z.to(device)
            # yprime = calcProbClass(wsc(x))
            # this corresponds to a direct summation of the probability along all the signal directions
            # prob = torch.tensordot(yprime, projection, dims=[[1], [0]])#.to(device)
            
            prob = wsc(x,p).flatten()
            loss_func = nn.BCEWithLogitsLoss(weight=wclass[y], pos_weight=torch.FloatTensor([nbkg_train/nsig_train])).to(device)
            loss = loss_func(prob, z)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, (x, y, z, p) in enumerate(validationDataLoader):
                # x = x.to(device)
                # y = y.to(device)
                # z = z.to(device)
                # yprime = calcProbClass(wsc(x))
                # prob = torch.tensordot(yprime, projection, dims=[[1], [0]])#.to(device)
                
                prob = wsc(x, p).flatten()
                
                loss_func = nn.BCEWithLogitsLoss(weight=wclass[y], pos_weight=torch.FloatTensor([nbkg_train/nsig_train])).to(device)
                lossVal = loss_func(prob, z)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)

        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
        print(epoch)
        print(time.time() - t0)

        if ((epoch+1) > least_epochs) and ((epoch+1) % epochs_interval == 0):
            model_list[(epoch+1)] = copy.deepcopy(wsc.cpu().eval())
            model_list[str(epoch+1)+'valloss'] = val_loss
            wsc.to(device)
            
    # wsc.to(device).eval()
    
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    foo = ax[1].hist(calcProbClass(wsc(torch.FloatTensor(X_train).to(device), torch.FloatTensor(P_train).to(device))).cpu().detach().numpy().flatten(), range=(0, 1), bins=20, density=True, histtype="step")
    foo = ax[1].hist(calcProbClass(wsc(torch.FloatTensor(X_test ).to(device), torch.FloatTensor(P_test ).to(device))).cpu().detach().numpy().flatten(), range=(0, 1), bins=20, density=True, histtype="step")

    # plt.show()
    plt.savefig(save_path)
    plt.close()
    
    return model_list

## Module to compute the Pearson correlation

### Testing if the signal is with P_corr -1or1 or just 1

In [393]:
Pearson_test = np.load('../Data_cached/Pearson_test.npy')

In [394]:
Pearson_test.shape

In [136]:
def Compute_P_corr_test(waveform):
    
    delta_max = 100
    
    assert waveform.shape[1:] == (2,400)
    
    num_length = len(waveform)
        
    P_corr_list = np.empty((num_length, 2 * delta_max + 1))
    
    for delta in range(-delta_max, delta_max+1):
        # if delta < 0:
        #     H = waveform[:,1,:].copy()[:,delta_max:]
        #     L = waveform[:,0,:].copy()[:,delta_max+delta:delta]
        # elif delta == 0:
        #     H = waveform[:,1,:].copy()[:,delta_max:]
        #     L = waveform[:,0,:].copy()[:,delta_max:]
        # else:
        #     H = waveform[:,1,:].copy()[:,delta_max-delta:-delta]
        #     L = waveform[:,0,:].copy()[:,delta_max:]
        H = waveform[:,1,:].copy()[:,delta_max:-delta_max]
        L = waveform[:,0,:].copy()[:,delta_max - delta: 200 + delta_max - delta]
            
        # print(H.shape)
        # print(L.shape)
        
        H_mean = np.mean(H, axis = -1).reshape(-1,1)
        L_mean = np.mean(L, axis = -1).reshape(-1,1)
        
        H_linear = H - H_mean
        L_linear = L - L_mean
        
        H_squared = np.square(H_linear)
        L_squared = np.square(L_linear)
    
        nominator = (-1) * np.sum(L_linear * H_linear, axis = 1)
        denominator = np.sqrt((np.sum(L_squared,axis = 1)) * (np.sum(H_squared, axis = 1)))
        
        cache = nominator / denominator
        
        P_corr_list[:,delta + delta_max] = cache.copy()
    
    P_corr_max = np.max(P_corr_list, axis = 1)
    P_corr_min = np.min(P_corr_list, axis = 1)
    
    return P_corr_max, P_corr_min, P_corr_list

In [ ]:
def Compute_P_corr_test(waveform):
    
    delta_max = 100
    
    assert waveform.shape[1:] == (2,400)
    
    num_length = len(waveform)
        
    P_corr_list = np.empty((num_length, 2 * delta_max + 1))
    
    for delta in range(-delta_max, delta_max+1):
        # if delta < 0:
        #     H = waveform[:,1,:].copy()[:,delta_max:]
        #     L = waveform[:,0,:].copy()[:,delta_max+delta:delta]
        # elif delta == 0:
        #     H = waveform[:,1,:].copy()[:,delta_max:]
        #     L = waveform[:,0,:].copy()[:,delta_max:]
        # else:
        #     H = waveform[:,1,:].copy()[:,delta_max-delta:-delta]
        #     L = waveform[:,0,:].copy()[:,delta_max:]
        H = waveform[:,1,:].copy()[:,delta_max:-delta_max]
        L = waveform[:,0,:].copy()[:,delta_max - delta: 200 + delta_max - delta]
            
        # print(H.shape)
        # print(L.shape)
        
        H_mean = np.mean(H, axis = -1).reshape(-1,1)
        L_mean = np.mean(L, axis = -1).reshape(-1,1)
        
        H_linear = H - H_mean
        L_linear = L - L_mean
        
        H_squared = np.square(H_linear)
        L_squared = np.square(L_linear)
    
        nominator = (-1) * np.sum(L_linear * H_linear, axis = 1)
        denominator = np.sqrt((np.sum(L_squared,axis = 1)) * (np.sum(H_squared, axis = 1)))
        
        cache = nominator / denominator
        
        P_corr_list[:,delta + delta_max] = cache.copy()
    
    P_corr_max = np.max(P_corr_list, axis = 1)
    P_corr_min = np.min(P_corr_list, axis = 1)
    
    return P_corr_max, P_corr_min, P_corr_list

In [396]:
max, min, list = Compute_P_corr_test(Pearson_test)

In [364]:
max

In [376]:
plt.hist(max, histtype='step')
plt.hist(np.abs(min), histtype='step')

In [373]:
np.argmax(np.abs(min))

In [379]:
list[4167]

In [375]:
min[4167]

In [374]:
max[4167]

In [371]:
np.argmin((max+min))

In [352]:
Pearson_test

In [388]:
plt.hist(max, histtype='step')
plt.hist(np.abs(min), histtype='step')

In [397]:
np.argmin(min)

In [399]:
max[4167]

In [398]:
min[4167]

In [400]:
list[4167]

### A proper way to compute the P corr is to starting from the initial waveform and do the computation. Here there's one thing not for sure, is that the correlation is not from a 200-timesteps

In [333]:
def algCorrPearson(data):
    res = np.empty((81, len(data)))
    
    mat0 = data[:, 0] - np.array([np.mean(data[:, 0], axis=1)]).transpose();
    mat1 = data[:, 1] - np.array([np.mean(data[:, 1], axis=1)]).transpose();
    
    res[40] = -np.sum(mat0*mat1, axis=1) / (np.sum(mat0**2, axis=1) * np.sum(mat1**2, axis=1))**0.5;
    
    for i in range(1, 41):
        mat0 = data[:, 0, i:] - np.array([np.mean(data[:, 0, i:], axis=1)]).transpose();
        mat1 = data[:, 1, :-i] - np.array([np.mean(data[:, 1, :-i], axis=1)]).transpose();
        res[40-i] = -np.sum(mat0*mat1, axis=1) / (np.sum(mat0**2, axis=1) * np.sum(mat1**2, axis=1))**0.5;
        
        mat0 = data[:, 1, i:] - np.array([np.mean(data[:, 1, i:], axis=1)]).transpose();
        mat1 = data[:, 0, :-i] - np.array([np.mean(data[:, 0, :-i], axis=1)]).transpose();
        res[40+i] = -np.sum(mat0*mat1, axis=1) / (np.sum(mat0**2, axis=1) * np.sum(mat1**2, axis=1))**0.5;
        
    return res;

In [138]:
def max_absolute_value_array(arr):
    # 获取每一行绝对值最大的元素的索引
    max_indices = np.argmax(np.abs(arr), axis=-1)
    # 使用索引返回原数组中的元素
    max_values = arr[np.arange(arr.shape[0]), max_indices]
    return max_values.reshape(-1, 1)  # 将结果重塑为(40000, 1)

In [169]:
def Compute_P_corr(waveform):
    
    delta_max = 40
    
    assert waveform.shape[1:] == (2,200)
    
    num_length = len(waveform)
        
    P_corr_list = np.empty((num_length, 2 * delta_max + 1))
    
    for delta in range(-delta_max, delta_max+1):
    
        H = waveform[:,1,:].copy()[:,max(0, delta):min(200, 200+delta)]
        L = waveform[:,0,:].copy()[:,max(0, -delta):min(200, 200-delta)]
        
        H_mean = np.mean(H, axis = -1).reshape(-1,1)
        L_mean = np.mean(L, axis = -1).reshape(-1,1)
        
        H_linear = H - H_mean
        L_linear = L - L_mean
        
        H_squared = np.square(H_linear)
        L_squared = np.square(L_linear)
    
        nominator = (-1) * np.sum(L_linear * H_linear, axis = 1)
        denominator = np.sqrt((np.sum(L_squared,axis = 1)) * (np.sum(H_squared, axis = 1)))
        
        cache = np.abs(nominator / denominator)
        
        P_corr_list[:,delta + delta_max] = cache.copy()
    
    P_corr = np.max(P_corr_list, axis=-1)
    
    return P_corr

In [408]:
waveform = data_GWAK[-2:-1]

In [419]:
import numpy as np

In [429]:
del max, min

In [430]:
max(0,20)

In [426]:
delta=20
H = waveform[:,1,:].copy()[:,np.max(0,20):np.min(200,160)]
H.shape

In [411]:
Compute_P_corr(waveform)

In [344]:
algCorrPearson(waveform)

In [350]:
(Compute_P_corr(waveform).flatten() - algCorrPearson(waveform).flatten())

In [301]:
(np.sum(L_squared[:,max(0, -delta):min(200, 200-delta)],axis = 1))

In [300]:
np.sum(H_squared[:,max(0, delta):min(200, 200+delta)], axis = 1)

In [302]:
(np.sum(L_squared[:,max(0, -delta):min(200, 200-delta)],axis = 1)) * (np.sum(H_squared[:,max(0, delta):min(200, 200+delta)], axis = 1))

In [311]:
def Compute_P_corr_another(waveform):
    
    delta_max = 40
    
    assert waveform.shape[1:] == (2,200)
    
    num_length = len(waveform)
    
    H = np.pad(waveform[:,1,:].copy(), pad_width=40, mode='constant', constant_values=0)
    L = np.pad(waveform[:,0,:].copy(), pad_width=40, mode='constant', constant_values=0)
    
    H_mean = np.mean(waveform[:,1,:], axis = -1).reshape(-1,1)
    L_mean = np.mean(waveform[:,0,:], axis = -1).reshape(-1,1)
    
    H_linear = H - H_mean
    L_linear = L - L_mean
    
    H_squared = np.square(H_linear)
    L_squared = np.square(L_linear)
    
    P_corr_list = np.empty((num_length, 2 * delta_max + 1))
    
    for delta in range(-delta_max, delta_max+1):
        
        nominator = (-1) * np.sum(L_linear[:,delta_max - delta:delta_max - delta + 200] * H_linear[:,delta_max:delta_max+200], axis = 1)
        denominator = np.sqrt((np.sum(L_squared[:,delta_max - delta:delta_max - delta + 200],axis = 1)) * (np.sum(H_squared[:,:,delta_max:delta_max+200], axis = 1)))
        
        cache = nominator / denominator
        
        P_corr_list[:,delta + delta_max] = cache.copy()
    
    P_corr = np.max(P_corr_list, axis = 1)
    
    return P_corr

In [217]:
np.array([[1],[2],[3]]) / np.array([[2],[3],[4]])

In [264]:
np.max(np.array([[1,2,3],[4,5,6]]), axis = 1)

In [140]:
data_GWAK = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/bbh_for_challenge.npy')

In [141]:
data_GWAK.shape

In [142]:
data_GWAK[1:2].shape

In [143]:
waveform = data_GWAK[0:2]

In [144]:
H = waveform[:,1,:].copy()
L = waveform[:,0,:].copy()

H_mean = np.mean(waveform[:,1,:], axis = -1).reshape(-1,1)
L_mean = np.mean(waveform[:,0,:], axis = -1).reshape(-1,1)

H_linear = H - H_mean
L_linear = L - L_mean

H_squared = np.square(H_linear)
L_squared = np.square(L_linear)

In [145]:
print(L_squared.shape)

In [146]:
L_linear.shape

In [147]:
L_linear[:,max(0, -delta):min(200, 200-delta)].shape

In [148]:
np.sum(L_linear[:,max(0, -delta):min(200, 200-delta)] * H_linear[:,max(0, delta):min(200, 200+delta)], axis = 1)

In [275]:
L_squared[:,max(0, -delta):min(200, 200-delta)].shape

In [257]:
np.sqrt((np.sum(L_squared[:,max(0, -delta):min(200, 200-delta)])) * (np.sum(H_squared[:,max(0, delta):min(200, 200+delta)])))

In [259]:
num_length = 1
delta_max = 40

P_corr_list = np.empty((num_length, 2 * delta_max + 1))
    
for delta in range(-delta_max, delta_max+1):
    
    nominator = (-1) * np.sum(L_linear[:,max(0, -delta):min(200, 200-delta)] * H_linear[:,max(0, delta):min(200, 200+delta)])
    denominator = np.sqrt((np.sum(L_squared[:,max(0, -delta):min(200, 200-delta)])) * (np.sum(H_squared[:,max(0, delta):min(200, 200+delta)])))
    
    cache = nominator / denominator
    
    P_corr_list[:,delta + delta_max] = cache.copy()

P_corr = np.max(P_corr_list, axis = 1)

In [265]:
P_corr_list

In [313]:
from scipy.stats import pearsonr

pearsonr(data_GWAK[0,0],data_GWAK[0,1])

In [163]:
P_corr = Compute_P_corr(data_GWAK).flatten()
negative_idx = P_corr<0

In [164]:
np.sum(negative_idx)

In [165]:
negative_idx

In [167]:
for i in range(10):
    print(P_corr[negative_idx][i])
    print(P_corr_GWAK[negative_idx][i])

In [320]:
algCorrPearson(data_GWAK[0:1])

In [260]:
P_corr

In [245]:
P_corr_list

In [405]:
data_GWAK.shape

In [431]:
Compute_P_corr(data_GWAK)

In [292]:
Compute_P_corr(data_GWAK[-2:-1]) - P_corr_GWAK[-2]

In [288]:
validation

In [293]:
np.max(np.abs(validation))

In [153]:
P_corr_GWAK = np.load('../Data_cached/Pearson_GWAK_BBH.npy')

In [154]:
P_corr_GWAK

In [325]:
algCorrPearson(data_GWAK[-2:-1])

In [317]:
Compute_P_corr(data_GWAK[-2:-1])

### Benchmark, making the set with the Pearson corr and do the supervised first

In [170]:
def Normalize_FFT_and_Pearson(dataset):
    
    assert dataset.shape[1:] == (2,200)
    
    P_Corr = Compute_P_corr(dataset).reshape(-1,1)
    
    Normalize = dataset.copy()
    Normalize = Normalize.reshape(-1,200)
    Normalize = Normalize / np.linalg.norm(Normalize, axis = -1).reshape(-1,1)
    Normalize = Normalize.reshape(-1,2,200)
    
    # FFT = Normalize.copy()
    FFT = np.abs(np.fft.rfft(Normalize, axis = -1))
    FFT = FFT / np.linalg.norm(FFT, axis = -1).reshape(-1,2,1)
    
    return Normalize, FFT, P_Corr

In [484]:
# dataset_raw = {}
dataset_fft = {}
dataset_Pcorr = {}

noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:400000]
noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:400000]

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

np.random.shuffle(glitch_raw_L)
np.random.shuffle(glitch_raw_H)

glitch_set = np.concatenate((
    np.concatenate((noise_raw_L[80000:120000].reshape(-1,1,200), glitch_raw_H[:40000].reshape(-1,1,200)), axis = 1), 
    np.concatenate((glitch_raw_L[:40000].reshape(-1,1,200), noise_raw_H[80000:120000].reshape(-1,1,200)), axis = 1)
)
                            , axis = 0)

# dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
# dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
# dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

# dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
# dataset_fft['noise'] = 
# dataset_fft['noise'] = dataset_fft['noise'] / 

_, dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:80000].reshape(-1,1,200), noise_raw_H[:80000].reshape(-1,1,200)), axis = 1))

_, dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H


for datatype in ['BBH', 'SGHF', 'SGLF']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [:20000], axis=0)
    
    _, dataset_fft[datatype], dataset_Pcorr[datatype] = Normalize_FFT_and_Pearson(cache)
    
    # dataset_raw[datatype] = cache.copy()    
    
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
        
# for datatype in ['SGLF']:
#     cache = np.empty((0,2,200))
#     for snr in ['5-12','12-24','24-48','48-96']:
#         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
#                           [:2500])
        
#     dataset_raw[datatype] = cache.copy()    
    
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
#     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)

In [230]:
# Making the testing set

dataset_raw = {}
dataset_fft = {}
dataset_Pcorr = {}

noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[200000:400000]
noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[200000:400000]

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

np.random.shuffle(glitch_raw_L)
np.random.shuffle(glitch_raw_H)

glitch_set = np.concatenate((
    np.concatenate((noise_raw_L[65000:67500].reshape(-1,1,200), glitch_raw_H[40000:42500].reshape(-1,1,200)), axis = 1), 
    np.concatenate((glitch_raw_L[40000:42500].reshape(-1,1,200), noise_raw_H[65000:67500].reshape(-1,1,200)), axis = 1)
)
                            , axis = 0)

# dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
# dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
# dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

# dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
# dataset_fft['noise'] = 
# dataset_fft['noise'] = dataset_fft['noise'] / 

_, dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1))

_, dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H


for datatype in ['BBH', 'SGHF', 'SGLF']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [20000:22500], axis=0)
    
    _, dataset_fft[datatype], dataset_Pcorr[datatype] = Normalize_FFT_and_Pearson(cache)
    
    # dataset_raw[datatype] = cache.copy()    
    
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
        
# for datatype in ['SGLF']:
#     cache = np.empty((0,2,200))
#     for snr in ['5-12','12-24','24-48','48-96']:
#         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
#                           [:2500])
        
#     dataset_raw[datatype] = cache.copy()    
    
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
#     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)

# For high snr set, we also do the uncorrelation and the noise replacement job

noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[400000:407500].reshape(-1,1,200)
noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[400000:407500].reshape(-1,1,200)

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

noise_standby = np.concatenate((noise_raw_L, noise_raw_H), axis = 1)

i = 0
for datatype in ['BBH', 'SGHF', 'SGLF']:
    cache = np.empty((0,2,200))
    for snr in ['48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [40000:42500], axis=0)
    
    # _, dataset_fft[datatype+'highsnrraw'], dataset_Pcorr[datatype+'highsnrraw'] = Normalize_FFT_and_Pearson(cache)
    
    # np.random.shuffle(cache[:,0,:])
    
    # _, dataset_fft[datatype+'uncorrelated'], dataset_Pcorr[datatype+'uncorrelated'] = Normalize_FFT_and_Pearson(cache)
    
    # events_pick = np.random.choice(2500,2500, replace = False)
    # detector_pick = np.random.choice(2,2500)
    
    # cache[events_pick, detector_pick, :] = noise_standby[range(i*2500, i*2500+2500),detector_pick, :]
    
    # _, dataset_fft[datatype+'halfsig'], dataset_Pcorr[datatype+'halfsig'] = Normalize_FFT_and_Pearson(cache)
    
    i += 1

In [231]:
for key in dataset_fft.keys():
    print(key)
    print(dataset_fft[key].shape)
    print(np.sum(np.linalg.norm(dataset_fft[key].reshape(-1,202), axis = -1)))
    print(dataset_Pcorr[key].mean())

In [232]:
dataset_all = {}
dataset_all['fft'] = dataset_fft.copy()
dataset_all['Pcorr'] = dataset_Pcorr.copy()

In [233]:
for key in dataset_fft.keys():
    print(key)
    perm = np.random.choice(len(dataset_fft[key]), len(dataset_fft[key]), replace=False)
    dataset_fft[key] = dataset_fft[key][perm]
    dataset_Pcorr[key] = dataset_Pcorr[key][perm]
    print(dataset_fft[key].shape)
    print(np.sum(np.linalg.norm(dataset_fft[key].reshape(-1,202), axis = -1)))
    print(dataset_Pcorr[key].mean())

In [234]:
dataset_all['fft_shuffled'] = dataset_fft.copy()
dataset_all['Pcorr_shuffled'] = dataset_Pcorr.copy()

In [235]:
dataset_all['fft']['noise'][0] - dataset_all['fft_shuffled']['noise'][0]

In [634]:
with open('../Data_cached/Tentative_set_with_shuffled.pickle', 'wb') as handle:
    pickle.dump(dataset_all, handle)

In [241]:
with open('../Data_cached/Tentative_set_with_shuffled.pickle', 'rb') as handle:
    dataset_all = pickle.load(handle)

In [242]:
dataset_all

In [243]:
dataset_all['fft_shuffled'].keys()

In [244]:
for key in dataset_all['fft_shuffled'].keys():
    print(dataset_all['fft_shuffled'][key].shape)

In [726]:
del dataset_all

In [245]:
dataset_for_training = {}
dataset_for_training_Pcorr = {}
i = 0
for key in ['glitch','noise','BBH','SGHF']:
    dataset_for_training[i] = dataset_all['fft_shuffled'][key].reshape(-1,202)
    dataset_for_training_Pcorr[i] = dataset_all['Pcorr_shuffled'][key].reshape(-1,1)
    i+=1

In [246]:
dataset_for_training.keys()

In [247]:
for i in range(4):
    print(dataset_for_training_Pcorr[i].shape)

In [729]:
torch.save(dataset_for_training, '../Data_cached/wsc_training_set_new_80k_withPcorr_shuffled.json')
torch.save(dataset_for_training_Pcorr, '../Data_cached/wsc_training_set_new_80k_withPcorr_Pcorr_shuffled.json')

In [706]:
dataset_all['fft'].keys()

In [635]:
with open('../Data_cached/Tentative_set_with_Pcorr_testing.pickle', 'wb') as handle:
    pickle.dump(dataset_all, handle)

In [636]:
with open('../Data_cached/Tentative_set_with_Pcorr_testing.pickle', 'rb') as handle:
    dataset_all = pickle.load(handle)

In [190]:
dataset_for_training_5 = {}

for trainingsettime in range(1):
    # dataset_raw = {}
    dataset_fft = {}
    dataset_Pcorr = {}

    noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:400000]
    noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:400000]

    np.random.shuffle(noise_raw_L)
    np.random.shuffle(noise_raw_H)

    glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
    glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

    np.random.shuffle(glitch_raw_L)
    np.random.shuffle(glitch_raw_H)

    glitch_set = np.concatenate((
        np.concatenate((noise_raw_L[80000:120000].reshape(-1,1,200), glitch_raw_H[:40000].reshape(-1,1,200)), axis = 1), 
        np.concatenate((glitch_raw_L[:40000].reshape(-1,1,200), noise_raw_H[80000:120000].reshape(-1,1,200)), axis = 1)
    )
                                , axis = 0)

    # dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
    # dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
    # dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
    # dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
    # dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

    # dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
    # dataset_fft['noise'] = 
    # dataset_fft['noise'] = dataset_fft['noise'] / 

    _, dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:80000].reshape(-1,1,200), noise_raw_H[:80000].reshape(-1,1,200)), axis = 1))

    _, dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

    del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H

    perm = np.random.choice(40000,20000,replace=False)

    for datatype in ['BBH', 'SGHF', 'SGLF']:
        cache = np.empty((0,2,200))
        for snr in ['5-12','12-24','24-48','48-96']:
            cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                            [:40000][perm][:20000], axis=0)
        
        _, dataset_fft[datatype], dataset_Pcorr[datatype] = Normalize_FFT_and_Pearson(cache)
        
        # dataset_raw[datatype] = cache.copy()    
        
        # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
        # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
        # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
            
    # for datatype in ['SGLF']:
    #     cache = np.empty((0,2,200))
    #     for snr in ['5-12','12-24','24-48','48-96']:
    #         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
    #                           [:2500])
            
    #     dataset_raw[datatype] = cache.copy()    
        
    #     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    #     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    #     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)

    for key in dataset_fft.keys():
        # print(key)
        perm = np.random.choice(len(dataset_fft[key]), len(dataset_fft[key]), replace=False)
        dataset_fft[key] = dataset_fft[key][perm]
        dataset_Pcorr[key] = dataset_Pcorr[key][perm]
        # print(dataset_fft[key].shape)
        # print(np.sum(np.linalg.norm(dataset_fft[key].reshape(-1,202), axis = -1)))
        # print(dataset_Pcorr[key].mean())

    dataset_for_training = {}
    dataset_for_training_Pcorr = {}

    i = 0
    for key in ['glitch','noise','BBH','SGHF','SGLF']:
        dataset_for_training[i] = dataset_fft[key].reshape(-1,202)
        dataset_for_training_Pcorr[i] = dataset_Pcorr[key].reshape(-1,1)
        i+=1

    dataset_for_training_5[str(trainingsettime) + 'waveform'] = dataset_for_training.copy()
    dataset_for_training_5[str(trainingsettime) + 'Pcorr'] = dataset_for_training_Pcorr.copy()

In [252]:
dataset_for_testing_5 = {}

for testingsettime in range(5):
    # dataset_raw = {}
    dataset_fft = {}
    dataset_Pcorr = {}

    noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[400000:800000]
    noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[400000:800000]

    np.random.shuffle(noise_raw_L)
    np.random.shuffle(noise_raw_H)

    glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
    glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

    np.random.shuffle(glitch_raw_L)
    np.random.shuffle(glitch_raw_H)

    glitch_set = np.concatenate((
        np.concatenate((noise_raw_L[80000:82500].reshape(-1,1,200), glitch_raw_H[:2500].reshape(-1,1,200)), axis = 1), 
        np.concatenate((glitch_raw_L[:2500].reshape(-1,1,200), noise_raw_H[80000:82500].reshape(-1,1,200)), axis = 1)
    )
                                , axis = 0)

    # dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
    # dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
    # dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
    # dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
    # dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

    # dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
    # dataset_fft['noise'] = 
    # dataset_fft['noise'] = dataset_fft['noise'] / 

    _, dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1))

    _, dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

    del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H

    perm = np.random.choice(14999,14999,replace=False)

    for datatype in ['BBH', 'SGHF', 'SGLF']:
        cache = np.empty((0,2,200))
        for snr in ['5-12','12-24','24-48','48-96']:
            cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                            [40000:54999][perm][:2500], axis=0)
        
        _, dataset_fft[datatype], dataset_Pcorr[datatype] = Normalize_FFT_and_Pearson(cache)
        
        # dataset_raw[datatype] = cache.copy()    
        
        # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
        # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
        # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
            
    # for datatype in ['SGLF']:
    #     cache = np.empty((0,2,200))
    #     for snr in ['5-12','12-24','24-48','48-96']:
    #         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
    #                           [:2500])
            
    #     dataset_raw[datatype] = cache.copy()    
        
    #     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    #     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    #     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)

    # for key in dataset_fft.keys():
    #     # print(key)
    #     perm = np.random.choice(len(dataset_fft[key]), len(dataset_fft[key]), replace=False)
    #     dataset_fft[key] = dataset_fft[key][perm]
    #     dataset_Pcorr[key] = dataset_Pcorr[key][perm]
        # print(dataset_fft[key].shape)
        # print(np.sum(np.linalg.norm(dataset_fft[key].reshape(-1,202), axis = -1)))
        # print(dataset_Pcorr[key].mean())

    dataset_for_testing = np.empty((0,202))
    dataset_for_testing_Pcorr = np.empty((0,1))

    i = 0
    for key in ['glitch','noise','BBH','SGHF','SGLF']:
        dataset_for_testing = np.append(dataset_for_testing, dataset_fft[key].reshape(-1,202), axis=0)
        dataset_for_testing_Pcorr = np.append(dataset_for_testing_Pcorr, dataset_Pcorr[key].reshape(-1,1), axis = 0)
        i+=1

    dataset_for_testing_5[str(testingsettime) + 'waveform'] = dataset_for_testing.copy()
    dataset_for_testing_5[str(testingsettime) + 'Pcorr'] = dataset_for_testing_Pcorr.copy()

In [253]:
dataset_for_testing_5['0waveform'].shape

In [192]:
dataset_for_training_5['0Pcorr']

In [271]:
torch.save(dataset_for_training_5, '../Data_cached/dataset_for_training_supervised_withPearson_1sets.json')

In [272]:
torch.save(dataset_for_testing_5, '../Data_cached/dataset_for_testing_supervised_withPearson_5sets.json')

### Training the model

In [211]:
trans_dict = {'glitch':0, 
              'noise':1, 
              'BBH':2, 
              'SGHF':3, 
              'SGLF':4}

In [226]:
model_list = {}

for trainingsettime in range(1):
    for training_scheme in range(4):

        dataset_for_training = dataset_for_training_5[str(trainingsettime) + 'waveform'].copy()
        dataset_for_training_P_corr = dataset_for_training_5[str(trainingsettime) + 'Pcorr'].copy()

        if training_scheme == 0:
            for training_time in range(5):
                # continue
                model = return_model_with_least_valloss(trainSeriesWSC_struct_upd_2w_withoutPcorr(dataset_for_training, dataset_for_training_P_corr, [202,64,16,5], 'PearsonSupervisedStudy_{}{}{}'.format(trainingsettime, training_scheme, training_time)))
                model_list['{}{}{}'.format(trainingsettime, training_scheme, training_time)] = copy.deepcopy(model)
                print('{}-th time for training scheme {} and training set {} finished. '.format(training_time, training_scheme, trainingsettime))
        else:
            if training_scheme == 1:
                for key in dataset_for_training_P_corr.keys():
                    dataset_for_training_P_corr[key] = np.random.uniform(0,1,len(dataset_for_training_P_corr[key])).reshape(-1,1)

            elif training_scheme == 2:
                for key in dataset_for_training_P_corr.keys():
                    dataset_for_training_P_corr[key] = np.ones(len(dataset_for_training_P_corr[key])).reshape(-1,1)

            for training_time in range(5):
                model = return_model_with_least_valloss(trainSeriesWSC_struct_upd_2w_withPcorr(dataset_for_training, dataset_for_training_P_corr, [202,64,16,5], 'PearsonSupervisedStudy_{}{}{}'.format(trainingsettime, training_scheme, training_time)))
                model_list['{}{}{}'.format(trainingsettime, training_scheme, training_time)] = copy.deepcopy(model)
                print('{}-th time for training scheme {} and training set {} finished. '.format(training_time, training_scheme, trainingsettime))

        


    

In [228]:
model_list

In [229]:
torch.save(model_list, '../Model_cached/With_Pearson_correlation_fluc.json')

In [202]:
dataset_for_training_P_corr

In [495]:
dataset_all['Pcorr_shuffled']['noise'] - dataset_Pcorr['noise']

In [599]:
dataset_all['fft_shuffled'].keys()

In [602]:
dataset_fft_dicted = {}
dataset_Pcorr_dicted = {}

for key in dataset_all['fft_shuffled'].keys():
    dataset_fft_dicted[trans_dict[key]] = dataset_all['fft_shuffled'][key].reshape(-1,202)
    dataset_Pcorr_dicted[trans_dict[key]] = dataset_all['Pcorr_shuffled'][key]

In [250]:
dataset_fft_dicted = {}
dataset_Pcorr_dicted = {}

for key in dataset_all['fft_shuffled'].keys():
    dataset_fft_dicted[trans_dict[key]] = dataset_all['fft_shuffled'][key].reshape(-1,202)
    dataset_Pcorr_dicted[trans_dict[key]] = np.random.uniform(0,1,len(dataset_all['fft_shuffled'][key])).reshape(-1,1)

In [251]:
for key in dataset_fft_dicted.keys():
    print(dataset_fft_dicted[key].shape)

In [252]:
dataset_Pcorr_dicted[1]

In [613]:
model_list = trainSeriesWSC_struct_upd_2w_withPcorr(dataset_fft_dicted, dataset_Pcorr_dicted, [202,64,16,5], 'null')

In [501]:
print('already finished')

In [259]:
model_list_withoutPcorr = trainSeriesWSC_struct_upd_2w_withoutPcorr(dataset_fft_dicted, dataset_Pcorr_dicted, [202,64,16,5], 'null')

In [502]:
model_list

In [608]:
torch.save(model_list, '../Model_cached/Tentative_5-class_supervised_with_uniform_Pearson.json')

In [614]:
torch.save(model_list, '../Model_cached/Tentative_5-class_supervised_with_random_Pearson.json')

In [337]:
model_list = torch.load('../Model_cached/Tentative_5-class_supervised_with_Pearson.json')
model = return_model_with_least_valloss(model_list)

In [339]:
dataset_fft.keys()

In [237]:
dataset_wsl_fft_all_test = np.empty((0,202))
dataset_wsl_fft_all_test_Pcorr = np.empty((0,1))

for key in ['glitch', 'noise', 'BBH', 'SGHF', 'SGLF']:
    dataset_wsl_fft_all_test = np.append(dataset_wsl_fft_all_test, dataset_fft[key].reshape(-1,202), axis = 0)
    dataset_wsl_fft_all_test_Pcorr = np.append(dataset_wsl_fft_all_test_Pcorr, dataset_Pcorr[key].reshape(-1,1), axis = 0)

In [238]:
dataset_wsl_fft_all_test.shape

In [239]:
dataset_wsl_fft_all_test_Pcorr.shape

In [240]:
dataset_wsl_fft_all_test_Pcorr[8156]

In [732]:
dataset_wsl_fft_all_test_shuffled = dataset_wsl_fft_all_test.copy()
dataset_wsl_fft_all_test_Pcorr_shuffled = dataset_wsl_fft_all_test_Pcorr.copy()

perm = np.random.choice(100000,100000,replace=False)
dataset_wsl_fft_all_test_shuffled = dataset_wsl_fft_all_test_shuffled[perm]
dataset_wsl_fft_all_test_Pcorr_shuffled = dataset_wsl_fft_all_test_Pcorr_shuffled[perm]

np.save('../Data_cached/wsc_test_set_new_LM_withPcorr_shuffled.npy', dataset_wsl_fft_all_test_shuffled)
np.save('../Data_cached/wsc_test_set_new_LM_withPcorr_Pcorr_shuffled.npy', dataset_wsl_fft_all_test_Pcorr_shuffled)

In [216]:
os.getcwd()

In [217]:
dataset_wsl_fft_all_test_shuffled = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_withPcorr_shuffled.npy')
dataset_wsl_fft_all_test_Pcorr_shuffled = np.load('./Sida_temp/for_K8S_training/Data/wsc_test_set_new_LM_withPcorr_Pcorr_shuffled.npy')

In [735]:
perm

In [734]:
dataset_wsl_fft_all_test_shuffled

In [218]:
dataset_wsl_fft_all_test_Pcorr_shuffled

In [738]:
dataset_wsl_fft_all_test_Pcorr_shuffled

In [514]:
model

In [238]:
BKG_class_score = nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_test[:70000]), torch.FloatTensor(dataset_wsl_fft_all_test_Pcorr[:70000]))).detach().numpy().flatten()
    
SIG_class_score = nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_test[70000:]), torch.FloatTensor(dataset_wsl_fft_all_test_Pcorr[70000:]))).detach().numpy().flatten()

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

In [541]:
BKG_class_score

In [542]:
threshold

In [543]:
SIG_class_score

In [677]:
FPR

In [236]:
# FPR with Pearson, 2nd time check
FPR

In [233]:
# FPR with uniform Pearson
FPR

In [239]:
# FPR with random Pearson
FPR

In [545]:
plt.hist(SIG_class_score, histtype='step', bins = 50, density = True)
plt.hist(BKG_class_score, histtype='step', bins = 50, density = True)
plt.axvline(threshold)

In [527]:
np.sum(SIG_class_score > threshold)

In [537]:
SIG_class_score = SIG_class_score.flatten()

In [538]:
SIG_class_score.sort()

In [528]:
int(0.1 * len(SIG_class_score))

In [532]:
SIG_class_score.sort()

In [539]:
SIG_class_score

In [582]:
dataset_wsl_fft_all_test.shape

In [254]:
model_list

In [255]:
testing_set = dataset_for_testing_5['0waveform']
testing_set_Pcorr = dataset_for_testing_5['0Pcorr']

FPR_list = np.empty((4,5))

for training_scheme in range(4):
    for training_time in range(5):
        
        model = model_list['0{}{}'.format(training_scheme, training_time)]
        
        BKG_class_score = nn.Sigmoid()(model(torch.FloatTensor(testing_set[:70000]), torch.FloatTensor(testing_set_Pcorr[:70000]))).detach().numpy().flatten()
    
        SIG_class_score = nn.Sigmoid()(model(torch.FloatTensor(testing_set[70000:]), torch.FloatTensor(testing_set_Pcorr[70000:]))).detach().numpy().flatten()

        SIG_class_score.sort()

        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list[training_scheme, training_time] = FPR

In [257]:
FPR_list

In [258]:
np.mean(FPR_list, axis = -1)

In [259]:
np.var(FPR_list, axis = -1)

In [265]:
model_list_best = {}

model_list_best[0] = model_list['001']
model_list_best[1] = model_list['030']
model_list_best[2] = model_list['012']
model_list_best[3] = model_list['022']

In [266]:
testing_set = dataset_for_testing_5['0waveform']
testing_set_Pcorr = dataset_for_testing_5['0Pcorr']

FPR_list = np.empty((4,5))

for training_scheme in range(4):
    
    model = model_list_best[training_scheme]
    
    for testing_time in range(5):
        
        testing_set = dataset_for_testing_5[str(testing_time) + 'waveform'].copy()
        testing_set_Pcorr = dataset_for_testing_5[str(testing_time) + 'Pcorr'].copy()
        
        BKG_class_score = nn.Sigmoid()(model(torch.FloatTensor(testing_set[:70000]), torch.FloatTensor(testing_set_Pcorr[:70000]))).detach().numpy().flatten()
    
        SIG_class_score = nn.Sigmoid()(model(torch.FloatTensor(testing_set[70000:]), torch.FloatTensor(testing_set_Pcorr[70000:]))).detach().numpy().flatten()

        SIG_class_score.sort()

        threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

        FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
        
        FPR_list[training_scheme, testing_time] = FPR

In [267]:
FPR_list

In [268]:
np.mean(FPR_list, axis = -1)

In [269]:
np.var(FPR_list, axis = -1)

### Score distribution

In [637]:
dataset_all.keys()

In [641]:
dataset_fft = dataset_all['fft']
dataset_Pcorr = dataset_all['Pcorr']

In [640]:
for key in dataset_fft.keys():
    print(key)
    print(dataset_fft[key].shape)

In [697]:
dataset_wsl_separated = {}
dataset_wsl_separated_Pcorr = {}

dataset_wsl_separated[0] = dataset_wsl_fft_all_test[:5000].copy().reshape(-1,202)
dataset_wsl_separated[1] = dataset_wsl_fft_all_test[5000:70000].copy().reshape(-1,202)
dataset_wsl_separated[2] = dataset_wsl_fft_all_test[70000:80000].copy().reshape(-1,202)
dataset_wsl_separated[3] = dataset_wsl_fft_all_test[80000:90000].copy().reshape(-1,202)
dataset_wsl_separated[4] = dataset_wsl_fft_all_test[90000:].copy().reshape(-1,202)

dataset_wsl_separated_Pcorr[0] = dataset_wsl_fft_all_test_Pcorr[:5000].copy().reshape(-1,1)
dataset_wsl_separated_Pcorr[1] = dataset_wsl_fft_all_test_Pcorr[5000:70000].copy().reshape(-1,1)
dataset_wsl_separated_Pcorr[2] = dataset_wsl_fft_all_test_Pcorr[70000:80000].copy().reshape(-1,1)
dataset_wsl_separated_Pcorr[3] = dataset_wsl_fft_all_test_Pcorr[80000:90000].copy().reshape(-1,1)
dataset_wsl_separated_Pcorr[4] = dataset_wsl_fft_all_test_Pcorr[90000:].copy().reshape(-1,1)

dataset_wsl_separated['bg'] = dataset_wsl_fft_all_test[5000:70000].copy()
dataset_wsl_separated_Pcorr['bg'] = dataset_wsl_fft_all_test_Pcorr[5000:70000].copy()


i = 0
for type in ['BBH', 'SGHF', 'SGLF']:
    j = 0
    for snr in ['5-12','12-24','24-48','48-96']:
        dataset_wsl_separated[type + '_' + snr] = dataset_wsl_separated[i+2][j * 2500:j * 2500 + 2500].copy()
        dataset_wsl_separated_Pcorr[type + '_' + snr] = dataset_wsl_separated_Pcorr[i+2][j * 2500:j * 2500 + 2500].copy()
        # print()
        j+=1
    i+=1

In [643]:
trans_dict = {'BBH':'bbh',
              'SGHF':'hfsg',
              'SGLF':'lfsg'}
trans_dict_1 = {'bbh':'BBH',
              'hfsg':'SGHF',
              'lfsg':'SGLF'}

In [645]:
model

In [698]:
# model_number = 1321

# with open(ModelDir + '/FinalWSC_{}_ratio_scan_FFTTTv26.pickle'.format(model_number), 'rb') as handle:
#     model_list = pickle.load(handle)
    
# # with open(ModelDir + '/FinalWSC_test.pickle', 'rb') as handle:
# #     model_list = pickle.load(handle)

# valloss = np.empty(0)

# for k in range(len(epochs_list)):
#     valloss = np.append(valloss, model_list[str(epochs_list[k]) + 'valloss'])

# model = model_list[epochs_list[np.argmin(valloss)]]

ModelDir_LastWSC = "../K8S_training/LastWSCtest/Output"
model_list = torch.load(ModelDir_LastWSC + "/WSCmodel_{}_retraintime{}_v11.pt".format(0,0), map_location='cpu')

val_loss = np.empty(0)
for k in range(len(range(120,501,20))):
    val_loss = np.append(val_loss, model_list[str(epochs_list[k])+'valloss'])
model_2 = model_list[epochs_list[np.argmin(val_loss)]]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

threshold_list = {}
FPR_list = {}

fig, axs = plt.subplots(4, 3, figsize = (19*1.5,20*1.5))

row = 0

for type in datatype_list[:5]:
    if type in ['glitch', 'bg']:
        continue
        datatype = type
        plt.title('Score distribution of datatype {}'.format(datatype))
        Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
        plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
        print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
        
        
        for idx in range(1):
            blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            selected_idx = blind_test_set_ans[:,1] == type
            print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[selected_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))

            plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = type + '_{}'.format(idx), color = colors[idx+1])
            
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
                
        plt.legend()
        plt.show()
        
    else:
        col = 0
        for snr in snr_range:
            datatype = trans_dict_1[type] + '_' + snr
            
            Score_bg = np.sum(nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_separated['bg']), torch.FloatTensor(dataset_wsl_separated_Pcorr['bg']))).detach().numpy(), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            # if type != 'ccsn':
            #     Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            #     plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = datatype + '_benchmark', color = colors[0])
            #     print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated[datatype])))
            
            # for idx in range(5):
            # blind_test_set = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_loud_{}.npy'.format(idx))
            # blind_test_set_ans = np.load('./Sida_temp/for_K8S_training/Data/blind_dataset_ans_loud_{}.npy'.format(idx))
            
            # blind_test_set = blind_test_set / np.linalg.norm(blind_test_set, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = np.abs(np.fft.rfft(blind_test_set, axis = -1))
            # blind_test_set_ffted = blind_test_set_ffted / np.linalg.norm(blind_test_set_ffted, axis = -1).reshape(-1,2,1)
            
            # blind_test_set_ffted = blind_test_set_ffted.reshape(-1,202)
            
            # noise_idx = (blind_test_set_ans[:,0] == '0')
            # signal_idx = (blind_test_set_ans[:,0] == '1')
            
            # Noise_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # Signal_score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[signal_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            
            # Signal_score.sort()
            
            # threshold = Signal_score[int(0.1 * len(Signal_score))]
            
            # FPR = np.sum(Noise_score > threshold) / len(Noise_score)
            
            # print(FPR)
            
            # selected_idx = np.logical_and(blind_test_set_ans[:,1] == type, blind_test_set_ans[:,2] == snr)
            # print('Totally {} events in blind set {}.'.format(np.sum(selected_idx), idx))
            # if np.sum(selected_idx) < 100:
            #     print('Not enough events, skip plotting. ')
            #     continue
            
            # Score = np.sum(nn.Softmax(dim = 1)(model(torch.FloatTensor(blind_test_set_ffted[noise_idx]))).detach().numpy() * np.array([0,0,1,1,1]), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('Noise', np.sum(noise_idx)))

            # plt.hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Noise')

            Score = np.sum(nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_separated[datatype]), torch.FloatTensor(dataset_wsl_separated_Pcorr[datatype]))).detach().numpy(), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold1 = Score[int(0.1 * len(Score))]
            FPR1 = np.sum(Score_bg > threshold1) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'Include the pearson', color = colors[1])
            
            
            Score_bg = np.sum(nn.Sigmoid()(model_2(torch.FloatTensor(dataset_wsl_separated['bg']))).detach().numpy(), axis = -1)
            axs[col, row].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark_w/o_Pearson', color = colors[3])
            print('Totally {} events in benchmark set.'.format(len(dataset_wsl_separated['bg'])))
            
            
            Score = np.sum(nn.Sigmoid()(model_2(torch.FloatTensor(dataset_wsl_separated[datatype]))).detach().numpy(), axis = -1)
            # print('For datatype = {}, totally {} events. '.format('signal', np.sum(signal_idx)))
            
            Score.sort()
            threshold2 = Score[int(0.1 * len(Score))]
            FPR2 = np.sum(Score_bg > threshold2) / len(Score_bg)
            
            threshold_list[datatype] = threshold
            FPR_list[datatype] = FPR
            
            print("For datatype {}, the threshold is {} and the FPR is {}.".format(datatype, threshold, FPR))

            axs[col, row].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'No pearson', color = colors[2])
            
            
            axs[col, row].axvline(threshold1, color = colors[1], linestyle = '--')
            axs[col, row].axvline(threshold2, color = colors[2], linestyle = '--')
            axs[col, row].set_ylim(0,3.3)
            
            # threshold_chd = np.around(threshold, 3)
            
            axs[col, row].set_title('{}, FPR_w/_pearson = {:.3f}, FPR_w/o_pearson = {:.3f}'.format(datatype, FPR1, FPR2))
            # plt.title('Score distribution for blind dataset {}.'.format(idx))
            # plt.axvline(threshold)
            col+=1
    row += 1
    
plt.legend()
plt.show()

In [214]:
dataset_wsl_fft_all_test.shape

In [680]:
dataset_wsl_fft_all_test[:5000] - dataset_wsl_separated[0]

In [689]:
BKG_class_score = nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_test[:70000]), torch.FloatTensor(dataset_wsl_fft_all_test_Pcorr[:70000]))).detach().numpy().flatten()
    
SIG_class_score = nn.Sigmoid()(model(torch.FloatTensor(dataset_wsl_fft_all_test[70000:]), torch.FloatTensor(dataset_wsl_fft_all_test_Pcorr[70000:]))).detach().numpy().flatten()

SIG_class_score.sort()

threshold_1 = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(threshold_1)

FPR

In [690]:
BKG_class_score = nn.Sigmoid()(model_2(torch.FloatTensor(dataset_wsl_fft_all_test[:70000]))).detach().numpy().flatten()
    
SIG_class_score = nn.Sigmoid()(model_2(torch.FloatTensor(dataset_wsl_fft_all_test[70000:]))).detach().numpy().flatten()

SIG_class_score.sort()

threshold_2 = SIG_class_score[int(0.1 * len(SIG_class_score))]

print(threshold_2)

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)
FPR

In [660]:
type(threshold)

In [657]:
plt.title(np.around(threshold, 3))

### Study whether our model do solve the problem

In [584]:
dataset_fft.keys()

In [588]:
fig, axes = plt.subplots(1,3,figsize = (6.4 * 3, 4.8 * 1))

# model_number = 1614

# models = torch.load(ModelDir + '/SeriesWSC_{}_ratio_scan_FFTTTv13.json'.format(model_number),map_location=torch.device('cpu'))
# model = models['last_wsc']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

i = 0

for key in ['BBH', 'SGHF', 'SGLF']:
    Score_bg = np.sum(nn.Sigmoid()(model(torch.FloatTensor(dataset_fft['noise'].reshape(-1,202)), torch.FloatTensor(dataset_Pcorr['noise']))).detach().numpy(), axis = -1)
    axes[i].hist(Score_bg, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = 'bg_benchmark', color = colors[0])
    j = 1
    for suffix in ['highsnrraw', 'uncorrelated', 'halfsig']:
        Score = np.sum(nn.Sigmoid()(model(torch.FloatTensor(dataset_fft[key + suffix].reshape(-1,202)), torch.FloatTensor(dataset_Pcorr[key + suffix]))).detach().numpy(), axis = -1)
        axes[i].hist(Score, range = (-0.1,1.1), bins = 50, histtype='step', density = True, label = suffix, color = colors[j])
        j += 1
    axes[i].set_title(key)
    
    i+=1
plt.legend()

In [589]:
dataset_Pcorr['BBHhighsnrraw']

In [590]:
dataset_Pcorr['BBHuncorrelated']

In [591]:
dataset_Pcorr['BBHhalfsig']

In [595]:
plt.hist(dataset_Pcorr['BBHhighsnrraw'], bins = 50, histtype='step', label = 'raw')
plt.hist(dataset_Pcorr['BBHuncorrelated'], bins = 50, histtype='step', label = 'uncorrelated')
plt.hist(dataset_Pcorr['BBHhalfsig'], bins = 50, histtype='step', label = 'halfsig')
plt.legend()

## One question. Can DNN study the information from the Pearson correlation? Let's do a test, scanning over multiple hyperparameter sets to make a solid conclusion

### Define the structure and the function for training the model. We are trying to use all the DNN structure we have right now for learning

In [6]:
class WSC_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()


        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        return x

In [7]:
class WSC_1det_struct_upd_2(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct_upd_2, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.sm = nn.Softmax(dim=1)
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()
        self.last_linear = nn.Linear(self.encoder_struct[-1], 1)
        # self.sig = nn.Sigmoid()

        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        # print(self.sm(x))
        x = self.last_linear(self.sm(x))

        return x;

In [87]:
def trainWSC_Pearson(waveform, Pearson, struct, save_path):
# dataset0: bkg set from AE
# dataset1: identified signal from AE

    epochs = 100
    
    least_epoch = epochs//5
    interval = (epochs * 0.8) // 10

    wsc_list = {}

    wsc = WSC_1det_struct(struct).to(device)
    # trainable_params_WSC = sum(p.numel() for p in wsc.parameters() if p.requires_grad)
    # Nsize = 20

    # print('{}, bkg events and {} signal events passed to WSC for training. '.format(len(dataset0), len(dataset1)))

    nTotal0 = len(waveform)
    nTrain0 = int(rTrain * nTotal0)
    nTest0  = int(rTest * nTotal0)

    X_train = waveform[:nTrain0]
    X_test = waveform[-nTest0:] 
    X_validation = waveform[nTrain0:-nTest0]
    
    Y_train = Pearson[:nTrain0]
    Y_test = Pearson[-nTest0:]
    Y_validation = Pearson[nTrain0:-nTest0]

    train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.FloatTensor(Y_train).to(device))
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).to(device), torch.FloatTensor(Y_validation).to(device))

    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size, shuffle = True, drop_last=True)

    # wsc = WSClassifier_3class().to(device)
    optimizer = optim.Adam(wsc.parameters(), lr=0.00005)
    loss_func = nn.MSELoss().to(device)
    
    loss_train = np.empty(epochs)
    loss_validation = np.empty(epochs)

    for epoch in range(epochs):
        t0 = time.time()
        wsc.train()
        for batchidx, (x, y) in enumerate(trainDataLoader):
            yprime = nn.Sigmoid()(wsc(x)).flatten()
            loss = loss_func(yprime, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, (x, y) in enumerate(validationDataLoader):
                yprime = nn.Sigmoid()(wsc(x)).flatten()
                lossVal = loss_func(yprime, y)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)

        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
        
        if (epoch+1) >= least_epoch and (epoch+1-least_epoch) % interval == 0:
            wsc_list[(epoch+1)] = copy.deepcopy(wsc).cpu()
            wsc_list[str(epoch+1) + '_valloss'] = val_loss 
        
        print('Current epoch {}. Training time {}'.format(epoch+1, time.time() - t0))
        
    wsc.cpu().eval()
    
    _, ax = plt.subplots(1, 3, figsize=(21, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    err_train = np.mean((nn.Sigmoid()(wsc(torch.FloatTensor(X_train))).detach().numpy().reshape(-1,1) - Y_train.reshape(-1,1))**2, axis = -1).flatten()
    err_test = np.mean((nn.Sigmoid()(wsc(torch.FloatTensor(X_test ))).detach().numpy().reshape(-1,1) - Y_test.reshape(-1,1))**2, axis = -1).flatten()
    foo = ax[1].hist(err_train, range=(0.8 * min(err_train), 1.2 * max(err_train)), bins=20, density=True, histtype="step")
    foo = ax[1].hist(err_test, range=(0.8 * min(err_test), 1.2 * max(err_test)), bins=20, density=True, histtype="step")
    ax[2].hist(Y_test, range=(0, 1), bins=20, density=True, histtype="step")
    ax[2].hist(nn.Sigmoid()(wsc(torch.FloatTensor(X_test ))).detach().numpy().flatten(), range=(0, 1), bins=20, density=True, histtype="step")
    
    
    
    # plt.savefig(save_path)
    plt.title(struct)
    plt.savefig(save_path)
    plt.close()
    # logger.info("training figure saved to "+save_path)
    
    return wsc_list

In [140]:
def return_model_with_least_valloss(model_list):
    # valloss_list = np.empty((0))
    # epoch_list = np.empty((0))
    # extracting the epochs the model is using
    # print(model_list.keys())
    epoch_list = [int(re.search(r'(\d+)_valloss', item).group(1)) for item in model_list.keys() if isinstance(item, str) and '_valloss' in item]
    valloss_list = [model_list[key] for key in [item for item in model_list.keys() if isinstance(item, str) and 'valloss' in item]]
    # print(epoch_list[valloss_list.index(min(valloss_list))])
    return(model_list[epoch_list[valloss_list.index(min(valloss_list))]])

In [122]:
wsc_list['[400, 64, 16, 1]'].keys()

In [126]:
[int(re.search(r'(\d+)_valloss', item).group(1)) for item in wsc_list['[400, 64, 16, 1]'].keys() if isinstance(item, str) and '_valloss' in item]

In [128]:
keys = [item for item in wsc_list['[400, 64, 16, 1]'].keys() if isinstance(item, str) and 'valloss' in item]

In [130]:
keys

In [134]:
min([wsc_list['[400, 64, 16, 1]'][key] for key in keys])

In [139]:
return_model_with_least_valloss(wsc_list['[400, 64, 16, 1]'])

### Preparing the dataset

#### Training set

In [9]:
def Normalize_FFT_and_Pearson(dataset):
    
    assert dataset.shape[1:] == (2,200)
    
    P_Corr = Compute_P_corr(dataset).reshape(-1,1)
    
    Normalize = dataset.copy()
    Normalize = Normalize.reshape(-1,200)
    Normalize = Normalize / np.linalg.norm(Normalize, axis = -1).reshape(-1,1)
    Normalize = Normalize.reshape(-1,2,200)
    
    # FFT = Normalize.copy()
    FFT = np.abs(np.fft.rfft(Normalize, axis = -1))
    FFT = FFT / np.linalg.norm(FFT, axis = -1).reshape(-1,2,1)
    
    return Normalize, FFT, P_Corr

In [11]:
def Compute_P_corr(waveform):
    
    delta_max = 40
    
    assert waveform.shape[1:] == (2,200)
    
    num_length = len(waveform)
        
    P_corr_list = np.empty((num_length, 2 * delta_max + 1))
    
    for delta in range(-delta_max, delta_max+1):
    
        H = waveform[:,1,:].copy()[:,max(0, delta):min(200, 200+delta)]
        L = waveform[:,0,:].copy()[:,max(0, -delta):min(200, 200-delta)]
        
        H_mean = np.mean(H, axis = -1).reshape(-1,1)
        L_mean = np.mean(L, axis = -1).reshape(-1,1)
        
        H_linear = H - H_mean
        L_linear = L - L_mean
        
        H_squared = np.square(H_linear)
        L_squared = np.square(L_linear)
    
        nominator = (-1) * np.sum(L_linear * H_linear, axis = 1)
        denominator = np.sqrt((np.sum(L_squared,axis = 1)) * (np.sum(H_squared, axis = 1)))
        
        cache = np.abs(nominator / denominator)
        
        P_corr_list[:,delta + delta_max] = cache.copy()
    
    P_corr = np.max(P_corr_list, axis = 1)
    
    return P_corr

In [12]:
N_for_every_dataset = 30000

dataset_raw = {}
dataset_fft = {}
dataset_Pcorr = {}

noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:400000]
noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:400000]

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

np.random.shuffle(glitch_raw_L)
np.random.shuffle(glitch_raw_H)

glitch_set = np.concatenate((
    np.concatenate((noise_raw_L[N_for_every_dataset:N_for_every_dataset + N_for_every_dataset//2].reshape(-1,1,200), glitch_raw_H[:N_for_every_dataset//2].reshape(-1,1,200)), axis = 1), 
    np.concatenate((glitch_raw_L[:N_for_every_dataset//2].reshape(-1,1,200), noise_raw_H[N_for_every_dataset:N_for_every_dataset + N_for_every_dataset//2].reshape(-1,1,200)), axis = 1)
)
                            , axis = 0)

# dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
# dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
# dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

# dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
# dataset_fft['noise'] = 
# dataset_fft['noise'] = dataset_fft['noise'] / 

dataset_raw['noise'], dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:N_for_every_dataset].reshape(-1,1,200), noise_raw_H[:N_for_every_dataset].reshape(-1,1,200)), axis = 1))

dataset_raw['glitch'], dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H


for datatype in ['BBH', 'SGHF', 'SGLF']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [:N_for_every_dataset//4], axis=0)
    
    dataset_raw[datatype], dataset_fft[datatype], dataset_Pcorr[datatype] = Normalize_FFT_and_Pearson(cache)
    
    # dataset_raw[datatype] = cache.copy()    
    
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
        
# for datatype in ['SGLF']:
#     cache = np.empty((0,2,200))
#     for snr in ['5-12','12-24','24-48','48-96']:
#         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
#                           [:2500])
        
#     dataset_raw[datatype] = cache.copy()    
    
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
#     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)



In [14]:
for key in dataset_fft.keys():
    print(key)
    print(dataset_raw[key].shape)
    print(np.sum(np.linalg.norm(dataset_fft[key].reshape(-1,202), axis = -1)))
    print(dataset_Pcorr[key].mean())

In [20]:
dataset_for_training_rawwaveform = np.concatenate([dataset_raw[key].reshape(-1,400) for key in dataset_raw.keys()], axis = 0)
dataset_for_training_Pearson = np.concatenate([dataset_Pcorr[key] for key in dataset_Pcorr.keys()], axis = 0)
dataset_for_training_fft = np.concatenate([dataset_fft[key].reshape(-1,202) for key in dataset_fft.keys()], axis = 0)

In [22]:
perm = np.random.choice(len(dataset_for_training_rawwaveform), len(dataset_for_training_rawwaveform), replace=False)
dataset_for_training_rawwaveform = dataset_for_training_rawwaveform[perm]
dataset_for_training_Pearson = dataset_for_training_Pearson[perm]
dataset_for_training_fft = dataset_for_training_fft[perm]

#### Testing set

In [148]:
# N_for_every_dataset = 30000

# dataset_raw = {}
# dataset_fft = {}
# dataset_Pcorr = {}

# noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:400000]
# noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:400000]

# np.random.shuffle(noise_raw_L)
# np.random.shuffle(noise_raw_H)

# glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
# glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

# np.random.shuffle(glitch_raw_L)
# np.random.shuffle(glitch_raw_H)

# glitch_set = np.concatenate((
#     np.concatenate((noise_raw_L[N_for_every_dataset:N_for_every_dataset + N_for_every_dataset//2].reshape(-1,1,200), glitch_raw_H[:N_for_every_dataset//2].reshape(-1,1,200)), axis = 1), 
#     np.concatenate((glitch_raw_L[:N_for_every_dataset//2].reshape(-1,1,200), noise_raw_H[N_for_every_dataset:N_for_every_dataset + N_for_every_dataset//2].reshape(-1,1,200)), axis = 1)
# )
#                             , axis = 0)

# dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
# dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
# dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

# dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
# dataset_fft['noise'] = 
# dataset_fft['noise'] = dataset_fft['noise'] / 

# dataset_raw['noise'], dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:N_for_every_dataset].reshape(-1,1,200), noise_raw_H[:N_for_every_dataset].reshape(-1,1,200)), axis = 1))

# dataset_raw['glitch'], dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

# del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H


for datatype in ['BBH']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [50000:], axis=0)
    
    dataset_raw_BBH_test, dataset_fft_BBH_test, dataset_Pcorr_BBH_test = Normalize_FFT_and_Pearson(cache)
    
    # dataset_raw[datatype] = cache.copy()    
    
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
        
# for datatype in ['SGLF']:
#     cache = np.empty((0,2,200))
#     for snr in ['5-12','12-24','24-48','48-96']:
#         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
#                           [:2500])
        
#     dataset_raw[datatype] = cache.copy()    
    
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
#     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)



In [149]:
dataset_raw_BBH_test.shape

In [150]:
dataset_fft_BBH_test.shape

In [151]:
# perm = np.random.choice(len(dataset_raw_BBH_test), len(dataset_raw_BBH_test), replace=False)
dataset_for_training_rawwaveform_BBH_test = dataset_raw_BBH_test.reshape(-1,400)
dataset_for_training_Pearson_BBH_test = dataset_Pcorr_BBH_test.reshape(-1,1)
dataset_for_training_fft_BBH_test = dataset_fft_BBH_test.reshape(-1,202)

### Training the models

#### For a hybrid mode

In [36]:
Trial_model = trainWSC_Pearson(dataset_for_training_rawwaveform, dataset_for_training_Pearson.flatten(), [400,64,16,1])

In [72]:
wsc_list = {}

for struct in [[400,64,16,1], [400,64,16,5,1], [400,128,32,5,1], [400,128,32,16,5,1]]:
    wsc_list[str(struct)] = trainWSC_Pearson(dataset_for_training_rawwaveform, dataset_for_training_Pearson.flatten(), struct)

In [75]:
wsc_list

In [51]:
test = np.array([1.,2.,3.,4.,5.], dtype=np.float64)

In [52]:
min(test)

In [55]:
test_target = np.array([5,4,3,2,1])

err_train = np.var(test.reshape(-1,1) - test_target.reshape(-1,1), axis = -1).flatten()
err_test = np.var(test.reshape(-1,1) - test_target.reshape(-1,1), axis = -1).flatten()
foo = plt.hist(err_train, range=(0.8 * min(err_train), 1.2 * max(err_train)), bins=20, density=True, histtype="step")
foo = plt.hist(err_test, range=(0.8 * min(err_test), 1.2 * max(err_test)), bins=20, density=True, histtype="step")

In [57]:
test

In [58]:
test_target

In [59]:
np.var(test.reshape(-1,1) - test_target.reshape(-1,1), axis = -1).flatten()

In [68]:
test.reshape(-1,5) - test_target.reshape(-1,5)

In [69]:
np.mean((test.reshape(-1,5) - test_target.reshape(-1,5))**2, axis = -1)

##### Now let's test will this give us any idea about

In [146]:
plt.hist(dataset_for_training_Pearson_BBH_test, range=(0,1), bins = 50, density=True, histtype='step')
for key in wsc_list.keys():
    wsc = return_model_with_least_valloss(wsc_list[key])
    plt.hist(nn.Sigmoid()(wsc(torch.FloatTensor(dataset_for_training_rawwaveform_BBH_test))).detach().numpy().flatten(), range=(0, 1), bins=50, density=True, histtype="step", label = key)
plt.legend()

plt.title('Pearson reconstruction plot for DNN models')

#### For just BBH mode

In [77]:
# N_for_every_dataset = 30000

# dataset_raw = {}
# dataset_fft = {}
# dataset_Pcorr = {}

# noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:400000]
# noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:400000]

# np.random.shuffle(noise_raw_L)
# np.random.shuffle(noise_raw_H)

# glitch_raw_L = np.load('../Data_cached/real_glitches_H_snrlt5_59732_4000Hz_25ms.npz')['strain_time_data']
# glitch_raw_H = np.load('../Data_cached/real_glitches_snrlt5_60132_4000Hz_25ms.npz')['strain_time_data']

# np.random.shuffle(glitch_raw_L)
# np.random.shuffle(glitch_raw_H)

# glitch_set = np.concatenate((
#     np.concatenate((noise_raw_L[N_for_every_dataset:N_for_every_dataset + N_for_every_dataset//2].reshape(-1,1,200), glitch_raw_H[:N_for_every_dataset//2].reshape(-1,1,200)), axis = 1), 
#     np.concatenate((glitch_raw_L[:N_for_every_dataset//2].reshape(-1,1,200), noise_raw_H[N_for_every_dataset:N_for_every_dataset + N_for_every_dataset//2].reshape(-1,1,200)), axis = 1)
# )
#                             , axis = 0)

# dataset_raw['noise'] = np.concatenate((noise_raw_L[:65000].reshape(-1,1,200), noise_raw_H[:65000].reshape(-1,1,200)), axis = 1)
# dataset_Pcorr['noise'] = Compute_P_corr(dataset_raw['noise'])
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,200)
# dataset_raw['noise'] = dataset_raw['noise'] / np.linalg.norm(dataset_raw['noise'], axis=-1).reshape(-1,1)
# dataset_raw['noise'] = dataset_raw['noise'].reshape(-1,2,200)

# dataset_fft['noise'] = np.abs(np.fft.rfft(dataset_raw['noise'], axis = -1))
# dataset_fft['noise'] = 
# dataset_fft['noise'] = dataset_fft['noise'] / 

# dataset_raw['noise'], dataset_fft['noise'], dataset_Pcorr['noise'] = Normalize_FFT_and_Pearson(np.concatenate((noise_raw_L[:N_for_every_dataset].reshape(-1,1,200), noise_raw_H[:N_for_every_dataset].reshape(-1,1,200)), axis = 1))

# dataset_raw['glitch'], dataset_fft['glitch'], dataset_Pcorr['glitch'] = Normalize_FFT_and_Pearson(glitch_set)

# del noise_raw_L, noise_raw_H, glitch_raw_L, glitch_raw_H


for datatype in ['BBH']:
    cache = np.empty((0,2,200))
    for snr in ['5-12','12-24','24-48','48-96']:
        cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
                          [:40000], axis=0)
    
    dataset_raw_BBH, dataset_fft_BBH, dataset_Pcorr_BBH = Normalize_FFT_and_Pearson(cache)
    
    # dataset_raw[datatype] = cache.copy()    
    
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
    # dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
    # dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)
        
# for datatype in ['SGLF']:
#     cache = np.empty((0,2,200))
#     for snr in ['5-12','12-24','24-48','48-96']:
#         cache = np.append(cache, np.load('../Data_cached/injected_' + datatype + '_55k_snr' + snr + '_0th_events_before_merger_time_windowlength_200.npz')['strain']
#                           [:2500])
        
#     dataset_raw[datatype] = cache.copy()    
    
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,200)
#     dataset_raw[datatype] = dataset_raw[datatype] / np.linalg.norm(dataset_raw[datatype], axis=-1).reshape(-1,1)
#     dataset_raw[datatype] = dataset_raw[datatype].reshape(-1,2,200)



In [78]:
dataset_raw_BBH.shape

In [109]:
dataset_fft_BBH.shape

In [84]:
perm = np.random.choice(len(dataset_raw_BBH), len(dataset_raw_BBH), replace=False)
dataset_for_training_rawwaveform = dataset_raw_BBH[perm].reshape(-1,400)
dataset_for_training_Pearson = dataset_Pcorr_BBH[perm].reshape(-1,1)
dataset_for_training_fft = dataset_fft_BBH[perm].reshape(-1,202)

In [85]:
dataset_for_training_rawwaveform.shape

In [111]:
wsc_list_BBH = {}

for struct in [[400,64,16,1], [400,64,16,5,1], [400,128,32,5,1], [400,128,32,16,5,1]]:
    wsc_list_BBH[str(struct)] = trainWSC_Pearson(dataset_for_training_rawwaveform, dataset_for_training_Pearson.flatten(), struct, 'SC_pearson_' + str(struct) + '_BBH.png')

##### Now let's test will this give us any idea about DNN learning the Pearson

In [119]:
wsc_list_BBH

In [145]:
plt.hist(dataset_for_training_Pearson_BBH_test, range=(0,1), bins = 50, density=True, histtype='step')
for key in wsc_list_BBH.keys():
    wsc = return_model_with_least_valloss(wsc_list_BBH[key])
    plt.hist(nn.Sigmoid()(wsc(torch.FloatTensor(dataset_for_training_rawwaveform_BBH_test))).detach().numpy().flatten(), range=(0, 1), bins=50, density=True, histtype="step", label = key)
plt.legend()

plt.title('Pearson reconstruction plot for DNN models')

In [152]:
plt.hist(dataset_for_training_Pearson_BBH_test[-5000:], range=(0,1), bins = 50, density=True, histtype='step')
for key in wsc_list_BBH.keys():
    wsc = return_model_with_least_valloss(wsc_list_BBH[key])
    plt.hist(nn.Sigmoid()(wsc(torch.FloatTensor(dataset_for_training_rawwaveform_BBH_test[-5000:]))).detach().numpy().flatten(), range=(0, 1), bins=50, density=True, histtype="step", label = key)
plt.legend()

plt.title('Pearson reconstruction plot for DNN models, high snr BBH')

In [153]:
plt.hist(dataset_for_training_Pearson_BBH_test[:5000], range=(0,1), bins = 50, density=True, histtype='step')
for key in wsc_list_BBH.keys():
    wsc = return_model_with_least_valloss(wsc_list_BBH[key])
    plt.hist(nn.Sigmoid()(wsc(torch.FloatTensor(dataset_for_training_rawwaveform_BBH_test[:5000]))).detach().numpy().flatten(), range=(0, 1), bins=50, density=True, histtype="step", label = key)
plt.legend()

plt.title('Pearson reconstruction plot for DNN models, low snr BBH')

In [154]:
os.getcwd()

In [157]:
np.load('../Data_cached/pure_BBH_snr48-96_0th_events_before_merger_time.npz')['strain'].shape

# Study the WSC overtraining problem

## Defining the sturucture and the training process

In [60]:
class WSC_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()


        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        return x

In [61]:
def trainSeriesSupC_struct_withunknownloss(datasets, struct, save_path):
# datasets: multiple datasets, datasets[0, 1, ...] are for the 1st, 2nd, ... class
# datasets should have the keys to be the integres 0, 1, 2, ...

    epochs_wsc = 500
    batch_size_wsc = 384
    lr_wsc = 2e-5
    
    least_epochs = 100
    epochs_interval = 20
    model_list = {}
    
    wsc = WSC_1det_struct(struct).to(device)
    # nparam = sum(p.numel() for p in wsc.parameters() if p.requires_grad)
    
    Nclass = len(datasets)
    wclass = torch.FloatTensor([len(datasets[0])/len(datasets[i]) for i in range(Nclass)]).to(device)
    nTotal = {}
    nTrain = {}
    nTest = {}
    nbkg_train = 0
    nsig_train = 0
    for i in np.arange(Nclass):
        nTotal[i] = datasets[i].shape[0]
        nTrain[i] = int(rTrain*nTotal[i])
        nTest[i] = int(rTest*nTotal[i])
        if i < 2:
            nbkg_train += nTrain[i]
        else:
            nsig_train += nTrain[i]

    # batch_cut = nTrain[-1] // batch_size_wsc

    X_train = np.concatenate([datasets[i][:nTrain[i]] for i in range(Nclass)])
    X_test = np.concatenate([datasets[i][-nTest[i]:] for i in range(Nclass)])
    X_validation = np.concatenate([datasets[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    Y_train = np.concatenate([i*np.ones(nTrain[i], dtype=int) for i in np.arange(Nclass)])
    Y_validation = np.concatenate([i*np.ones(nTotal[i]-nTrain[i]-nTest[i], dtype=int) for i in np.arange(Nclass)])
    
    total_batch_num = len(Y_validation) // batch_size_wsc
    batch_cut = (nTotal[Nclass-1]-nTrain[Nclass-1]-nTest[Nclass-1]) // batch_size_wsc

    train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(Y_train).to(device))
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).to(device), torch.LongTensor(Y_validation).to(device))
    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)

    # associated with a direct sum of probability, see below
    optimizer = optim.Adam(wsc.parameters(), lr=lr_wsc)
    loss_func = nn.CrossEntropyLoss(weight=wclass).to(device)
    loss_train = np.empty(epochs_wsc)
    loss_validation = np.empty(epochs_wsc)
    loss_validation_unknown = np.empty(epochs_wsc)
    loss_validation_known = np.empty(epochs_wsc)

    for epoch in range(epochs_wsc):
        wsc.train()
        for batchidx, (x, y) in enumerate(trainDataLoader):
            yprime = wsc(x)
            loss = loss_func(yprime, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            val_loss_unknown = 0
            val_loss_known = 0
            for batchidx, (x, y) in enumerate(validationDataLoader):
                yprime = wsc(x)
                lossVal = loss_func(yprime, y)
                val_loss += lossVal.item()
                if batchidx > total_batch_num - batch_cut:
                    val_loss_unknown += lossVal.item()
                else:
                    val_loss_known += lossVal.item()

            val_loss /= len(validationDataLoader)
            val_loss_unknown /= batch_cut
            val_loss_known /= total_batch_num - batch_cut
            
        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
        loss_validation_unknown[epoch] = val_loss_unknown
        loss_validation_known[epoch] = val_loss_known
        
        
        if ((epoch+1) > least_epochs) and ((epoch+1) % epochs_interval == 0):
            model_list[(epoch+1)] = copy.deepcopy(wsc.cpu().eval())
            model_list[str(epoch+1)+'valloss'] = val_loss
            wsc.to(device)
        
    wsc.to(device).eval()
    
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    ax[0].plot(loss_validation_unknown)
    ax[0].plot(loss_validation_known)
    foo = ax[1].hist(nn.Softmax(dim=1)(wsc(torch.FloatTensor(X_train).to(device))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")
    foo = ax[1].hist(nn.Softmax(dim=1)(wsc(torch.FloatTensor(X_test ).to(device))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")

    plt.show()
    plt.savefig(save_path)
    plt.close()
    
    return model_list

## Loading the training set, where the training set is 80k and the unknown set is 30k. We also have supervised learning case, where the SGLF has the same shape as the unknown set.

In [7]:
os.getcwd()

In [ ]:
with open('./Sida_temp/for_K8S_training/Data/wsc_training_set_largest_extended_shuffled.pickle', 'rb') as handle:
    wsc_training = pickle.load(handle)

In [ ]:
wsc_training.keys()

In [ ]:
for i in range(5):
    print(wsc_training[i].shape)

In [ ]:
wsc_training_pure_SGLF = wsc_training[4].copy()[:29993]

In [ ]:
wsc_training_mixed_SGLF = np.load('../K8S_training/LastWSCtest/Data/wsc_test_set_new_LM_same_ratio_30k_shuffled_1.npy')

In [ ]:
wsc_training_mixed_SGLF.shape

In [70]:
wsc_training

In [42]:
wsc_training[4] = wsc_training_pure_SGLF.copy()
Full_supervised_model = trainSeriesSupC_struct_withunknownloss(wsc_training, [202,64,16,5], 'full_supervised_loss_curve.png')

In [ ]:
wsc_training[4] = wsc_training_mixed_SGLF.copy()
Weakly_supervised_model = trainSeriesSupC_struct_withunknownloss(wsc_training, [202,64,16,5], 'weakly_supervised_loss_curve.png')

# Using the GWAK to train noise DNN and see difference

## Model definition and training function

In [5]:
class AE_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(AE_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.encoder_layers = nn.ModuleList()
        self.norm_encoder_layers = nn.ModuleList()
        self.decoder_layers = nn.ModuleList()
        self.norm_decoder_layers = nn.ModuleList()

        # print(self.encoder_struct.devide)
        

        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.encoder_layers.append(layer)
            self.norm_encoder_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

            layer = nn.Linear(self.encoder_struct[self.dep-1-i], self.encoder_struct[self.dep-2-i])
            nn.init.kaiming_normal_(layer.weight)
            self.decoder_layers.append(layer)
            self.norm_decoder_layers.append(nn.BatchNorm1d(self.encoder_struct[self.dep-2-i]))

    def forward(self, x):
        for i, layer in enumerate(self.encoder_layers):
            x = layer(x)
            x = self.relu(x)
            # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
            x = self.norm_encoder_layers[i](x)
            
        encoded = x;

        for i, layer in enumerate(self.decoder_layers):
            x = layer(x)
            if(i < self.dep-2):
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[self.dep-2-i])(x)
                x = self.norm_decoder_layers[i](x)

        decoded = nn.Sigmoid()(x)

        return encoded, decoded

In [6]:
def trainAE_struct(dataset, struct, save_path):
    
    nTotal = len(dataset);
    nTrain = int(rTrain * nTotal)
    nTest = int(rTest * nTotal)
    print("{} events into AE training. ".format(nTotal))

    X_train = dataset[:nTrain]
    X_test = dataset[-nTest:]
    X_validation = dataset[nTrain:-nTest]

    trainData = torch.FloatTensor(X_train)
    testData = torch.FloatTensor(X_test)
    validationData = torch.FloatTensor(X_validation)

    train_dataset = TensorDataset(trainData)
    test_dataset = TensorDataset(testData)
    validation_dataset = TensorDataset(validationData)

    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    autoencoder = AE_1det_struct(struct).to(device)
    optimizer = optim.Adam(autoencoder.parameters(), lr=0.00005)
    loss_func = nn.MSELoss().to(device)
    
    loss_train = np.empty(epochs)
    loss_validation = np.empty(epochs)

    for epoch in range(epochs):

        autoencoder.train()
        for batchidx, x in enumerate(trainDataLoader):
            x = x[0].to(device)
            encoded, decoded = autoencoder(x)
            loss_overall = loss_func(decoded, x)
            weighted_lossTrain = loss_overall

            optimizer.zero_grad()
            weighted_lossTrain.backward()
            optimizer.step()
            
        autoencoder.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, x in enumerate(validationDataLoader):
                x = x[0].to(device)
                encoded, decoded = autoencoder(x)
                lossVal = loss_func(decoded, x)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)

        loss_train[epoch] = weighted_lossTrain.item()
        loss_validation[epoch] = val_loss
    
    autoencoder.to(device).eval()
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    
    dcd_train = autoencoder(torch.FloatTensor(X_train).to(device))[1].cpu().detach().numpy()
    err_train = np.mean((X_train-dcd_train)**2, axis=1)
    dcd_test = autoencoder(torch.FloatTensor(X_test).to(device))[1].cpu().detach().numpy()
    err_test = np.mean((X_test-dcd_test)**2, axis=1)
    foo = ax[1].hist(err_train, range=(0, max(err_train)), bins=50, density=True, histtype="step")
    foo = ax[1].hist(err_test, range=(0, max(err_train)), bins=50, density=True, histtype="step")

    plt.savefig(save_path)
    plt.close()
    # logger.info("training figure saved to "+save_path+". ")
            
    return autoencoder.cpu().eval()

## Pre-trained models

In [ ]:
# modelDir = 

In [7]:
foo = torch.load("./Sida_temp/for_K8S_training/Model/noise_AE_freq_new.json")
model = foo["2det_202-40-20"].cpu().eval()

In [8]:
model

## Loading the 3 noise sets

In [7]:
noise_GWAK = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data']
noise_GWAK.shape

In [8]:
noise_GWAK = noise_GWAK / np.linalg.norm(noise_GWAK, axis = -1).reshape(-1,2,1)

noise_GWAK_fft = np.abs(np.fft.rfft(noise_GWAK, axis = -1))
noise_GWAK_fft = noise_GWAK_fft / np.linalg.norm(noise_GWAK_fft, axis = -1).reshape(-1,2,1)

noise_GWAK_fft = noise_GWAK_fft.reshape(-1,202)

In [13]:
noise_GWAK_fft.shape

In [14]:
noise_GWAK_fft.shape

In [13]:
noise_GWAK_fft = noise_GWAK_fft.reshape(-1,2,101)

In [59]:
noise_GWAK_fft[0]

In [60]:
noise_GWAK_fft = noise_GWAK_fft[:,[1,0],:]

In [61]:
noise_GWAK_fft[0]

In [62]:
noise_GWAK_fft = noise_GWAK_fft.reshape(-1,202)

In [192]:
noise_raw_L = np.load('../Data_cached/real_bkg_2202000_63917s_4000Hz_50ms.npy')[:2000000]
noise_raw_H = np.load('../Data_cached/real_bkg_H_1466640_58803s_4000Hz_50ms.npy')[:1400000]

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

noise_Longseg = np.concatenate((noise_raw_L[:100000].reshape(-1,1,200), noise_raw_H[:100000].reshape(-1,1,200)), axis = 1)

# print(noise_Longseg[0])
noise_Longseg = noise_Longseg - np.mean(noise_Longseg, axis = -1).reshape(-1,2,1)
# print(noise_Longseg[0])
del noise_raw_L, noise_raw_H

In [51]:
noise_Longseg.shape

In [193]:
noise_Longseg = noise_Longseg / np.linalg.norm(noise_Longseg, axis = -1).reshape(-1,2,1)

noise_Longseg_fft = np.abs(np.fft.rfft(noise_Longseg, axis = -1))
noise_Longseg_fft = noise_Longseg_fft / np.linalg.norm(noise_Longseg_fft, axis = -1).reshape(-1,2,1)

noise_Longseg_fft = noise_Longseg_fft.reshape(-1,202)

In [53]:
noise_Longseg_fft.shape

In [64]:
noise_Longseg_fft = noise_Longseg_fft.reshape(-1,2,101)
noise_Longseg_fft = noise_Longseg_fft[:,[1,0],:]
noise_Longseg_fft = noise_Longseg_fft.reshape(-1,202)

In [194]:
noise_raw_L = np.load('../Data_cached/real_bkg_L_60180_4271s_4000Hz_50ms.npy')
noise_raw_H = np.load('../Data_cached/real_bkg_H_60600_4271s_4000Hz_50ms.npy')

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

noise_Shortseg = np.concatenate((noise_raw_H[:60000].reshape(-1,1,200), noise_raw_L[:60000].reshape(-1,1,200)), axis = 1)

In [195]:
noise_Shortseg.shape

In [9]:
noise_Shortseg = noise_Shortseg / np.linalg.norm(noise_Shortseg, axis = -1).reshape(-1,2,1)

noise_Shortseg_fft = np.abs(np.fft.rfft(noise_Shortseg, axis = -1))
noise_Shortseg_fft = noise_Shortseg_fft / np.linalg.norm(noise_Shortseg_fft, axis = -1).reshape(-1,2,1)

noise_Shortseg_fft = noise_Shortseg_fft.reshape(-1,202)

In [66]:
noise_Shortseg_fft.shape

In [65]:
noise_Shortseg_fft = noise_Shortseg_fft.reshape(-1,2,101)
noise_Shortseg_fft = noise_Shortseg_fft[:,[1,0],:]
noise_Shortseg_fft = noise_Shortseg_fft.reshape(-1,202)

In [197]:
noise_raw_L = np.load('../Data_cached/real_bkg_L_61680_4271s_short_asd_4000Hz_50ms.npy')
noise_raw_H = np.load('../Data_cached/real_bkg_H_60660_4271s_short_asd_4000Hz_50ms.npy')

np.random.shuffle(noise_raw_L)
np.random.shuffle(noise_raw_H)

noise_Shortseg_shortasd = np.concatenate((noise_raw_H[:60000].reshape(-1,1,200), noise_raw_L[:60000].reshape(-1,1,200)), axis = 1)

In [198]:
noise_Shortseg_shortasd.shape

In [199]:
noise_Shortseg_shortasd = noise_Shortseg_shortasd / np.linalg.norm(noise_Shortseg_shortasd, axis = -1).reshape(-1,2,1)

noise_Shortseg_shortasd_fft = np.abs(np.fft.rfft(noise_Shortseg_shortasd, axis = -1))
noise_Shortseg_shortasd_fft = noise_Shortseg_shortasd_fft / np.linalg.norm(noise_Shortseg_shortasd_fft, axis = -1).reshape(-1,2,1)

noise_Shortseg_shortasd_fft = noise_Shortseg_shortasd_fft.reshape(-1,202)

In [200]:
noise_Shortseg_shortasd_fft.shape

In [12]:
noise_Shortseg_GWAK = np.load('../Data_cached/Noise_processing/Processed_noise_sets/noise_sets_v1.npy')

In [13]:
noise_Shortseg_GWAK.shape

In [14]:
noise_Shortseg_GWAK = noise_Shortseg_GWAK / np.linalg.norm(noise_Shortseg_GWAK, axis = -1).reshape(-1,2,1)

noise_Shortseg_GWAK_fft = np.abs(np.fft.rfft(noise_Shortseg_GWAK, axis = -1))
noise_Shortseg_GWAK_fft = noise_Shortseg_GWAK_fft / np.linalg.norm(noise_Shortseg_GWAK_fft, axis = -1).reshape(-1,2,1)

noise_Shortseg_GWAK_fft = noise_Shortseg_GWAK_fft.reshape(-1,202)

In [15]:
noise_Shortseg_GWAK_fft.shape

In [357]:
noise_Longseg_GWAK = np.load('../Data_cached/Noise_processing/Processed_noise_sets/noise_sets_v3_4.npy')

In [358]:
noise_Longseg_GWAK.shape

In [359]:
noise_Longseg_GWAK = noise_Longseg_GWAK / np.linalg.norm(noise_Longseg_GWAK, axis = -1).reshape(-1,2,1)

noise_Longseg_GWAK_fft = np.abs(np.fft.rfft(noise_Longseg_GWAK, axis = -1))
noise_Longseg_GWAK_fft = noise_Longseg_GWAK_fft / np.linalg.norm(noise_Longseg_GWAK_fft, axis = -1).reshape(-1,2,1)

noise_Longseg_GWAK_fft = noise_Longseg_GWAK_fft.reshape(-1,202)

In [360]:
noise_Longseg_GWAK_fft.shape

In [374]:
noise_Midseg_GWAK = np.load('../Data_cached/Noise_processing/Processed_noise_sets/noise_sets_v4_4.npy')

In [375]:
noise_Midseg_GWAK.shape

In [376]:
noise_Midseg_GWAK = noise_Midseg_GWAK / np.linalg.norm(noise_Midseg_GWAK, axis = -1).reshape(-1,2,1)

noise_Midseg_GWAK_fft = np.abs(np.fft.rfft(noise_Midseg_GWAK, axis = -1))
noise_Midseg_GWAK_fft = noise_Midseg_GWAK_fft / np.linalg.norm(noise_Midseg_GWAK_fft, axis = -1).reshape(-1,2,1)

noise_Midseg_GWAK_fft = noise_Midseg_GWAK_fft.reshape(-1,202)

In [377]:
noise_Midseg_GWAK_fft.shape

#### We somehow find that the GWAK noise is like a gaussian distribution. Try to make an gaussian noise of our own and do the training/testing

In [118]:
from scipy.stats import norm

In [72]:
noise_GWAK = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data']

100k versus 2M

In [120]:
dataset = noise_GWAK[:1000,0].flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [165]:
dataset = noise_GWAK.flatten()

plt.hist(dataset, bins = 50, range = (-0.4,0.4), density = True)

xmin, xmax = plt.xlim(-0.4,0.4)
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [161]:
dataset = noise_Longseg.flatten()

plt.hist(dataset, bins = 50, range = (-3,3), density = True)

xmin, xmax = plt.xlim(-3,3)
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [133]:
dataset = noise_Shortseg.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [168]:
dataset = noise_Shortseg.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [181]:
# shortseg, short asd, not normalized

dataset = noise_Shortseg_shortasd.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [175]:
# shortseg, short asd, normalized

dataset = noise_Shortseg_shortasd.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [252]:
# shortseg, GWAK processed, not normalized

dataset = noise_Shortseg_GWAK.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [250]:
# shortseg, GWAK processed, normalized

dataset = noise_Shortseg_GWAK.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [258]:
# shortseg 2, GWAK processed, normalized

dataset = noise_Shortseg_GWAK.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [268]:
# shortseg 2, GWAK processed, normalized

dataset = noise_Longseg_GWAK.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [300]:
# Longseg middle, normalized

dataset = noise_Longseg_GWAK.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [301]:
# Longseg middle 2, normalized

dataset = noise_Longseg_GWAK.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [125]:
dataset = noise_normal[:1000,0].flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [117]:
plt.hist(noise_GWAK[-100:,0].flatten(), bins = 50)

In [81]:
plt.hist(noise_GWAK[:1000,1].flatten(), bins = 50)

In [82]:
plt.hist(noise_GWAK[-1000:,1].flatten(), bins = 50)

In [87]:
std_Hanford = np.std(noise_GWAK[:,0].flatten())

In [88]:
np.mean(noise_GWAK[:,0].flatten())

In [ ]:
plt.hist(noise_GWAK[:1000,0].flatten(), bins = 50)

In [111]:
plt.hist(noise_Longseg[:1000,0].flatten(), bins = 50)

In [112]:
plt.hist(noise_Longseg[-1000:,0].flatten(), bins = 50)

In [114]:
plt.hist(noise_Shortseg[:100,0].flatten(), bins = 50)
plt.

In [115]:
plt.hist(noise_Shortseg[-100:,0].flatten(), bins = 50)

In [135]:
noise_normal = np.random.normal(0, 1, 200000*200).reshape(-1,2,200)

In [136]:
noise_normal = noise_normal / np.linalg.norm(noise_normal, axis = -1).reshape(-1,2,1)

noise_normal_fft = np.abs(np.fft.rfft(noise_normal, axis = -1))

noise_normal_fft[:,:,75:] = 0

noise_normal_fft = noise_normal_fft / np.linalg.norm(noise_normal_fft, axis = -1).reshape(-1,2,1)

noise_normal_fft = noise_normal_fft.reshape(-1,202)

In [143]:
trial = np.fft.rfft(noise_normal, axis = -1)

In [144]:
trial[:,:,75:] = 0

In [146]:
noise_bandpassed = np.fft.irfft(trial)

In [147]:
dataset = noise_bandpassed.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

In [137]:
plt.plot(noise_normal_fft[0])

In [138]:
plt.plot(noise_GWAK_fft[0])

### Making a gaussian and band-passed noise of our own

In [131]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# 设置参数
fs = 4096      # 采样频率
duration = 5   # 持续时间（秒）
N = int(fs * duration)  # 总样本数
f_low = 30      # 带通滤波器下限频率（Hz）
f_high = 1500    # 带通滤波器上限频率（Hz）

# 生成白噪声
white_noise = np.random.normal(0, 1, N)

# 带通滤波器设计
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

# 应用带通滤波器
def bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data)
    return y

# 生成带通噪声
bandpassed_noise = bandpass_filter(white_noise, f_low, f_high, fs)

# 生成高斯分布的振幅
amplitude = np.random.normal(loc=0, scale=1, size=N)
amplitude = np.abs(amplitude)  # 取绝对值确保振幅为正

# 振幅与带通噪声相乘
final_signal = amplitude * bandpassed_noise

# 绘制结果
plt.figure(figsize=(12, 6))

# 时域信号
plt.subplot(2, 1, 1)
plt.plot(np.linspace(0, duration, N), final_signal)
plt.title('Time Domain Signal with Gaussian Amplitude and Bandpass Noise')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')

# 频域信号
frequencies = np.fft.fftfreq(N, 1/fs)
spectrum = np.fft.fft(final_signal)
plt.subplot(2, 1, 2)
plt.plot(frequencies[:N//2], np.abs(spectrum)[:N//2])
plt.title('Frequency Domain Spectrum')
plt.xlim(0, fs/2)
plt.xlabel('Frequency [Hz]')
plt.ylabel('Magnitude')

plt.tight_layout()
plt.show()

In [132]:
dataset = final_signal.flatten()

plt.hist(dataset, bins = 50, density = True)

xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, np.mean(dataset), np.std(dataset))

plt.plot(x,p)

### Trying to find, is there overlapping between the sequences?

In [206]:
def check_overlap(array):
    num_sequences = array.shape[0]
    overlap_info = []  # 用于存储重合信息

    # 遍历所有序列对
    for i in range(num_sequences):         
        print('Inspecting seq {}'.format(i))
        for j in range(num_sequences):
            if i != j:  # 确保不比较同一个序列
                # 检查重合的最大长度
                max_overlap_length = min(array.shape[1], array.shape[1])
                for n in range(1, max_overlap_length + 1):
                    if np.array_equal(array[i, -n:], array[j, :n]):
                        overlap_info.append((i, j, n))  # (序列索引1, 序列索引2, 重合长度)
                        break  # 找到重合后可以跳出

    return overlap_info

# 示例数据
data = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][:,0,:]

# 检查重合
overlaps = check_overlap(data)

# 输出重合信息
for seq1, seq2, length in overlaps:
    print(f"序列 {seq1} 和序列 {seq2} 存在重合，重合长度为 {length}")

In [208]:
import numpy as np
import hashlib

def hash_sequence(seq):
    """计算序列的哈希值"""
    return hashlib.sha256(seq.tobytes()).hexdigest()

def check_overlap_with_hash(array):
    num_sequences = array.shape[0]
    overlap_info = []  # 用于存储重合信息
    hash_table = {}  # 哈希表

    # 遍历每个序列
    for i in range(num_sequences):
        print('Inspecting seq {}'.format(i))
        # 遍历可能的重合长度
        for n in range(1, array.shape[1] + 1):
            # 计算最后 N 个元素的哈希值
            seq_hash = hash_sequence(array[i, -n:])
            if seq_hash not in hash_table:
                hash_table[seq_hash] = i  # 存储序列的索引
            else:
                # 如果哈希值已存在，说明存在重合
                overlap_info.append((hash_table[seq_hash], i, n))
            # 检查下一个长度的哈希值
            if n == array.shape[1]:  # 防止超出边界
                break

    return overlap_info

# 示例数据
data = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][:,0,:]

# 检查重合
overlaps = check_overlap_with_hash(data)

# 输出重合信息
for seq1, seq2, length in overlaps:
    print(f"序列 {seq1} 和序列 {seq2} 存在重合，重合长度为 {length}")

In [214]:
overlaps

In [212]:
len(overlaps)

In [213]:
sum = 0
for _,_,i in overlaps:
    if i == 200:
        sum+=1
sum

In [210]:
np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][76046,0,:]

In [209]:
np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][76048,0,:]

In [215]:
# 示例数据
data = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][:,1,:]

# 检查重合
overlaps = check_overlap_with_hash(data)

# 输出重合信息
for seq1, seq2, length in overlaps:
    print(f"序列 {seq1} 和序列 {seq2} 存在重合，重合长度为 {length}")

In [218]:
overlaps

In [216]:
len(overlaps)

In [217]:
sum = 0
for _,_,i in overlaps:
    if i == 200:
        sum+=1
sum

In [219]:
np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][76046,1,:]

In [220]:
np.load('E://GWNMMAD_data/Tw_dataset/Datasets/background.npz')['data'][76048,1,:]

In [263]:
del overlaps

## Now, how about some signals?

In [16]:
BBH_GWAK = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/bbh_for_challenge.npy')
# noise_GWAK.shape
BBH_GWAK = BBH_GWAK / np.linalg.norm(BBH_GWAK, axis = -1).reshape(-1,2,1)

BBH_GWAK_fft = np.abs(np.fft.rfft(BBH_GWAK, axis = -1))
BBH_GWAK_fft = BBH_GWAK_fft / np.linalg.norm(BBH_GWAK_fft, axis = -1).reshape(-1,2,1)

BBH_GWAK_fft = BBH_GWAK_fft.reshape(-1,202)

In [17]:
BBH_new = np.load('../Data_cached/Noise_processing/BBH_injection/Output/BBH_events_v1_1(5s).npz')['events']
# noise_GWAK.shape
BBH_new = BBH_new / np.linalg.norm(BBH_new, axis = -1).reshape(-1,2,1)

BBH_new_fft = np.abs(np.fft.rfft(BBH_new, axis = -1))
BBH_new_fft = BBH_new_fft / np.linalg.norm(BBH_new_fft, axis = -1).reshape(-1,2,1)

BBH_new_fft = BBH_new_fft.reshape(-1,202)

In [18]:
BBH_GWAK_fft.shape

In [19]:
BBH_new_fft.shape

In [20]:
# 随机选择 10 个 (2, 200) 形状的数组
random_indices = np.random.choice(BBH_GWAK.shape[0], size=10, replace=False)
selected_data = BBH_GWAK[random_indices]

# 创建一个 5x2 的子图
fig, axs = plt.subplots(2, 5, figsize=(6 * 5, 4 * 2))  # 设置图形大小

# 绘制选定的数组
for i in range(10):
    row = i // 5  # 计算行索引
    col = i % 5   # 计算列索引
    
    axs[row, col].plot(selected_data[i, 0], label='H', color='blue')
    axs[row, col].plot(selected_data[i, 1], label='L', color='orange')
    axs[row, col].set_title(f'Selected Array {i + 1}')
    axs[row, col].legend()

# 调整布局
plt.tight_layout()

# 显示图形
plt.show()

In [21]:
# 随机选择 10 个 (2, 200) 形状的数组
random_indices = np.random.choice(BBH_new.shape[0], size=10, replace=False)
selected_data = BBH_new[random_indices]

# 创建一个 5x2 的子图
fig, axs = plt.subplots(2, 5, figsize=(6 * 5, 4 * 2))  # 设置图形大小

# 绘制选定的数组
for i in range(10):
    row = i // 5  # 计算行索引
    col = i % 5   # 计算列索引
    
    axs[row, col].plot(selected_data[i, 0], label='H', color='blue')
    axs[row, col].plot(selected_data[i, 1], label='L', color='orange')
    axs[row, col].set_title(f'Selected Array {i + 1}')
    axs[row, col].legend()

# 调整布局
plt.tight_layout()

# 显示图形
plt.show()

In [33]:
SGLF_GWAK = np.load('E://GWNMMAD_data/Tw_dataset/Datasets/sglf_for_challenge.npy')
# noise_GWAK.shape
SGLF_GWAK = SGLF_GWAK / np.linalg.norm(SGLF_GWAK, axis = -1).reshape(-1,2,1)
SGLF_GWAK_fft = np.abs(np.fft.rfft(SGLF_GWAK, axis = -1))
SGLF_GWAK_fft = SGLF_GWAK_fft / np.linalg.norm(SGLF_GWAK_fft, axis = -1).reshape(-1,2,1)

SGLF_GWAK_fft = SGLF_GWAK_fft.reshape(-1,202)

In [34]:
SGLF_new = np.load('../Data_cached/Noise_processing/SGLF_injection/Output/SGLF_events_v1(5s).npz')['events']
# noise_GWAK.shape
SGLF_new = SGLF_new / np.linalg.norm(SGLF_new, axis = -1).reshape(-1,2,1)

SGLF_new_fft = np.abs(np.fft.rfft(SGLF_new, axis = -1))
SGLF_new_fft = SGLF_new_fft / np.linalg.norm(SGLF_new_fft, axis = -1).reshape(-1,2,1)

SGLF_new_fft = SGLF_new_fft.reshape(-1,202)

## For pre-trained models, see its distribution for GWAK and Longseg

In [22]:
def Compute_errs(model, dataset):
    err = {}
    
    N_dataset = len(dataset)
    
    for i in range(N_dataset):
        
        rec = model(torch.FloatTensor(dataset[i]))[1].cpu().detach().numpy()
        error = np.mean((dataset[i] - rec)**2, axis = -1)
        
        err[i] = error
        
    return err

In [362]:
noise_set_combined = {}
noise_set_combined[0] = noise_GWAK_fft[-20000:]
noise_set_combined[1] = noise_Longseg_fft[-20000:]
noise_set_combined[2] = noise_Shortseg_fft[-10000:]
noise_set_combined[3] = noise_Shortseg_shortasd_fft[-10000:]
noise_set_combined[4] = noise_Shortseg_GWAK_fft[-20000:]
noise_set_combined[5] = noise_Longseg_GWAK_fft[-20000:]
noise_set_combined[6] = noise_Midseg_GWAK_fft[-20000:]

In [361]:
labels = {0:'GWAK',
         1:'Longseg',
         2:'Shortseg',
         3:'Shortseg, shortasd',
         4:'Shortseg, GWAK style',
         5:'Longseg, w glitches, GWAK style',
         6:'Midseg, w/o glitches, GWAK style'}

In [69]:
error_set_combined = Compute_errs(model, noise_set_combined)

In [70]:
error_set_combined[0]

In [71]:
for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

## Using GWAK set to train a sample

In [23]:
model_GWAK = trainAE_struct(noise_GWAK_fft[:-20000], [202,40,20], 'null')

In [24]:
set_combined = {}
set_combined[0] = noise_GWAK_fft[-20000:]
set_combined[1] = noise_Shortseg_GWAK_fft[-20000:]
set_combined[2] = BBH_GWAK_fft[-20000:]
set_combined[3] = BBH_new_fft

labels = ['GWAK noise', 'new noise', 'GWAK BBH', 'new BBH']

In [35]:
set_combined = {}
set_combined[0] = noise_GWAK_fft[-20000:]
set_combined[1] = noise_Shortseg_GWAK_fft[-20000:]
set_combined[2] = SGLF_GWAK_fft[-20000:]
set_combined[3] = SGLF_new_fft

labels = ['GWAK noise', 'new noise', 'GWAK SGLF', 'new SGLF']

In [55]:
BBH_new_fft_SNR = np.load('../Data_cached/Noise_processing/BBH_injection/Output/BBH_events_v3_1.npz')['SNR']

In [25]:
BBH_new_fft_SNR

In [29]:
idx_5_12 = np.logical_and(BBH_new_fft_SNR >= 5, BBH_new_fft_SNR < 12)
idx_12_24 = np.logical_and(BBH_new_fft_SNR >= 12, BBH_new_fft_SNR < 24)
idx_24_50 = np.logical_and(BBH_new_fft_SNR >= 24, BBH_new_fft_SNR < 50)

In [36]:
# SGLF, v1(5s)

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [27]:
# v1_1(5s)

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [56]:
# v3_1

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [57]:
# 创建一个空列表来存储结果
snr_range = []

# 遍历从 15 到 25 的每个数
for i in range(15, 26):  # 26 是上限，不包含
    for j in range(i, 26):  # j 从 i 开始到 25
        snr_range.append((i, j))

# 输出结果
# print(result)

In [58]:
snr_range

In [67]:
# 创建一个空列表来存储结果
snr_range_identical = []

# 遍历从 15 到 25 的每个数
for i in range(15, 26):  # 26 是上限，不包含
    for j in range(i, i+1):  # j 从 i 开始到 25
        snr_range_identical.append((i, j))

# 输出结果
# print(result)

In [68]:
snr_range_identical

In [72]:
# v3_1

error_set_combined_1 = Compute_errs(model_GWAK, set_combined)
error_set_combined_2 = Compute_errs(model_GWAK_BBH, set_combined)

# 创建一个 11x6 的子图网格（66个子图）
rows = 11
cols = 2
fig, axs = plt.subplots(rows, cols, figsize=(12, 44))  # 不指定 figsize，使用默认大小

# 生成一些示例数据并绘制每个子图
for i in range(rows):
    for j in range(1):
        # 计算当前子图索引
        index = i * (cols-1) + j
        # axs[i, j].plot(np.random.rand(10))  # 生成随机数据绘图
        # axs[i, j].set_title(f'Subplot {index + 1}')  # 设置子图标题
        snr_low, snr_high = snr_range_identical[index]
        idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)
        for k in range(len(set_combined)-1):
            axs[i,0].hist(error_set_combined_1[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
        axs[i,0].hist(error_set_combined_1[3][idx], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr_low}-{snr_high}({np.sum(idx)})')
        axs[i,0].legend()
        for k in range(len(set_combined)-1):
            axs[i,1].hist(error_set_combined_2[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
        axs[i,1].hist(error_set_combined_2[3][idx], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr_low}-{snr_high}({np.sum(idx)})')
        axs[i,1].legend()

# 调整布局，以免重叠
plt.tight_layout()

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

# plt.legend()

In [62]:
# v3_1

error_set_combined = Compute_errs(model_GWAK, set_combined)

# 创建一个 11x6 的子图网格（66个子图）
rows = 11
cols = 6
fig, axs = plt.subplots(rows, cols, figsize=(36, 44))  # 不指定 figsize，使用默认大小

# 生成一些示例数据并绘制每个子图
for i in range(rows):
    for j in range(cols):
        # 计算当前子图索引
        index = i * cols + j
        # axs[i, j].plot(np.random.rand(10))  # 生成随机数据绘图
        # axs[i, j].set_title(f'Subplot {index + 1}')  # 设置子图标题
        snr_low, snr_high = snr_range[index]
        idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)
        for k in range(len(set_combined)-1):
            axs[i, j].hist(error_set_combined[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
        axs[i, j].hist(error_set_combined[3][idx], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr_low}-{snr_high}')
        axs[i, j].legend()

# 调整布局，以免重叠
plt.tight_layout()

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

# plt.legend()

In [30]:
# v3

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [49]:
# v3

snr_low = 20
snr_high = 20

idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i] + f'_SNR_{snr_low}-{snr_high}')   

plt.legend()

In [50]:
np.sum(idx)

In [49]:
error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [65]:
# v2_2

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [40]:
error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [80]:
# v2_1
error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][np.logical_and(BBH_new_fft_SNR >= 19, BBH_new_fft_SNR < 20)], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_20') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [57]:
# v1_2

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [25]:
# v1_2

error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [61]:
# v2_2
for snr_low in range(12,17):
    for snr_high in range(18,21):
        
        snr_idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)
        
        error_set_combined = Compute_errs(model_GWAK, set_combined)

        for i in range(len(set_combined)-1):
            plt.hist(error_set_combined[i], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i])
         
        plt.hist(error_set_combined[3][snr_idx], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i] + '_SNR_{}-{}'.format(snr_low, snr_high))    
        plt.legend()
        plt.show()

In [54]:
for snr_low in range(12,17):
    for snr_high in range(18,21):
        
        snr_idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)
        
        error_set_combined = Compute_errs(model_GWAK, set_combined)

        for i in range(len(set_combined)-1):
            plt.hist(error_set_combined[i], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i])
         
        plt.hist(error_set_combined[3][snr_idx], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[i] + '_SNR_{}-{}'.format(snr_low, snr_high))    
        plt.legend()
        plt.show()

In [45]:
error_set_combined = Compute_errs(model_GWAK, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [104]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [109]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [139]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [187]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [202]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [240]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [260]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [272]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [279]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [295]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [306]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [315]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [324]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [331]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [338]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [378]:
error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [356]:
# v3_5

error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [363]:
# v3_4

error_set_combined = Compute_errs(model_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [40]:
for i in range(3):
    print(np.linalg.norm(noise_set_combined[i], axis = -1))

## Using shortened segment set to do the training

In [41]:
model_short = trainAE_struct(noise_Shortseg_fft[:-20000], [202,40,20], 'null')

In [42]:
error_set_combined = Compute_errs(model_short, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

## Using shortened segment, short asd set to do the training

In [203]:
model_short_shortasd = trainAE_struct(noise_Shortseg_shortasd_fft[:-20000], [202,40,20], 'null')

In [204]:
error_set_combined = Compute_errs(model_short_shortasd, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_short_shortasd(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

## Using shortened segment, GWAK processed set to do the training

In [261]:
model_short_GWAK = trainAE_struct(noise_Shortseg_GWAK_fft[:-20000], [202,40,20], 'null')

In [249]:
error_set_combined = Compute_errs(model_short_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_short_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [262]:
error_set_combined = Compute_errs(model_short_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_short_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

## Using mid segment, GWAK processed set to do the training

In [379]:
model_middle_GWAK = trainAE_struct(noise_Midseg_GWAK_fft[:-20000], [202,40,20], 'null')

In [380]:
error_set_combined = Compute_errs(model_middle_GWAK, noise_set_combined)

for i in [0,4,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

# plt.hist(np.mean((noise_normal_fft[-20000:] - model_short_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
#          bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

In [ ]:
error_set_combined = Compute_errs(model_short_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.hist(np.mean((noise_normal_fft[-20000:] - model_short_GWAK(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

plt.legend()

## Using longer segment to do the training

In [43]:
model_long = trainAE_struct(noise_Longseg_fft[:-20000], [202,40,20], 'null')

In [44]:
error_set_combined = Compute_errs(model_long, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [49]:
model_long = trainAE_struct(noise_Longseg_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [54]:
model_long = trainAE_struct(noise_Longseg_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

## Using longer segment, GWAK processed set to do the training

In [307]:
model_long_GWAK = trainAE_struct(noise_Longseg_GWAK_fft[:-20000], [202,40,20], 'null')

In [373]:
# v3_5 and v3_4

# model_long_GWAK = trainAE_struct(noise_Longseg_GWAK_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in [5,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [372]:
# v3_5 and v3_4

# model_long_GWAK = trainAE_struct(noise_Longseg_GWAK_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in [5,6]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [344]:
# v3_5

model_long_GWAK = trainAE_struct(noise_Longseg_GWAK_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [364]:
# v3_4

model_long_GWAK = trainAE_struct(noise_Longseg_GWAK_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [276]:
model_long_GWAK = trainAE_struct(noise_Longseg_GWAK_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [278]:
error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.legend()

In [308]:
error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in [0,4,5]:
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.legend()

In [ ]:
error_set_combined = Compute_errs(model_long_GWAK, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [ ]:
model_long = trainAE_struct(noise_Longseg_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [ ]:
model_long = trainAE_struct(noise_Longseg_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_long, noise_set_combined)

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

## Using normal segment set to do the training

In [98]:
model_normal = trainAE_struct(noise_normal_fft[:-20000], [202,40,20], 'null')

In [103]:
model_normal = trainAE_struct(noise_normal_fft[:-20000], [202,40,20], 'null')

error_set_combined = Compute_errs(model_normal, noise_set_combined)

plt.hist(np.mean((noise_normal_fft[-20000:] - model_normal(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [141]:
model_normal_bandpass = trainAE_struct(noise_normal_fft[:-20000], [202,40,20], 'null')


In [108]:

error_set_combined = Compute_errs(model_normal_bandpass, noise_set_combined)

plt.hist(np.mean((noise_normal_fft[-20000:] - model_normal_bandpass(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

In [142]:

error_set_combined = Compute_errs(model_normal_bandpass, noise_set_combined)

plt.hist(np.mean((noise_normal_fft[-20000:] - model_normal_bandpass(torch.FloatTensor(noise_normal_fft[-20000:]))[1].cpu().detach().numpy())**2, axis = -1), 
         bins = 50, range = (0,0.01), histtype='step', density = True, label = 'normal')

for i in range(len(noise_set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])

plt.legend()

## Now do the Signal things. Using GWAK BBH to train a model

In [28]:
model_GWAK_BBH = trainAE_struct(BBH_GWAK_fft[:-20000], [202,20,20], 'null')

In [48]:
model_GWAK_BBH = model_normal

In [63]:
# v3_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

# 创建一个 11x6 的子图网格（66个子图）
rows = 11
cols = 6
fig, axs = plt.subplots(rows, cols, figsize=(36, 44))  # 不指定 figsize，使用默认大小

# 生成一些示例数据并绘制每个子图
for i in range(rows):
    for j in range(cols):
        # 计算当前子图索引
        index = i * cols + j
        # axs[i, j].plot(np.random.rand(10))  # 生成随机数据绘图
        # axs[i, j].set_title(f'Subplot {index + 1}')  # 设置子图标题
        snr_low, snr_high = snr_range[index]
        idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)
        for k in range(len(set_combined)-1):
            axs[i, j].hist(error_set_combined[k], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[k])
        axs[i, j].hist(error_set_combined[3][idx], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[3] + f'_SNR_{snr_low}-{snr_high}')
        axs[i, j].legend()

# 调整布局，以免重叠
plt.tight_layout()

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

# plt.legend()

In [46]:
# New GWAK from v3

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [48]:
# v3

snr_low = 20
snr_high = 20

idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + f'_SNR_{snr_low}-{snr_high}') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [50]:
# New GWAK from v2_2

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [50]:
BBH_new_fft_SNR

In [51]:
# New GWAK from v2_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [75]:
# New GWAK from v2_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [77]:
# New GWAK from v2_1, snr 12-20

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



plt.hist(error_set_combined[3][np.argwhere(np.logical_and(BBH_new_fft_SNR >= 19, BBH_new_fft_SNR <= 20))], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [64]:
# New GWAK from v2_1, snr 12-20

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



plt.hist(error_set_combined[3][np.argwhere(np.logical_and(BBH_new_fft_SNR >= 12, BBH_new_fft_SNR <= 20))], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-20') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [44]:
error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [61]:
# v1_3

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

## Now do the Signal things. Using new BBH to train a model

In [83]:
model_new_BBH = trainAE_struct(BBH_new_fft[np.logical_and(BBH_new_fft_SNR >= 12, BBH_new_fft_SNR <= 20)], [202,20,20], 'null')

In [ ]:
model_GWAK_BBH = model_normal

In [ ]:
# New GWAK from v2_2

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
BBH_new_fft_SNR

In [ ]:
# New GWAK from v2_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# New GWAK from v2_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [87]:
# New GWAK from v2_1, snr 12-20

error_set_combined = Compute_errs(model_new_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i])



plt.hist(error_set_combined[3][np.argwhere(np.logical_and(BBH_new_fft_SNR >= 12, BBH_new_fft_SNR <= 20))], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# New GWAK from v2_1, snr 12-20

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



plt.hist(error_set_combined[3][np.argwhere(np.logical_and(BBH_new_fft_SNR >= 12, BBH_new_fft_SNR <= 20))], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-20') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# v1_3

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

## Now do the Signal things. Using GWAK SGLF to train a model

In [37]:
model_GWAK_SGLF = trainAE_struct(SGLF_GWAK_fft[:-20000], [202,20,20], 'null')

In [ ]:
# v3_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

# 创建一个 11x6 的子图网格（66个子图）
rows = 11
cols = 6
fig, axs = plt.subplots(rows, cols, figsize=(36, 44))  # 不指定 figsize，使用默认大小

# 生成一些示例数据并绘制每个子图
for i in range(rows):
    for j in range(cols):
        # 计算当前子图索引
        index = i * cols + j
        # axs[i, j].plot(np.random.rand(10))  # 生成随机数据绘图
        # axs[i, j].set_title(f'Subplot {index + 1}')  # 设置子图标题
        snr_low, snr_high = snr_range[index]
        idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)
        for k in range(len(set_combined)-1):
            axs[i, j].hist(error_set_combined[k], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[k])
        axs[i, j].hist(error_set_combined[3][idx], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[3] + f'_SNR_{snr_low}-{snr_high}')
        axs[i, j].legend()

# 调整布局，以免重叠
plt.tight_layout()

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

# plt.legend()

In [ ]:
# New GWAK from v3

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# v3

snr_low = 20
snr_high = 20

idx = np.logical_and(BBH_new_fft_SNR >= snr_low, BBH_new_fft_SNR <= snr_high)

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + f'_SNR_{snr_low}-{snr_high}') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# New GWAK from v2_2

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
BBH_new_fft_SNR

In [ ]:
# New GWAK from v2_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# New GWAK from v2_1

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# New GWAK from v2_1, snr 12-20

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



plt.hist(error_set_combined[3][np.argwhere(np.logical_and(BBH_new_fft_SNR >= 19, BBH_new_fft_SNR <= 20))], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# New GWAK from v2_1, snr 12-20

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])



plt.hist(error_set_combined[3][np.argwhere(np.logical_and(BBH_new_fft_SNR >= 12, BBH_new_fft_SNR <= 20))], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i] + '_SNR_12-20') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

# plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
# plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
# plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

In [ ]:
# v1_3

error_set_combined = Compute_errs(model_GWAK_BBH, set_combined)

for i in range(len(set_combined)-1):
    plt.hist(error_set_combined[i], bins = 50, range = (0,0.004), histtype='step', density = True, label = labels[i])

plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

plt.legend()

## Trying to find proper snr

In [30]:
snr_list = [16,17,18]

for snr in snr_list:
    BBH_new = np.load(f'../Data_cached/Noise_processing/BBH_injection/Output/BBH_events_v1_1(5s).npz')['events']
    # noise_GWAK.shape
    BBH_new = BBH_new / np.linalg.norm(BBH_new, axis = -1).reshape(-1,2,1)

    BBH_new_fft = np.abs(np.fft.rfft(BBH_new, axis = -1))
    BBH_new_fft = BBH_new_fft / np.linalg.norm(BBH_new_fft, axis = -1).reshape(-1,2,1)

    BBH_new_fft = BBH_new_fft.reshape(-1,202)
    
    # set_combined = {}
    # set_combined[0] = noise_GWAK_fft[-20000:]
    # set_combined[1] = noise_Shortseg_GWAK_fft[-20000:]
    # set_combined[2] = BBH_GWAK_fft[-20000:]
    # set_combined[3] = BBH_new_fft

    labels = ['GWAK noise', 'new noise', 'GWAK BBH', 'new BBH']
    
    BBH_new_fft_SNR = np.load('../Data_cached/Noise_processing/BBH_injection/Output/BBH_events_v1_1(5s).npz')['SNR']
    
    idx = (BBH_new_fft_SNR == snr)
    
    BBH_new_fft = BBH_new_fft[idx]
    
    set_combined = {}
    set_combined[0] = noise_GWAK_fft[-20000:]
    set_combined[1] = noise_Shortseg_GWAK_fft[-20000:]
    set_combined[2] = BBH_GWAK_fft[-20000:]
    set_combined[3] = BBH_new_fft
    
    # assert np.all(BBH_new_fft_SNR == snr)
    
        # v3_1

    error_set_combined_1 = Compute_errs(model_GWAK, set_combined)
    error_set_combined_2 = Compute_errs(model_GWAK_BBH, set_combined)

    # 创建一个 11x6 的子图网格（66个子图）
    rows = 1
    cols = 2
    fig, axs = plt.subplots(rows, cols, figsize=(12, 4))  # 不指定 figsize，使用默认大小

    # 生成一些示例数据并绘制每个子图
    for i in range(rows):
        for j in range(1):
            # 计算当前子图索引
            index = i * (cols-1) + j
            # axs[i, j].plot(np.random.rand(10))  # 生成随机数据绘图
            # axs[i, j].set_title(f'Subplot {index + 1}')  # 设置子图标题
            for k in range(len(set_combined)-1):
                axs[0].hist(error_set_combined_1[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
            axs[0].hist(error_set_combined_1[3], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr}({len(BBH_new_fft)})')
            axs[0].legend()
            for k in range(len(set_combined)-1):
                axs[1].hist(error_set_combined_2[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
            axs[1].hist(error_set_combined_2[3], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr}({len(BBH_new_fft)})')
            axs[1].legend()

    # 调整布局，以免重叠
    plt.tight_layout()

    # plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
    # plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
    # plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

    # plt.legend()

In [40]:
snr_list = [16,17,18]

for snr in snr_list:
    SGLF_new = np.load(f'../Data_cached/Noise_processing/SGLF_injection/Output/SGLF_events_v1(5s).npz')['events']
    # noise_GWAK.shape
    SGLF_new = SGLF_new / np.linalg.norm(SGLF_new, axis = -1).reshape(-1,2,1)

    SGLF_new_fft = np.abs(np.fft.rfft(SGLF_new, axis = -1))
    SGLF_new_fft = SGLF_new_fft / np.linalg.norm(SGLF_new_fft, axis = -1).reshape(-1,2,1)

    SGLF_new_fft = SGLF_new_fft.reshape(-1,202)
    
    # set_combined = {}
    # set_combined[0] = noise_GWAK_fft[-20000:]
    # set_combined[1] = noise_Shortseg_GWAK_fft[-20000:]
    # set_combined[2] = BBH_GWAK_fft[-20000:]
    # set_combined[3] = BBH_new_fft

    labels = ['GWAK noise', 'new noise', 'GWAK SGLF', 'new SGLF']
    
    SGLF_new_fft_SNR = np.load('../Data_cached/Noise_processing/SGLF_injection/Output/SGLF_events_v1(5s).npz')['SNR']
    
    idx = (SGLF_new_fft_SNR == snr)
    
    SGLF_new_fft = SGLF_new_fft[idx]
    
    set_combined = {}
    set_combined[0] = noise_GWAK_fft[-20000:]
    set_combined[1] = noise_Shortseg_GWAK_fft[-20000:]
    set_combined[2] = SGLF_GWAK_fft[-20000:]
    set_combined[3] = SGLF_new_fft
    
    # assert np.all(BBH_new_fft_SNR == snr)
    
        # v3_1

    error_set_combined_1 = Compute_errs(model_GWAK, set_combined)
    error_set_combined_2 = Compute_errs(model_GWAK_SGLF, set_combined)

    # 创建一个 11x6 的子图网格（66个子图）
    rows = 1
    cols = 2
    fig, axs = plt.subplots(rows, cols, figsize=(12, 4))  # 不指定 figsize，使用默认大小

    # 生成一些示例数据并绘制每个子图
    for i in range(rows):
        for j in range(1):
            # 计算当前子图索引
            index = i * (cols-1) + j
            # axs[i, j].plot(np.random.rand(10))  # 生成随机数据绘图
            # axs[i, j].set_title(f'Subplot {index + 1}')  # 设置子图标题
            for k in range(len(set_combined)-1):
                axs[0].hist(error_set_combined_1[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
            axs[0].hist(error_set_combined_1[3], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr}({len(BBH_new_fft)})')
            axs[0].legend()
            for k in range(len(set_combined)-1):
                axs[1].hist(error_set_combined_2[k], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[k])
            axs[1].hist(error_set_combined_2[3], bins = 50, range = (0,0.006), histtype='step', density = True, label = labels[3] + f'_SNR_{snr}({len(BBH_new_fft)})')
            axs[1].legend()

    # 调整布局，以免重叠
    plt.tight_layout()

    # plt.hist(error_set_combined[3][idx_5_12], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_5-12') 
    # plt.hist(error_set_combined[3][idx_12_24], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_12-24') 
    # plt.hist(error_set_combined[3][idx_24_50], bins = 50, range = (0,0.01), histtype='step', density = True, label = labels[i] + '_SNR_24-50')    

    # plt.legend()

# Using supervised classifier and SVM to replace the node

In [22]:
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC
from sklearn.svm import OneClassSVM

# First, come up with a supervised model for later SVM study. Here only glitch and noise are entering the training scheme

In [18]:
training_set = torch.load('./Sida_temp/for_K8S_training/Data/wsc_training_set_GWAK.json')

In [19]:
training_set.keys()

In [9]:
class WSC_1det_struct(nn.Module):
    def __init__(self, 
                 encoder_struct):
        super(WSC_1det_struct, self).__init__()
        
        self.dep = len(encoder_struct)
        self.encoder_struct = torch.IntTensor(encoder_struct)

        self.relu = nn.ReLU()  # 激活函数
        self.layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()


        for i in range(self.dep-1):
            layer = nn.Linear(self.encoder_struct[i], self.encoder_struct[i+1])
            nn.init.kaiming_normal_(layer.weight)
            self.layers.append(layer)
            self.norm_layers.append(nn.BatchNorm1d(self.encoder_struct[i+1]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < self.dep-2:
                x = self.relu(x)
                # x = nn.BatchNorm1d(self.encoder_struct[i+1])(x)
                x = self.norm_layers[i](x)

        return x

In [28]:
device = 'cuda:0'

def trainSeriesSupC_struct(datasets, struct, save_path):
# datasets: multiple datasets, datasets[0, 1, ...] are for the 1st, 2nd, ... class
# datasets should have the keys to be the integres 0, 1, 2, ...

    epochs_wsc = 500
    batch_size_wsc = 384
    lr_wsc = 2e-5
    
    least_epochs = 100
    epochs_interval = 20
    model_list = {}
    
    wsc = WSC_1det_struct(struct).to(device)
    nparam = sum(p.numel() for p in wsc.parameters() if p.requires_grad)
    print(nparam)
    Nclass = len(datasets)
    wclass = torch.FloatTensor([len(datasets[0])/len(datasets[i]) for i in range(Nclass)]).to(device)
    nTotal = {}
    nTrain = {}
    nTest = {}
    nbkg_train = 0
    nsig_train = 0
    for i in np.arange(Nclass):
        nTotal[i] = datasets[i].shape[0]
        nTrain[i] = int(rTrain*nTotal[i])
        nTest[i] = int(rTest*nTotal[i])
        if i < 2:
            nbkg_train += nTrain[i]
        else:
            nsig_train += nTrain[i]

    X_train = np.concatenate([datasets[i][:nTrain[i]] for i in range(Nclass)])
    X_test = np.concatenate([datasets[i][-nTest[i]:] for i in range(Nclass)])
    X_validation = np.concatenate([datasets[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    Y_train = np.concatenate([i*np.ones(nTrain[i], dtype=int) for i in np.arange(Nclass)])
    Y_validation = np.concatenate([i*np.ones(nTotal[i]-nTrain[i]-nTest[i], dtype=int) for i in np.arange(Nclass)])

    train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(Y_train).to(device))
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).to(device), torch.LongTensor(Y_validation).to(device))
    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)

    # associated with a direct sum of probability, see below
    optimizer = optim.Adam(wsc.parameters(), lr=lr_wsc)
    loss_func = nn.CrossEntropyLoss(weight=wclass).to(device)
    loss_train = np.empty(epochs_wsc)
    loss_validation = np.empty(epochs_wsc)

    for epoch in range(epochs_wsc):
        wsc.train()
        for batchidx, (x, y) in enumerate(trainDataLoader):
            yprime = wsc(x)
            loss = custom_cross_entropy_loss(yprime, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, (x, y) in enumerate(validationDataLoader):
                yprime = wsc(x)
                lossVal = custom_cross_entropy_loss(yprime, y)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)
            
        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
        
        if ((epoch+1) > least_epochs) and ((epoch+1) % epochs_interval == 0):
            model_list[(epoch+1)] = copy.deepcopy(wsc.cpu().eval())
            model_list[str(epoch+1)+'valloss'] = val_loss
            wsc.to(device)
        
    wsc.to(device).eval()
    
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    foo = ax[1].hist(nn.Sigmoid()(wsc(torch.FloatTensor(X_train).to(device))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")
    foo = ax[1].hist(nn.Sigmoid()(wsc(torch.FloatTensor(X_test ).to(device))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")

    # plt.show()
    plt.savefig(save_path)
    plt.close()
    
    return model_list

In [18]:
def custom_cross_entropy_loss(input, target):
    """
    自定义的交叉熵损失函数
    
    参数:
    input (torch.Tensor): 模型输出的logits,shape为 (batch_size, num_classes)
    target (torch.Tensor): 真实标签,shape为 (batch_size,)
    """
    
    print(target.shape)
    batch_size = input.size(0)
    # print(batch_size)
    
    # 使用 sigmoid 计算概率
    prob = torch.sigmoid(input)
    prob = torch.clamp(prob, 1e-5, 1-1e-5)
    # print(prob.shape)
    
    # 计算 log(1-p) 部分
    loss_pos = -torch.log(prob.gather(1, target.unsqueeze(1))).squeeze(1)
    # print(loss_pos)

    # 获取非目标位置的索引
    non_target_idx = torch.ones_like(input, dtype=torch.bool)
    # print(non_target_idx)
    non_target_idx.scatter_(1, target.unsqueeze(1), 0)
    # print(non_target_idx)

    # 计算非目标位置的 log 损失
    # print(prob.masked_select(non_target_idx))
    loss_neg = -torch.log(1 - prob).masked_select(non_target_idx)
    # print(loss_neg)
    
    # 计算平均损失
    loss = (loss_pos.sum() + loss_neg.sum()) / batch_size
    
    return loss

In [19]:
Picture_dir = '../Pic_cached/Supervised_study'

In [20]:
training_set.keys()

In [21]:
# 要提取的键
keys_to_extract = [0,1,2]

# 使用字典推导式获取部分键值数据
partial_dict = {key: training_set[key] for key in keys_to_extract if key in training_set}

In [22]:
partial_dict.keys()

In [29]:
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
model_list = trainSeriesSupC_struct(partial_dict, [202,128,16,3], Picture_dir + '/202-128-16-3_gwak_for_pipeline.png')

In [41]:
# Something wrong with the cuda kernel?
# Already have the result from another kernel

model_list = torch.load('../Model_cached/Supervised+Cluster_analysis/202-128-16-3_gwak_firststep.json')

In [42]:
model = return_model_with_least_valloss(model_list)

In [43]:
Score_distribution_train = {}
for i in training_set.keys():
    print(i)
    Score_distribution_train[i] = nn.Sigmoid()(model(torch.FloatTensor(training_set[i]))).detach().numpy()

In [23]:

FPR_SVM_Glitch = 0.1
FPR_SVM_Noise = 0.1
FPR_SVM_BBH = 0.1



# 使用 One-Class SVM 进行单类别聚类
oc_svm_glitch = OneClassSVM(nu=FPR_SVM_Glitch, kernel='rbf', gamma='auto')
oc_svm_glitch.fit(Score_distribution_train[0])

# 预测数据点
# y_pred = oc_svm_glitch.predict(Score_distribution[0])

# np.sum(y_pred == 1)

# 使用 One-Class SVM 进行单类别聚类
oc_svm_noise = OneClassSVM(nu=FPR_SVM_Noise, kernel='rbf', gamma='auto')
oc_svm_noise.fit(Score_distribution_train[1])

oc_svm_BBH = OneClassSVM(nu=FPR_SVM_BBH, kernel='rbf', gamma='auto')
oc_svm_BBH.fit(Score_distribution_train[2])

# 预测数据点
# y_pred = oc_svm_noise.predict(Score_distribution[1])
# y_test = clf_noise.predict(Score_distribution[1])

# plt.scatter(Score_distribution[1][:, 1], Score_distribution[1][:, 2], c=y_test)
# np.sum(y_test == 1)
# y_test = clf_glitch.predict(Score_distribution[0])

# plt.scatter(Score_distribution[0][:, 0], Score_distribution[0][:, 1], c=y_test)
# # np.sum(y_test == -1)
# X = np.concatenate((Score_distribution_train[0][np.argwhere(oc_svm_glitch.predict(Score_distribution_train[0]) == 1).flatten()], 
#                     Score_distribution_train[1][np.argwhere(oc_svm_noise.predict(Score_distribution_train[1]) == 1).flatten()]), axis = 0)
# # np.argwhere(clf_glitch.predict(Score_distribution[0]) == 1).shape
# # X.shape
# # from sklearn.mixture import GaussianMixture

# # 假设 X 是你的特征矩阵
# # X = ...  # 你的特征数据
# print(X.shape)

# # 初始化GMM
# gmm = GaussianMixture(n_components=2, random_state=0)



# # 进行聚类
# gmm.fit(X)

# # 预测标签
# labels = gmm.predict(X)

# # 评估新数据点
# # new_data = np.array([[1.5, 1.5]])  # 新数据点
# # probability = gmm.score_samples(new_data)

# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(X[:1000][:, 0], X[:1000][:, 1], X[:1000][:, 2], c='purple', cmap='rainbow', marker='o', alpha=0.5)
# ax.scatter(X[15000:15500][:, 0], X[15000:15500][:, 1], X[15000:15500][:, 2], c='r', cmap='rainbow', marker='o', alpha=0.5)
# # ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=labels, cmap='rainbow', marker='o', alpha=0.5)

# # # 设置标题和标签
# ax.set_title('Gaussian Mixture Model Clustering in 3D')
# ax.set_xlabel('Feature 1')
# ax.set_ylabel('Feature 2')
# ax.set_zlabel('Feature 3')
# plt.show()

# # 评估新数据点
# # new_data = np.array([[0.5, 0.5, 0.5]])  # 新数据点
# # probability = gmm.score_samples(new_data)

# # # 判断是否为噪声（设定阈值）
# # threshold = -10  # 示例阈值
# # if probability < threshold:
# #     print("新数据点被认为是噪声")
# # else:
# #     print("新数据点属于已知聚类")
# probability_bkgclass = gmm.score_samples(np.concatenate([Score_distribution[i] for i in range(2)], axis = 0))
# probability_sigclass = gmm.score_samples(np.concatenate([Score_distribution[i] for i in [2,3]], axis = 0))

# # assert probability_bkgclass.shape[0] == 10000 * 2
# # assert probability_sigclass.shape[0] == 10000 * 2
# # new_data.shape
# # print(np.sum(probability_sigclass < threshold) / len(probability_sigclass))
# # print(np.sum(probability_bkgclass < threshold) / len(probability_bkgclass))
# probability_bkgclass.sort()
# # probability_sigclass
# FPR_list = np.arange(0,1,0.01)
# TPR_list = np.empty(100)
# threshold_list = np.empty(100)

# for (i,FPR) in enumerate(FPR_list):
#     threshold = probability_bkgclass[int(FPR * len(probability_bkgclass))]
#     TPR = np.sum(probability_sigclass < threshold) / len(probability_sigclass)
#     TPR_list[i] = TPR
#     threshold_list[i] = threshold
    
# plt.plot(FPR_list, TPR_list)
# # print(FPR_list[np.argwhere(TPR_list > 0.9)][0])
# # from sklearn.metrics import auc
# # print(auc(FPR_list, TPR_list))

# FPR_final_list[k] = FPR_list[np.argwhere(TPR_list > 0.9)][0,0]
# print(FPR_list[np.argwhere(TPR_list > 0.9)][0,0])



In [44]:
Score_distribution_SGLF = nn.Sigmoid()(model(torch.FloatTensor(dataset_combined_extra_SGHF))).detach().numpy()

In [46]:
Score_distribution_SGLF.shape

In [47]:
np.sum((oc_svm_noise.predict(Score_distribution_SGLF) == -1))

In [49]:
train_dataset_afterSVM = {}


train_dataset_afterSVM[0] = training_set[0]
                          
train_dataset_afterSVM[1] = training_set[1]

train_dataset_afterSVM[2] = training_set[2]

train_dataset_afterSVM[3] = dataset_combined_extra_SGHF[np.argwhere(((oc_svm_glitch.predict(Score_distribution_SGLF) == -1) & (oc_svm_noise.predict(Score_distribution_SGLF) == -1)) & (oc_svm_BBH.predict(Score_distribution_SGLF) == -1)).flatten()]

In [64]:
np.sum(((oc_svm_glitch.predict(Score_distribution_SGLF) == -1) & (oc_svm_noise.predict(Score_distribution_SGLF) == -1)) & (oc_svm_BBH.predict(Score_distribution_SGLF) == -1)[:2500])

In [65]:
idx = ((oc_svm_glitch.predict(Score_distribution_SGLF) == -1) & (oc_svm_noise.predict(Score_distribution_SGLF) == -1)) & (oc_svm_BBH.predict(Score_distribution_SGLF) == -1)

In [69]:
np.sum(idx[2500:35000])

In [70]:
np.sum(idx[35000:40000])

In [71]:
np.sum(idx[40000:45000])

In [72]:
np.sum(idx[45000:50000])

In [51]:
training_set[3].shape

In [52]:
train_dataset_afterSVM[3].shape

In [30]:
device = 'cuda:0'

In [31]:
def trainSeriesSupC_struct(datasets, struct, save_path):
# datasets: multiple datasets, datasets[0, 1, ...] are for the 1st, 2nd, ... class
# datasets should have the keys to be the integres 0, 1, 2, ...

    epochs_wsc = 500
    batch_size_wsc = 384
    lr_wsc = 2e-5
    
    least_epochs = 100
    epochs_interval = 20
    model_list = {}
    
    wsc = WSC_1det_struct(struct).to(device)
    nparam = sum(p.numel() for p in wsc.parameters() if p.requires_grad)
    
    Nclass = len(datasets)
    wclass = torch.FloatTensor([len(datasets[0])/len(datasets[i]) for i in range(Nclass)]).to(device)
    nTotal = {}
    nTrain = {}
    nTest = {}
    nbkg_train = 0
    nsig_train = 0
    for i in np.arange(Nclass):
        nTotal[i] = datasets[i].shape[0]
        nTrain[i] = int(rTrain*nTotal[i])
        nTest[i] = int(rTest*nTotal[i])
        if i < 2:
            nbkg_train += nTrain[i]
        else:
            nsig_train += nTrain[i]

    X_train = np.concatenate([datasets[i][:nTrain[i]] for i in range(Nclass)])
    X_test = np.concatenate([datasets[i][-nTest[i]:] for i in range(Nclass)])
    X_validation = np.concatenate([datasets[i][nTrain[i]:-nTest[i]] for i in range(Nclass)])

    Y_train = np.concatenate([i*np.ones(nTrain[i], dtype=int) for i in np.arange(Nclass)])
    Y_validation = np.concatenate([i*np.ones(nTotal[i]-nTrain[i]-nTest[i], dtype=int) for i in np.arange(Nclass)])

    train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(Y_train).to(device))
    validation_dataset = TensorDataset(torch.FloatTensor(X_validation).to(device), torch.LongTensor(Y_validation).to(device))
    trainDataLoader = DataLoader(dataset=train_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)
    validationDataLoader = DataLoader(dataset=validation_dataset, batch_size=batch_size_wsc, shuffle=True, drop_last=True)

    # associated with a direct sum of probability, see below
    optimizer = optim.Adam(wsc.parameters(), lr=lr_wsc)
    loss_func = nn.CrossEntropyLoss(weight=wclass).to(device)
    loss_train = np.empty(epochs_wsc)
    loss_validation = np.empty(epochs_wsc)

    for epoch in range(epochs_wsc):
        wsc.train()
        for batchidx, (x, y) in enumerate(trainDataLoader):
            yprime = wsc(x)
            loss = loss_func(yprime, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        wsc.eval()
        with torch.no_grad():
            val_loss = 0
            for batchidx, (x, y) in enumerate(validationDataLoader):
                yprime = wsc(x)
                lossVal = loss_func(yprime, y)
                val_loss += lossVal.item()

            val_loss /= len(validationDataLoader)
            
        loss_train[epoch] = loss.item()
        loss_validation[epoch] = val_loss
        
        if ((epoch+1) > least_epochs) and ((epoch+1) % epochs_interval == 0):
            model_list[(epoch+1)] = copy.deepcopy(wsc.cpu().eval())
            model_list[str(epoch+1)+'valloss'] = val_loss
            wsc.to(device)
        
    wsc.to(device).eval()
    
    _, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(loss_train)
    ax[0].plot(loss_validation)
    foo = ax[1].hist(nn.Softmax(dim=1)(wsc(torch.FloatTensor(X_train).to(device))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")
    foo = ax[1].hist(nn.Softmax(dim=1)(wsc(torch.FloatTensor(X_test ).to(device))).cpu().detach().numpy().dot([0., 0.]+[1.]*(Nclass-2)), range=(0, 1), bins=20, density=True, histtype="step")

    plt.show()
    plt.savefig(save_path)
    plt.close()
    
    return model_list

In [32]:
len(train_dataset_afterSVM)

In [53]:
final_model_list = trainSeriesSupC_struct(train_dataset_afterSVM, [202, 64, 16, 4], './null_3.png')

In [54]:
model = return_model_with_least_valloss(final_model_list)

In [55]:
torch.save(model, '../Model_cached/Supervised+Cluster_analysis/202-128-16-3_gwak_secondstep_1.pt')

In [60]:
model = torch.load('../Model_cached/Supervised+Cluster_analysis/202-128-16-3_gwak_secondstep_1.pt')

In [61]:
model

In [56]:
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)

# for dt in range(5):
#     # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
#     err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
#     passidx = err_score > threshold
#     # print(cutAE[iStep-1])
#     # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
#     print(np.sum(passidx))

In [74]:
BKG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[:35000]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    
SIG_class_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_combined_extra_SGHF[35000:]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)

SIG_class_score.sort()

threshold = SIG_class_score[int(0.1 * len(SIG_class_score))]

FPR = np.sum(BKG_class_score > threshold) / len(BKG_class_score)

print(FPR)

for dt in range(5):
    # dcd = models['last_wsc'](torch.FloatTensor(dataset_wsl_separated[dt]))[1].detach().numpy()
    err_score = np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[dt]))).detach().numpy() * np.array([0,0,1,1])), axis = 1)
    passidx = err_score > threshold
    # print(cutAE[iStep-1])
    # dataset_wsl_separated[dt] = dataset_wsl_separated[dt][passidx]
    
    print(np.sum(passidx))

In [73]:
threshold

In [63]:
key2label = ['glitch', 'noise', 'bbh', 'sghf', 'sglf']


dataset_wsl_separated = {}

dataset_wsl_separated[0] = dataset_combined_extra_SGHF[:2500].copy()        # glitch
dataset_wsl_separated[1] = dataset_combined_extra_SGHF[2500:35000].copy()   # noise
dataset_wsl_separated[2] = dataset_combined_extra_SGHF[35000:40000].copy()  # bbh
dataset_wsl_separated[3] = dataset_combined_extra_SGHF[40000:45000].copy()  # sglf
dataset_wsl_separated[4] = dataset_combined_extra_SGHF[45000:].copy()       # sghf

# for key in dataset_wsl_separated.keys():
for key in [0,1,2,3,4]:
    print(dataset_wsl_separated[key].shape)
    plt.hist(np.sum((nn.Softmax(dim = 1)(model(torch.FloatTensor(dataset_wsl_separated[key]))).detach().numpy() * np.array([0,0,1,1])), axis = 1), density = True, histtype='step', bins = 50, label = key2label[key])
plt.legend()

# Time to think about can we do the simulation of noise ourself?